In [ ]:
# The environment to use here is: celltraj_env
#%%capture
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sys
import trajectory
import imageprep as imprep
import utilities
import features
import model
import h5py
import pickle
import os
import subprocess
import time
#import bootstrap
import umap
import pyemma.coordinates as coor
import scipy
import csaps
import skimage
from IPython.display import clear_output
import os
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
np.float=np.dtype(float)
np.int=int
import pandas as pd
import itertools
from itertools import chain
import imageprep as imprep
import sklearn
import json
from collections import defaultdict
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import ray

In [ ]:
green_check = "\033[92m\u2714\033[0m"
red_x = "\033[91m\u274C\033[0m"


#### Load File Details

In [ ]:
date = '4aug26' 
#All files will be saved in a folder with this date. Note: This folder will be made in the folder from
#which the jupyter notebook was launched.
os.system(f'mkdir -p {date}')

In [ ]:
# Make sure the model names are correct since these are what the file names have been called. If you want to rename
# the files, use script 'file_rename'
modelNames = [f'20240629_OS17lungInh_RM',
              f'20240703_OS17lungInh_RM',
              f'20241004_OS17lungInh2_RM',
              f'20241028_OS17lungFGFRi2_RM',
              f'20250629_OS17lungFGFRi2_RM',
              f'20250630_OS17lungFGFRi2_RM'] 

datapaths=['/home/groups/CEDAR/copperma/dyn-cell-sys/projects/rawan/mcl1i_28apr25/20240629_OS17_PCLS_R3W3M2_Mcl1',
           '/home/groups/CEDAR/copperma/dyn-cell-sys/projects/rawan/mcl1i_28apr25/20240703_OS17_PCLS_R3W3M2_N2_Mcl1/Controls',
           '/home/groups/CEDAR/copperma/dyn-cell-sys/projects/rawan/mcl1i_28apr25/20240810_PCLS_OS17_R6M1_inhset2_N1/20241216_segredo_last',
           '/home/groups/CEDAR/copperma/dyn-cell-sys/projects/rawan/mcl1i_28apr25/20241028_PCLS_OS17_FGFRi_Mcl1_combos2',
           '/home/groups/CEDAR/copperma/dyn-cell-sys/projects/rawan/mcl1i_28apr25/20250629_PCLS_Mcli50nM_Pazo_AZD_N1/AZD_files_for_death_analysis', #wasl elements
           '/home/groups/CEDAR/copperma/dyn-cell-sys/projects/rawan/mcl1i_28apr25/20250630_PCLS_Mcli50nM_Pazo_AZD_N2/AZD_files_for_death_analysis'] #was elements

datapaths_to_file_dict = {model: i for i, model in enumerate(modelNames)}

datapaths_to_file_dict = {model: i for i, model in enumerate(modelNames)}

#nxy = [30,24,23]# XY positions for each file
#sumnxy=sum(nxy)
#print(sumnxy)

In [ ]:
treatment_file = '/home/groups/CEDAR/copperma/dyn-cell-sys/rawan/mcl1i_28apr25/SITE_platemap_combined_combos_rep6_29oct25.xlsx'  # Update with the actual file path
treatment_df = pd.read_excel(treatment_file, header=None)

# Set the column names
treatment_df.columns = [ 'XY Position', 'Treatment','File']

tmSet = treatment_df['Treatment'].drop_duplicates().tolist()
nf=len(tmSet)

treatment_to_index = {treatment: index for index, treatment in enumerate(tmSet)}

# Step 3: Map the 'Treatment' column in the DataFrame to the corresponding indices
inds_tmSet = treatment_df['Treatment'].map(treatment_to_index)
inds_tmSet=inds_tmSet.tolist()

print("tmSet (from treatment_df):", tmSet)
print("Mapped indices:", inds_tmSet)
print(treatment_to_index)

In [ ]:
colorSet=['gray','blue','purple','red','magenta']

In [ ]:
modelList = []
inds_imagingSet = []
inds_imagingSet_models = []
inds_tmSet_models = []
indexedtreatment = []

# Iterate over the range of the number of rows in treatment_df
for iwell in range(len(treatment_df)):
    ixy = treatment_df.iloc[iwell]['XY Position'] - 1  # Use .iloc for row access by position
    modelname = treatment_df.iloc[iwell]['File']
    idatapath = datapaths_to_file_dict[modelname]
    mlist = imprep.list_images(f'{datapaths[idatapath]}/{modelname}_xy{ixy}_roi*.h5')

    for mstring in mlist:
        modelList.append(mstring)
        inds_imagingSet_models.append(idatapath)
        inds_tmSet_models.append(inds_tmSet[iwell])
        indexedtreatment.append(tmSet[inds_tmSet[iwell]])
        print(f'{green_check} found model {mstring} cond: {tmSet[inds_tmSet[iwell]]}')

# Convert lists to NumPy arrays
inds_imagingSet_models = np.array(inds_imagingSet_models)
inds_tmSet_models = np.array(inds_tmSet_models)
nmodels = len(modelList)
print(f'{nmodels} models found')


### load models and data

In [ ]:
#%%capture
nmodels=len(modelList)
modelSet=[None]*nmodels
indgood_models=np.array([]).astype(int)
for i in range(nmodels): #now go back through and try to load data
    try:
        modelSet[i]=trajectory.Trajectory(h5filename=modelList[i])
        sctm=modelSet[i]
        status=modelSet[i].load_from_h5('/cell_data_m0/')
        if hasattr(sctm,'fmsks_cancer'):
            del modelSet[i].fmsks_cancer; del modelSet[i].fmsks_primary #delete off these for memory conservation
        modelSet[i].get_image_shape()#some missing info, add in to metadata in future
        
        cyto_msk_channel=np.where(sctm.mask_channels==np.array(['cyto']).astype('S32'))[0][0]
        nuc_img_channel=np.where(sctm.channels==np.array(['DAPI']).astype('S32'))[0][0]
        rep_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
        #bf_img_channel=np.where(sctm.channels==np.array(['BF']).astype('S32'))[0][0]
        erk_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
        akt_img_channel=np.where(sctm.channels==np.array(['TRITC']).astype('S32'))[0][0]
        fra1_img_channel=np.where(sctm.channels==np.array(['CFP']).astype('S32'))[0][0]
        foreground_fmskchannel=np.where(sctm.fmask_channels==np.array(['foreground']).astype('S32'))[0][0]
        nuc_fmsk_channel=np.where(sctm.fmask_channels==np.array(['nuc']).astype('S32'))[0][0]
        mskchannel=cyto_msk_channel
        if status:
            ncells=sctm.cells_indSet.size
            print(f'loaded {sctm.h5filename} with {sctm.mask_channels} mask channels, {sctm.channels} image channels, cell index on {mskchannel} with {ncells} cells') 
        else:
            print(f'cell info on {mskchannel} not found') 
        if status:
            attr_name=f'img{erk_img_channel}_m{mskchannel}m{nuc_fmsk_channel}_ratio'
            if hasattr(sctm,attr_name):
                erkcn=getattr(sctm,attr_name)
                nerkcn=erkcn.size
                print(f'{green_check} ERK features, size {nerkcn}')
                if hasattr(sctm,f'img{akt_img_channel}_m{mskchannel}m{nuc_fmsk_channel}_ratio'):
                    print(f'{green_check} AKT features')
                    if hasattr(sctm,'linSet'):
                        print(f'{green_check} tracking data')
                        attr_name=f'boundary_library'
                        if hasattr(sctm,attr_name):
                            blib=getattr(sctm,attr_name)
                            maxid=np.max(blib['global_index'])
                            nblib=maxid+1
                            print(f'{green_check} boundary library, nindex {nblib}')
                            if hasattr(sctm,f'mm_properties'):
                                print(f'{green_check} boundary multipole features')
                                print(f'mm_properties shape: {sctm.mm_properties.shape}')
                                n_mm=sctm.mm_properties.shape[0]
                                nlist=[nerkcn,nblib,n_mm]
                                if all(x==nlist[0] for x in nlist):
                                    indgood_models=np.append(indgood_models,i)
                                    print(f'{green_check} n_erk: {nerkcn}, n_blib: {nblib}, n_mm: {n_mm}')
                                else:
                                    print(f'{red_x} missing features or index mismatch n_erk: {nerkcn}, n_blib: {nblib}, n_mm: {n_mm}')
    except Exception as e:
        print(e)
        pass

In [ ]:
#%%capture
#do a quick filtering on models on segmentation stability
#nframes=modelSet[0].nt
nframes=4
cellnumber_stdSet=np.ones(nmodels)*np.inf
cellnumber_frames=np.arange(int(nframes/2),nframes).astype(int)
cellnumber_std_cut=100.
meancellSet=np.zeros(nmodels)
for i in indgood_models:
    ncells=np.zeros(nframes)
    ncells_smooth=np.zeros_like(ncells)
    for iS in range(nframes):
        ncells[iS]=np.sum(modelSet[i].cells_frameSet==iS)
    print(f'model {i} mean cellnumber {np.nanmean(ncells)}')
    meancellSet[i]=np.nanmean(ncells)
    splfov=csaps.CubicSmoothingSpline(np.arange(nframes), ncells/np.nanmean(ncells[ncells>0][0:5]), smooth=5.e-2)
    ncells_smooth=splfov(np.arange(nframes))*ncells[0]
    cellnumber_std=np.std(ncells[cellnumber_frames]-ncells_smooth[cellnumber_frames])/np.mean(ncells[cellnumber_frames])
    cellnumber_stdSet[i]=cellnumber_std

indhigh_std=np.where(cellnumber_stdSet>cellnumber_std_cut)[0]
ind_enoughcells=np.where(meancellSet>=1.)[0]
indgood_models=np.intersect1d(indgood_models,ind_enoughcells)

In [ ]:
import bayesian_bootstrap
import csaps
nframes=modelSet[0].nt
timestep=.5
fw=3
cellnumber_stdSet=np.ones(nmodels)*np.inf
cellnumber_frames=np.arange(int(nframes/2),nframes).astype(int)
cellnumber_std_cut=.05
meancellSet=np.zeros(nmodels)
cellnumber_set=np.ones((nmodels,nframes))*np.nan
cellnumber_set_smooth=[None]*nmodels
for i in indgood_models:
    ncells=np.zeros(nframes)
    ncells_smooth=np.zeros_like(ncells)
    for iS in range(nframes):
        ncells[iS]=np.sum(modelSet[i].cells_frameSet==iS)
    cellnumber_set[i,:]=ncells
    print(f'model {i} mean cellnumber {np.nanmean(ncells)}')
    meancellSet[i]=np.nanmean(ncells)
    splfov=csaps.CubicSmoothingSpline(np.arange(nframes), ncells/ncells[0], smooth=5.e-2)
    cellnumber_set_smooth[i]=splfov
    ncells_smooth=splfov(np.arange(nframes))*np.nanmean(ncells[0:fw])
    cellnumber_std=np.std(ncells[cellnumber_frames]-ncells_smooth[cellnumber_frames])/np.mean(ncells[cellnumber_frames])
    cellnumber_stdSet[i]=cellnumber_std
    
cellnumber_set_norm=np.zeros_like(cellnumber_set)
for imodel in range(nmodels):
    cellnumber_set_norm[imodel,:]=np.divide(cellnumber_set[imodel,:],np.mean(cellnumber_set[imodel,0:3]))
    
cellnumber_set_norm[cellnumber_set_norm==np.inf]=np.nan
cellnumber_tmSet=np.zeros((nf,nframes))
cellnumber_bounds=np.zeros((nf,nframes,2))
#plt.figure()
#ax=plt.gca()
for i_tm in range(nf):
    plt.figure()
    ax=plt.gca()
    indmodels=np.where(inds_tmSet_models==i_tm)[0]
    indmodels=np.intersect1d(indmodels,indgood_models)
    cellnumber_tmSet[i_tm,:]=np.nanmean(cellnumber_set_norm[indmodels,:],axis=0)
    for iframe in range(nframes):
        uf=iframe+fw
        if uf>nframes:
            uf=nframes
        cellnumber_tmSet[i_tm,iframe]=np.nanmean(cellnumber_tmSet[i_tm,iframe:uf])
    for iframe in range(nframes):
        cns=cellnumber_set_norm[indmodels,iframe]
        cns=cns[np.isfinite(cns)]
        cns_boots=bayesian_bootstrap.mean(cns, n_replications=10000)
        cellnumber_bounds[i_tm,iframe,:]=bayesian_bootstrap.central_credible_interval(cns_boots, alpha=0.05) #bootstrap.get_CI(cns,1000)
    ax.plot(np.arange(nframes)*timestep,cellnumber_tmSet[i_tm,:],'o',color=colorSet[i_tm],label=tmSet[i_tm])
    spl=csaps.CubicSmoothingSpline(np.arange(nframes)*timestep, cellnumber_tmSet[i_tm,:], smooth=5.e-2)
    ax.plot(np.arange(nframes)*timestep,spl(np.arange(nframes)*timestep),'-',color=colorSet[i_tm],label=tmSet[i_tm])
    ax.fill_between(np.arange(nframes)*timestep,cellnumber_bounds[i_tm,:,0],cellnumber_bounds[i_tm,:,1],color=colorSet[i_tm],alpha=.1,label=f'{tmSet[i_tm]} 95% CI')
    plt.ylabel('relative cell number')
    plt.xlabel('time (hrs)')
    plt.ylim(0.3,1.2)
    plt.xlim(0,16)
    plt.legend(bbox_to_anchor=(1.5, 1.0))
    plt.savefig(f'{date}/{tmSet[i_tm]}_cellnumber.svg')

In [ ]:
data_columns=['treatment','roi_label','ncells']
n_tm=len(tmSet)
data_array=[]
for i_tm in range(n_tm):
    indmodels=np.where(inds_tmSet_models==i_tm)[0]
    for imodel in indmodels:
        sctm=modelSet[imodel]
        print(f'{tmSet[inds_tmSet_models[imodel]]} {sctm.figid.decode()} with {int(meancellSet[imodel]):d} cells')
        data_array.append(np.array([tmSet[inds_tmSet_models[imodel]],sctm.figid.decode(),int(meancellSet[imodel])],dtype=object))

data_array=np.array(data_array)
df=pd.DataFrame(data_array,columns=data_columns)
df.to_csv(f'{date}/SITE_methods_cellnumbers.csv')

In [ ]:
#choose features to keep
sctm=modelSet[indgood_models[0]]
indkeep_features=[]
for ifeat in range(sctm.mm_property_names.size):
    pname=sctm.mm_property_names[ifeat].decode()
    if 'dx' in pname or 'curvatures' in pname or 'shapes' in pname or 'radius' in pname or 'matrix' in pname or 'l=2' in pname:
        print(f'excluding {pname}')
    else:
        indkeep_features.append(ifeat)

print('\n')
indkeep_features=np.array(indkeep_features).astype(int)
for ifeat in indkeep_features:
    print(f'keeping {sctm.mm_property_names[ifeat].decode()}')

In [ ]:
#extract features
Xf_feature_list=np.array(modelSet[indgood_models[0]].mm_property_names)[indkeep_features]
nfeat=indkeep_features.size
Xf=np.zeros((0,nfeat))
indtreatment=np.array([]).astype(int)
indcellSet=np.array([]).astype(int)
indcells_models=np.array([]).astype(int)
indframes_models=np.array([]).astype(int)
for i in indgood_models:
    sctm=modelSet[i]
    Xf_model=sctm.mm_properties[:,indkeep_features].copy()
    for ifeat in range(Xf_model.shape[1]):
        Xf_model[:,ifeat]=imprep.znorm(Xf_model[:,ifeat])
    sctm.Xf=Xf_model
    maxl=np.max(sctm.boundary_library['global_index'])
    print(f'model {i}, libindex: {maxl+1}, mm: {modelSet[i].mm_properties.shape[0]}, ncells {sctm.cells_indSet.size}')
    Xf=np.append(Xf,Xf_model,axis=0)
    indtreatment=np.append(indtreatment,inds_tmSet_models[i]*np.ones(modelSet[i].cells_indSet.size).astype(int))
    indcellSet=np.append(indcellSet,modelSet[i].cells_indSet)
    indcells_models=np.append(indcells_models,i*np.ones(modelSet[i].cells_indSet.size).astype(int))
    indframes_models=np.append(indframes_models,modelSet[i].cells_frameSet)

In [ ]:
#run this for centroid mobility
for i in indgood_models:
    modelSet[i].dx=np.zeros(modelSet[i].cells_indSet.size)
    modelSet[i].dxv=np.zeros((modelSet[i].cells_indSet.size,modelSet[i].ndim))
    for icell in range(modelSet[i].cells_indSet.size):
        modelSet[i].dx[icell]=np.linalg.norm(modelSet[i].get_dx(icell)*[sctm.zscale,1.,1.])*sctm.micron_per_pixel
        modelSet[i].dxv[icell,:]=modelSet[i].get_dx(icell)

In [ ]:
#%%capture
#signaling features
cratio_erk=np.array([]).astype(float)
for i in indgood_models:
    sctm=modelSet[i]
    nuc_img_channel=np.where(sctm.channels==np.array(['DAPI']).astype('S32'))[0][0]
    rep_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
    erk_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
    akt_img_channel=np.where(sctm.channels==np.array(['TRITC']).astype('S32'))[0][0]
    attr_str=f'img{erk_img_channel}_m{cyto_msk_channel}m{nuc_fmsk_channel}_ratio'
    cc=getattr(sctm,attr_str)
    cratio_erk=np.append(cratio_erk,cc)
    if cc.size != sctm.cells_indSet.size:
        print(f'{sctm.h5filename} c/n erk: {cc.size}, feat: {sctm.cells_indSet.size}')

cratio_erk_std=np.nanstd(cratio_erk);cratio_erk_mean=np.nanmean(cratio_erk);#cratio_akt=imprep.znorm(cratio_akt)

for i in indgood_models:
    sctm=modelSet[i]
    nuc_img_channel=np.where(sctm.channels==np.array(['DAPI']).astype('S32'))[0][0]
    rep_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
    erk_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
    akt_img_channel=np.where(sctm.channels==np.array(['TRITC']).astype('S32'))[0][0]
    attr_str=f'img{erk_img_channel}_m{cyto_msk_channel}m{nuc_fmsk_channel}_ratio'
    cc=getattr(sctm,attr_str)
    cratio_z_model=cc #(cc.T-cratio_erk_mean)/cratio_erk_std
    modelSet[i].x_signal=cratio_z_model.copy()
    #modelSet[i].save_to_h5('/cell_data_m0/',['x_signal'],overwrite=True)

In [ ]:
#%%capture
#signaling features
cratio_akt=np.array([]).astype(float)
for i in indgood_models:
    sctm=modelSet[i]
    nuc_img_channel=np.where(sctm.channels==np.array(['DAPI']).astype('S32'))[0][0]
    rep_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
    erk_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
    akt_img_channel=np.where(sctm.channels==np.array(['TRITC']).astype('S32'))[0][0]
    attr_str=f'img{akt_img_channel}_m{cyto_msk_channel}m{nuc_fmsk_channel}_ratio'
    cc=getattr(sctm,attr_str)
    cratio_akt=np.append(cratio_akt,cc)

cratio_akt_std=np.nanstd(cratio_akt);cratio_akt_mean=np.nanmean(cratio_akt);#cratio_akt=imprep.znorm(cratio_akt)

for i in indgood_models:
    sctm=modelSet[i]
    nuc_img_channel=np.where(sctm.channels==np.array(['DAPI']).astype('S32'))[0][0]
    rep_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
    erk_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
    akt_img_channel=np.where(sctm.channels==np.array(['TRITC']).astype('S32'))[0][0]
    attr_str=f'img{akt_img_channel}_m{cyto_msk_channel}m{nuc_fmsk_channel}_ratio'
    cc=getattr(sctm,attr_str)
    cratio_z_model=cc #(cc.T-cratio_akt_mean)/cratio_akt_std
    modelSet[i].x_signal_akt=cratio_z_model.copy()

In [ ]:
timestep_imagingSet=[.5,.5,.5,.5,.5,.5]
initialframe_imagingSet=[0,0,50,0,0,0]
finalframe_imagingSet=[40,40,90,40,40,40]
initial_frames_model=[None]*nmodels
final_frames_model=[None]*nmodels
timestep_model=[None]*nmodels
for imodel in range(nmodels):
    i_imagingSet=inds_imagingSet_models[imodel]
    initial_frames_model[imodel]=initialframe_imagingSet[i_imagingSet]
    final_frames_model[imodel]=finalframe_imagingSet[i_imagingSet]
    timestep_model[imodel]=timestep_imagingSet[i_imagingSet]

In [ ]:
times_models=np.zeros(indframes_models.size)
rep_models=np.zeros(indframes_models.size)
for imodel in indgood_models:
    start_frame=initial_frames_model[imodel]
    timestep=timestep_model[imodel]
    indmodel=np.where(indcells_models==imodel)[0]
    times_model=timestep*(indframes_models[indmodel]-start_frame)
    times_models[indmodel]=times_model
    rep_models[indmodel]=inds_imagingSet_models[imodel]
    sctm=modelSet[imodel]
    sctm.cells_timeSet=times_model.copy()
    print(f'rep {inds_imagingSet_models[imodel]} model {imodel} nframes {np.max(sctm.cells_frameSet)} start {start_frame} {sctm.figid.decode()}')

In [ ]:
#%%capture
#signaling features
fra1_nuc=np.array([]).astype(float)
for i in indgood_models:
    sctm=modelSet[i]
    attr_str=f'nucintensity_fra1'
    cc=getattr(sctm,attr_str)
    fra1_nuc=np.append(fra1_nuc,cc)

indstw=np.where(np.logical_and(times_models>0.,times_models<1.))[0]
fra1_nuc_std=np.nanstd(fra1_nuc[indstw]);fra1_nuc_mean=np.nanmean(fra1_nuc[indstw]);#cratio_akt=imprep.znorm(cratio_akt)

fra1_nuc=np.array([]).astype(float)
for i in indgood_models:
    print(i)
    sctm=modelSet[i]
    attr_str=f'nucintensity_fra1'
    cc=getattr(sctm,attr_str)
    indstw=np.where(np.logical_and(sctm.cells_timeSet>0.,sctm.cells_timeSet<1.))[0]
    fra1_nuc_std=np.nanstd(cc[indstw])
    fra1_nuc_mean=np.nanmean(cc[indstw]);
    fra1_nuc_z_model=(cc.T-fra1_nuc_mean)/fra1_nuc_std
    fra1_nuc=np.append(fra1_nuc,fra1_nuc_z_model)
    modelSet[i].x_signal_fra1=fra1_nuc_z_model.copy()

In [ ]:
#This tells us the comparison between C/N ratio and the cross correlation variable. they should correlate 1-1.
plt.figure(figsize=(8,7))
plt.scatter(cratio_erk,fra1_nuc,s=1,c='black',marker='.',alpha=.3)
plt.xlabel('c/n ratio')
plt.ylabel('fra-1')
plt.xlim(0.,2.)
plt.ylim(-4,4)

plt.savefig(os.path.join(date, f'ERK_ratio_fra1_comp.png'))
plt.savefig(os.path.join(date, f'ERK_ratio_fra1_comp.svg'), format='svg')

### ERK and AKT Plots

In [ ]:
nplots=int(np.ceil(np.sqrt(len(tmSet))))
fig,ax=plt.subplots(nplots,nplots,sharex=True,sharey=True,figsize=(12,12))
ax=ax.flatten()
xlow=0;xhigh=2;ylow=0.5;yhigh=1.5
for i_tm in range(len(tmSet)):
    inds=np.where(indtreatment==i_tm)[0]
    print(f'{tmSet[i_tm]}, {inds.size} measurements')
    ax[i_tm].scatter(cratio_erk[inds],cratio_akt[inds],s=10,c=indframes_models[inds],cmap=plt.cm.cividis,marker='.',label=tmSet[i_tm],alpha=.2)
    ax[i_tm].plot([xlow,xhigh],[1,1],'k--',linewidth=.5)
    ax[i_tm].plot([1,1],[ylow,yhigh],'k--',linewidth=.5)
    ax[i_tm].set_xlim(xlow,xhigh)
    ax[i_tm].set_ylim(ylow,yhigh)
    ax[i_tm].legend(loc='upper right')
    ax[i_tm].set_xlabel('ERK activity')
    ax[i_tm].set_ylabel('AKT activity')

# Hide any empty subplots
for i in range(len(tmSet), len(ax)):
    fig.delaxes(ax[i])

plt.tight_layout()
plt.savefig(os.path.join(date, f'signaling_cratio_scatterplot.png'))
plt.savefig(os.path.join(date, f'signaling_cratio_scatterplot.svg'), format='svg')
plt.show()

### Signaling trajectory analysis

In [ ]:
#Map all trajectories
all_trajSet=[None]*nmodels
for i in indgood_models:
    modelSet[i].get_unique_trajectories()
    all_trajSet[i]=modelSet[i].trajectories.copy()

In [ ]:
#set cutoff between ERK-low and ERK-high
lowcut=0.85
highcut=1.15
midcut=.5*highcut+.5*lowcut
print(midcut)

In [ ]:
#signal environment feature
scut=midcut
sigState=np.array([]).astype(int)
for i in indgood_models:
    sigState=np.append(sigState,modelSet[i].x_signal>scut)
    modelSet[i].sigState=modelSet[i].x_signal>scut

In [ ]:
def get_death_signal_contributions(self,inds_death,time_lags=None,x_pos=None,rmax=1000.,R=None,zscale=None,rescale_z=False):
    """
    Computes the spatial contributions of signaling between cells over a specified time lag. This method
    averages signals from nearby cells, weighted inversely by their distances, to assess local signaling interactions.

    Parameters
    ----------
    S : ndarray
        A binary array indicating the signaling status of cells (1 for active, 0 for inactive).
    time_lag : int, optional
        The time lag over which to assess signal contributions, defaulting to 0 for immediate interactions.
    x_pos : ndarray, optional
        Positions of cells. If None, the positions are taken from the instance's `x` attribute.
    rmax : float, optional
        The maximum radius within which to consider signal contributions from neighboring cells, default is 5.
    R : float, optional
        Normalization radius, typically set to the average cell diameter; defaults to the instance’s `cellpose_diam`.
    zscale : float, optional
        The scaling factor for the z-dimension, used if `rescale_z` is True.
    rescale_z : bool, optional
        If True, scales the z-coordinates of positions by `zscale`.

    Returns
    -------
    S_r : ndarray
        An array where each element is the averaged spatial signal contribution received by each cell, normalized
        by distance and weighted by the signaling status of neighboring cells.

    Examples
    --------
    >>> traj = Trajectory('path/to/data.h5')
    >>> S = np.random.randint(0, 2, size=traj.cells_indSet.size)
    >>> signal_contributions = traj.get_signal_contributions(S, time_lag=1, rmax=10, R=15)
    # This computes the signal contributions for each cell, considering interactions within a radius of 10 units.

    Notes
    -----
    - This method is useful for understanding the influence of cell-cell interactions within a defined spatial range
    and can be particularly insightful in dynamic cellular environments where signaling is a key factor.
    - The distances are normalized by the cell radius `R` to provide a relative measure of proximity, and the contributions
    are weighted by the inverse of these normalized distances.

    Raises
    ------
    ValueError
        If necessary parameters are missing or incorrectly formatted.
    """
    #S needs to be indexed so S[cell_inds] gives the correct binary signal, same with x_pos
    cellinds=np.arange(self.cells_indSet.size).astype(int)
    S=np.isin(cellinds,self.inds_death)
    if x_pos is None:
        x_pos=self.x.copy()
        if rescale_z:
            x_pos=np.multiply(x_pos,np.array([self.zscale,1.,1.]))
    if R is None:
        R=self.cellpose_diam
    if time_lags is None:
        time_lags=np.arange(self.nt).astype(int)
    nlags=time_lags.size
    S_r=np.ones((S.size,nlags))*np.nan #this will be the average spatial signal (S/r)
    for ilag in range(nlags):
        time_lag=time_lags[ilag]
        traj_pairSet=self.get_traj_segments(time_lag+1)
        if time_lag==0:
            traj_pairSet=np.concatenate((traj_pairSet,traj_pairSet),axis=1)
        else:
            traj_pairSet=np.concatenate((traj_pairSet[:,[0]],traj_pairSet[:,[-1]]),axis=1)
        indimgs=np.unique(self.cells_indimgSet[traj_pairSet[:,0]])
        for im in indimgs:
            cell_inds_img1=np.where(self.cells_indimgSet[traj_pairSet[:,0]]==im)[0]
            indcomm_ctraj1=traj_pairSet[cell_inds_img1,0]
            indcomm_ctraj2=traj_pairSet[cell_inds_img1,1]
            x_pos1=x_pos[indcomm_ctraj1,:]
            x_pos2=x_pos[indcomm_ctraj2,:]
            S1=S[indcomm_ctraj1] #signaling status at pair timepoints
            S2=S[indcomm_ctraj2]
            dmatr=utilities.get_dmat(x_pos1,x_pos2)/R #distance at time-lag
            for j in range(indcomm_ctraj1.size):
                d_r=dmatr[j,:]
                inds=np.where(np.logical_and(d_r>0.,d_r<rmax))[0]
                S_r[indcomm_ctraj1[j]]=np.sum(np.divide(S2[inds],d_r[inds]))
    S_r=np.nansum(S_r,axis=1)
    S_r=S_r/nlags
    return S_r

In [ ]:
S_r_death=[]
for i in indgood_models:
    sctm=modelSet[i]
    S_r=get_death_signal_contributions(sctm,sctm.inds_death,time_lags=np.arange(16).astype(int),x_pos=None,rmax=1000.,R=None,zscale=None,rescale_z=True)
    sctm.S_r_death=S_r.copy()
    sctm.S_r=S_r.copy()
    S_r_death.append(S_r)

S_r_death=np.concatenate(S_r_death)

In [ ]:
for i in indgood_models:
   indsf=np.where(indtreatment==i)[0]
   modelSet[i].Xpca=np.concatenate((modelSet[i].S_r[:,np.newaxis],modelSet[i].x_signal[:,np.newaxis]),axis=1)

In [ ]:
trajl=1
ndim=modelSet[indgood_models[0]].Xpca.shape[1]
Xpcat=np.zeros((0,ndim*trajl))
indtreatment_traj=np.array([]).astype(int)
indstack_traj=np.array([]).astype(int)
indframes_traj=np.array([]).astype(int)
cellinds0_traj=np.array([]).astype(int)
cellinds1_traj=np.array([]).astype(int)
corrc_traj=np.array([]).astype(float)
z_traj=np.array([]).astype(float)
corrc_full_traj=np.zeros((0,trajl))
cancer_cf_full_traj=np.zeros((0,trajl))
primary_cf_full_traj=np.zeros((0,trajl))
dx_full_traj=np.zeros((0,trajl))
dx_traj=np.array([])
SR_full_traj=np.zeros((0,trajl))
SR_traj=np.array([])
for i in indgood_models:
    print('building trajectory data for model {}...'.format(i))
    modelSet[i].trajectories=all_trajSet[i].copy()
    modelSet[i].trajl=trajl
    modelSet[i].traj=modelSet[i].get_traj_segments(trajl)
    data=modelSet[i].Xpca[modelSet[i].traj,:]
    #datacom=modelSet[i].Xf_com[modelSet[i].traj,:]
    data=data.reshape(modelSet[i].traj.shape[0],modelSet[i].Xpca.shape[1]*trajl)
    #datacom=datacom.reshape(modelSet[i].traj.shape[0],modelSet[i].Xf_com.shape[1]*trajl)
    #data=np.append(data,datacom,axis=1)
    indgood=np.where(np.sum(np.isnan(data),axis=1)==0)[0]
    data=data[indgood,:]
    modelSet[i].traj=modelSet[i].traj[indgood,:]
    Xpcat=np.append(Xpcat,data,axis=0)
    indtreatment_traj=np.append(indtreatment_traj,i*np.ones(data.shape[0]).astype(int))
    indstacks=modelSet[i].cells_imgfileSet[modelSet[i].traj[:,0]]
    indstack_traj=np.append(indstack_traj,indstacks)
    indframes=modelSet[i].cells_frameSet[modelSet[i].traj[:,0]]
    indframes_traj=np.append(indframes_traj,indframes)
    cellinds0=modelSet[i].traj[:,0]
    cellinds0_traj=np.append(cellinds0_traj,cellinds0)
    cellinds1=modelSet[i].traj[:,-1]
    cellinds1_traj=np.append(cellinds1_traj,cellinds1)
    z_traj=np.append(z_traj,np.mean(modelSet[i].cellblocks[:,0],axis=1)[cellinds1])
    dx_traj=np.append(dx_traj,modelSet[i].dx[cellinds1])
    dx_full_traj=np.append(dx_full_traj,modelSet[i].dx[modelSet[i].traj],axis=0)
    SR_traj=np.append(SR_traj,modelSet[i].S_r[:,np.newaxis][cellinds1,0])
    SR_full_traj=np.append(SR_full_traj,modelSet[i].S_r[:,np.newaxis][modelSet[i].traj,0],axis=0)

cellinds0_traj=cellinds0_traj.astype(int)
cellinds1_traj=cellinds1_traj.astype(int)

In [ ]:
x=Xpcat #x=np.matmul(Xpcat,np.real(M)) #x=np.concatenate((np.real(np.matmul(Xpcat,M)),np.imag(np.matmul(Xpcat,M))),axis=1) #x=np.concatenate((np.real(phi_X[:,indmodes_M1]),np.imag(phi_X[:,indmodes_M1])),axis=1)
for i in indgood_models:
    modelSet[i].trajectories=all_trajSet[i].copy()

neigen=x.shape[1]
inds_conditions=[None]*nf
for imf in range(nf):
    indmodels=np.intersect1d(indgood_models,np.where(inds_tmSet_models==imf)[0])
    indstm=np.array([])
    for imodel in indmodels:
        indtm=np.where(indtreatment_traj==imodel)
        indstm=np.append(indstm,indtm)
    inds_conditions[imf]=indstm.astype(int).copy()

neigen=x.shape[1]
inds_conditions=[None]*nf
for imf in range(nf):
    indmodels=np.intersect1d(indgood_models,np.where(inds_tmSet_models==imf)[0])
    indstm=np.array([])
    for imodel in indmodels:
        indtm=np.where(indtreatment_traj==imodel)
        indstm=np.append(indstm,indtm)
    inds_conditions[imf]=indstm.astype(int).copy()

#get transitions in the trajectory embedding space
x0set=np.zeros((0,neigen))
x1set=np.zeros((0,neigen))
inds_trajsteps_models=np.array([]).astype(int)
inds_trajsteps=np.zeros((0,2)).astype(int)
for i in indgood_models:
    print('getting flows from model: '+str(i))
    indstm=np.where(indtreatment_traj==i)[0]
    if indstm.size>0:
        modelSet[i].Xtraj=x[indstm,0:neigen]
        indstm_model=indstm-np.min(indstm) #index in model
        modelSet[i].get_trajectory_steps(inds=None,get_trajectories=False,traj=modelSet[i].traj[indstm_model,:],Xtraj=modelSet[i].Xtraj[indstm_model,:])
        x0=modelSet[i].Xtraj0
        x1=modelSet[i].Xtraj1
        x0set=np.append(x0set,x0,axis=0)
        x1set=np.append(x1set,x1,axis=0)
        inds_trajsteps_models=np.append(inds_trajsteps_models,np.ones(x0.shape[0]).astype(int)*i)
        inds_trajsteps=np.append(inds_trajsteps,modelSet[i].inds_trajp1,axis=0)

In [ ]:
def get_signal_death_rdf(self,S,inds_death=None,fl=None,fu=None,time_lag=0,x_pos=None,cell_inds=None,rbins=None,nr=8,rmax=100,zscale=None,rescale_z=False):
    #S needs to be indexed so S[cell_inds] gives the correct binary signal, same with x_pos
    if inds_death is None:
        inds_death=self.inds_death
    if cell_inds is None:
        cell_inds=np.arange(self.cells_indSet.size).astype(int)
    if x_pos is None:
        x_pos=self.x
    if rescale_z:
        x_pos=np.multiply(x_pos,np.array([self.zscale,1.,1.]))
    if rbins is None:
        rbins=np.linspace(0.,rmax,nr)
    nr=rbins.size
    traj_pairSet=self.get_traj_segments(time_lag+1)
    if time_lag==0:
        traj_pairSet=np.concatenate((traj_pairSet,traj_pairSet),axis=1)
    else:
        traj_pairSet=np.concatenate((traj_pairSet[:,[0]],traj_pairSet[:,[-1]]),axis=1)
    paircorr=np.zeros(nr)
    norm=np.zeros(nr)
    indimgs=np.unique(self.cells_indimgSet[traj_pairSet[:,0]])
    if fl is not None:
        frames=np.arange(fl,fu+1).astype(int)
        indimgs=np.intersect1d(frames,indimgs)
    for im in indimgs:
        cell_inds_img1=np.where(self.cells_indimgSet[traj_pairSet[:,0]]==im)[0]
        indcomm_ctraj1=traj_pairSet[cell_inds_img1,0]
        indcomm_ctraj2=traj_pairSet[cell_inds_img1,1]
        inds_on1=np.where(np.isin(cell_inds_img1,inds_death))[0] #cells marked as about to die
        #inds_on1=np.where(np.isin(cell_inds_img1,indcomm_ctraj1))[0] #cells marked as about to die
        #inds_on1=np.where(S[indcomm_ctraj1]>0)[0] #on at first time
        #x_pos1=x_pos[indcomm_ctraj1[inds_on1],:] #only look at cells which are "on"
        x_pos1=x_pos[cell_inds_img1[inds_on1],:] #only look at cells which are "on"
        x_pos2=x_pos[indcomm_ctraj2,:] #any cell, lagged timepoint
        #S1=S[indcomm_ctraj1[inds_on1]] #signaling at first timepoint
        S1=S[cell_inds_img1[inds_on1]] #signaling at first timepoint
        S1=np.ones_like(S1) #here, don't care about signaling for cell that died
        S2=S[indcomm_ctraj2] #signaling timelagged
        Smat=np.matmul(S1[:,np.newaxis],np.transpose(S2[:,np.newaxis]))
        dmatr=utilities.get_dmat(x_pos1,x_pos2) #distance at time 0
        indr=np.digitize(dmatr,rbins)
        for ir in range(1,nr):
            indcells_dr=np.where(indr==ir)
            if indcells_dr[0].size>0:
                #paircorr[ir]=paircorr[ir]+np.sum(S1[indcells_dr[0]]*S2[indcells_dr[1]])/indcells_dr[0].size
                paircorr[ir]=paircorr[ir]+np.sum(Smat[indcells_dr])/indcells_dr[0].size
                norm[ir]=norm[ir]+1
    rbins=rbins[1:]
    paircorr=paircorr[1:]
    norm=norm[1:]
    #print(f'tlag: {time_lag} norm: {norm}')
    for ir in range(nr-1):
        paircorr[ir]=paircorr[ir]/norm[ir]
    return rbins,paircorr

In [ ]:
nr=12;rmax=80.0/sctm.micron_per_pixel #5.6*sctm.cellpose_diam;
nt=20
scorr=np.zeros((nmodels,nt,nr-1))
scorr_mean_dt=np.zeros((nmodels,nt))
for i in indgood_models:
    print(f'calculation model {i}')
    for itt in range(nt):
        print(f' tlag {itt}')
        try:
            fl=initial_frames_model[i]
            fu=final_frames_model[i]
            sctm=modelSet[i]
            rbins,scorr[i,itt,:]=get_signal_death_rdf(sctm,sctm.sigState,fl=fl,fu=fu,time_lag=itt,nr=nr,rmax=rmax,zscale=sctm.zscale,rescale_z=True)
            inds_window=np.where(np.logical_and(sctm.cells_frameSet>fl+itt,sctm.cells_frameSet<fu))[0]
            scorr_mean_dt[i,itt]=np.nanmean(sctm.sigState[inds_window])
        except:
            print(f'fail at m{i}')

scorr_av=np.nanmean(scorr,axis=0)

In [ ]:
figid='SITE_OS17'

In [ ]:
sc.shape

In [ ]:
min_m=15
import bayesian_bootstrap
framebins=np.array([0,4,8,12,16]).astype(int) #,20]).astype(int)
rbins_mu=sctm.micron_per_pixel*np.insert(rbins[0:-1],0,0.)
rbins_mu_smth=np.linspace(np.min(rbins_mu),np.max(rbins_mu),101)
for i_tm in range(nf):
    print(tmSet[i_tm]+'\n\n\n')
    fig,axs=plt.subplots(framebins.size-1,1,sharex=True,figsize=(6,2*(framebins.size-1)))
    indmodels=np.where(inds_tmSet_models==i_tm)[0]
    indmodels=np.intersect1d(indmodels,indgood_models)
    axs=axs.flatten()
    axs=np.flip(axs)
    sigState=[]
    for imodel in indmodels:
        sctm=modelSet[imodel]
        sigState.append(sctm.sigState[np.isfinite(sctm.x_signal)])
    sigState=np.concatenate(sigState)
    meanS=np.nanmean(sigState)
    for ibin in range(0,framebins.size-1):
        sc_bin=np.zeros((nr-1,3))
        ax=axs[ibin]
        iframe=framebins[ibin]
        lf=iframe
        uf=framebins[ibin+1]
        scr=scorr[indmodels,lf:uf,:]
        for ir in range(nr-1):
            sc=scr[...,ir]
            sc=sc[np.isfinite(sc)]
            sc_boots = bayesian_bootstrap.mean(sc, n_replications=10000)
            if sc.size>=min_m:
                sc_bin[ir,0]=np.nanmean(sc)
                sc_bin[ir,1:]=bayesian_bootstrap.central_credible_interval(sc_boots, alpha=0.05) #bootstrap.get_CI(sc,1000)
        ind=np.isfinite(sc_bin[:,0])
        spl=csaps.CubicSmoothingSpline(rbins_mu[ind], sc_bin[ind,0], smooth=10.e-2)
        sc_bin_smooth=spl(rbins_mu_smth)
        ax.plot(rbins_mu_smth,sc_bin_smooth,'k-')
        ax.scatter(rbins_mu,sc_bin[:,0],s=20,c='black',alpha=.5)
        ax.fill_between(rbins_mu,sc_bin[:,1],sc_bin[:,2],color='black',alpha=.2)
        #ax.plot(sctm.micron_per_pixel*np.array([0,np.max(rbins)]),[0.,0.],'k--',linewidth=.5)
        #ax.plot(sctm.micron_per_pixel*np.array([0,np.max(rbins)]),[meanS,meanS],'k--',linewidth=.5)
        ax.set_ylim(0,1)
        ax.set_ylabel(f'joint ERK-on at {iframe*.5:.1f} hours')
        if ibin==0:
            ax.set_xlabel(r'separation ($\mu$m)')
    plt.tight_layout()
    plt.savefig(f'{date}/scERK_death_{figid}_{tmSet[i_tm]}.png')
    plt.savefig(f'{date}/scERK_death_{figid}_{tmSet[i_tm]}.svg')
    plt.show()

In [ ]:
n_tm=len(tmSet)
fig,axs=plt.subplots(3,2,figsize=(13,19.5))
axs=axs.flatten()
timestep=.5
tms=[0,1,2,3,4]
for iplot in range(len(tms)):
    i_tm=tms[iplot]
    indmodels=np.where(inds_tmSet_models==i_tm)[0]
    indmodels=np.intersect1d(indmodels,indgood_models)
    scorr_av=np.nanmean(scorr[indmodels,...],axis=0)
    S=[]
    for imodel in indmodels:
        S.append(modelSet[imodel].x_signal)
    S=np.concatenate(S)
    scorr_av[np.logical_not(np.isfinite(scorr_av))]=0.
    scorr_av_smth=scipy.ndimage.gaussian_filter(scorr_av,1.,mode='reflect')
    #plt.figure(figsize=(5,7))
    ax=axs[iplot]
    ax.imshow(scorr_av_smth,origin='lower',cmap=plt.cm.hot,clim=(0.18,.8))
    #ax.set_clim(0,1.)
    #cbar=plt.colorbar()
    #cbar.set_label('ERK-active joint correlation')
    #ax=plt.gca()
    ax.set(xticks=np.linspace(0, nr-2, int(nr/2)), xticklabels=["%0d" % x for x in (np.round(.1*np.linspace(0, rmax*sctm.micron_per_pixel, int(nr/2)))*10).astype(int)], yticks=np.linspace(0,nt-1,int(nt/2)), yticklabels=["%.0f" % x for x in timestep*np.linspace(0,nt-1,int(nt/2))]);
    ax.set_xlabel('distance (microns)')
    ax.set_ylabel('time (hrs)')
    #plt.ylim(0,15);plt.clim(40,60)
    ax.set_title(f'{tmSet[i_tm]} ERK')

plt.savefig(f'{date}/erk_on_death_spatiotemporal_all.png')
plt.savefig(f'{date}/erk_on_death_spatiotemporal_all.svg')
plt.show()

In [ ]:
plt.imshow(scorr_av_smth,origin='lower',cmap=plt.cm.hot,clim=(0.18,.8))
cbar=plt.colorbar()
cbar.set_label('ERK active probability')
plt.savefig(f'{date}/ERK_spatiotemporal_colorbar.png')
plt.savefig(f'{date}/ERK_spatiotemporal_colorbar.svg')

In [ ]:
scorr_null2=np.ones((nmodels,nt,nr-1))*np.nan
nrand=1
for i in indgood_models:
    print(f'calculation model {i}')
    fl=initial_frames_model[i]
    fu=final_frames_model[i]
    for itt in range(nt):
        print(f' tlag {itt}')
        if True: #try:
            sctm=modelSet[i]
            inds_death_orig=sctm.inds_death.copy()
            if np.isscalar(inds_death_orig):
                inds_death_orig=np.array([inds_death_orig]).astype(int)
            sc=np.ones((nrand,nr-1))*np.nan
            for irand in range(nrand):
                inds_death=np.zeros_like(inds_death_orig)
                for ideath in range(inds_death_orig.size):
                    inds=np.where(sctm.cells_frameSet[inds_death_orig[ideath]])[0]
                    inds_death[ideath]=np.random.choice(inds)
                np.random.shuffle(x_pos)
                rbins,sc[irand,:]=get_signal_death_rdf(sctm,sctm.sigState,inds_death=inds_death,x_pos=None,fl=fl,fu=fu,time_lag=itt,nr=nr,rmax=rmax,zscale=sctm.zscale,rescale_z=True)
            #rbins,scorr_null[i,itt,:]=get_signal_rdf(sctm,sctm.sigState,time_lag=itt,x_pos=x_pos,nr=nr,rmax=rmax,zscale=sctm.zscale,rescale_z=True)
            #rbins,scorr_null[i,itt,:]=get_signal_death_rdf(sctm,sctm.sigState,x_pos=x_pos,fl=fl,fu=fu,time_lag=itt,nr=nr,rmax=rmax,zscale=sctm.zscale,rescale_z=True)
            scorr_null2[i,itt,:]=np.nanmean(sc,axis=0)
        #except:
        #    print(f'fail at m{i}')

In [ ]:
n_tm=len(tmSet)
fig,axs=plt.subplots(3,2,figsize=(13,19.5))
axs=axs.flatten()
timestep=.5
tms=[0,1,2,3,4]
for iplot in range(len(tms)):
    i_tm=tms[iplot]
    indmodels=np.where(inds_tmSet_models==i_tm)[0]
    indmodels=np.intersect1d(indmodels,indgood_models)
    scorr_av=np.nanmean(scorr_null2[indmodels,...],axis=0)
    S=[]
    for imodel in indmodels:
        S.append(modelSet[imodel].x_signal)
    S=np.concatenate(S)
    scorr_av[np.logical_not(np.isfinite(scorr_av))]=0.
    scorr_av_smth=scipy.ndimage.gaussian_filter(scorr_av,1.,mode='reflect')
    #plt.figure(figsize=(5,7))
    ax=axs[iplot]
    ax.imshow(scorr_av_smth,origin='lower',cmap=plt.cm.hot,clim=(0.18,.8))
    #ax.set_clim(0,1.)
    #cbar=plt.colorbar()
    #cbar.set_label('ERK-active joint correlation')
    #ax=plt.gca()
    ax.set(xticks=np.linspace(0, nr-2, int(nr/2)), xticklabels=["%0d" % x for x in (np.round(.1*np.linspace(0, rmax*sctm.micron_per_pixel, int(nr/2)))*10).astype(int)], yticks=np.linspace(0,nt-1,int(nt/2)), yticklabels=["%.0f" % x for x in timestep*np.linspace(0,nt-1,int(nt/2))]);
    ax.set_xlabel('distance (microns)')
    ax.set_ylabel('time (hrs)')
    #plt.ylim(0,15);plt.clim(40,60)
    ax.set_title(f'{tmSet[i_tm]} ERK')

plt.savefig(f'{date}/erk_on_death_spatiotemporal_all_null2.png')
plt.savefig(f'{date}/erk_on_death_spatiotemporal_all_null2.svg')
plt.show()

In [ ]:
SR_clusters

In [ ]:
#modeling with SR
import sklearn.cluster
n_SR=4
SRmax=8.
Slow=lowcut
Shigh=highcut
#SR_clusters=sklearn.cluster.KMeans(n_clusters=n_SR,init='k-means++',random_state=0).fit(SR_traj[SR_traj<SRmax,np.newaxis])
#SR_clusters=np.sort(SR_clusters.cluster_centers_[:,0])
SR_clusters=np.array([0.01432603, 0.83945825, 3.31683432, 6.02338035])
clustercenters0=np.concatenate((SR_clusters[:,np.newaxis],Slow*np.ones_like(SR_clusters)[:,np.newaxis]),axis=1)
clustercenters1=np.concatenate((SR_clusters[:,np.newaxis],Shigh*np.ones_like(SR_clusters)[:,np.newaxis]),axis=1)
clustercenters=np.append(clustercenters0,clustercenters1,axis=0)
n_clusters=clustercenters.shape[0]
clusters=coor.clustering.AssignCenters(clustercenters, metric='euclidean')

#plot umap of the embedding space
plt.figure()
plt.scatter(x[:,0],x[:,1],s=20,c=x[:,1],cmap=plt.cm.seismic,clim=(Slow,Shigh))
plt.ylim(0.,3)
ind_clusters=clusters.assign(x)
x_clusters=np.zeros((n_clusters,2))
#for iclust in np.unique(ind_clusters):
    #indclust=np.where(ind_clusters==iclust)[0]
    #x_clusters[iclust,:]=np.mean(x[indclust,:],axis=0)
    #plt.text(x_clusters[iclust,0],x_clusters[iclust,1],str(iclust))

for iclust in range(n_clusters):
    plt.text(clustercenters[iclust,0],clustercenters[iclust,1],str(iclust),color='lightgray')
    

plt.xlabel('S_r')
plt.xlim(0,SRmax)
plt.ylabel('ERK activity (z-norm)')
plt.savefig(f'{date}/clusters_SD_ERKSR.png')
np.save(f'{date}/SR_clusters.npy',SR_clusters)

In [ ]:
#Markov state model, with errorbars of mean over fovs
fig,axs=plt.subplots(3,2,figsize=(12,18))
axs=axs.flatten()
MtSet=[None]*nf
tms=[0,1,2,3,4]
SR_clusters_smth=np.linspace(np.min(SR_clusters),np.max(SR_clusters),51)
nboots=100
tl=0
tu=8
for iplot in range(len(tms)):
    i_tm=tms[iplot]
    centers_minima=clusters.clustercenters.copy()
    clusters_minima=coor.clustering.AssignCenters(centers_minima, metric='euclidean')
    indmodels=np.where(inds_tmSet_models==i_tm)[0]
    P_ons_set=[]
    P_offs_set=[]
    P_ons_CR=np.zeros((n_SR,3))
    P_offs_CR=np.zeros((n_SR,3))
    imagingSets=np.unique(inds_imagingSet_models[indmodels])
    for iSet in imagingSets:
        indmodels_imagingSets=np.where(inds_imagingSet_models==iSet)[0]
        indmodels_Set=np.setdiff1d(indmodels,indmodels_imagingSets)
        indmodels_Set=np.intersect1d(indmodels_Set,indgood_models)
        for iboot in range(nboots):
            x0=[]
            x1=[]
            indmodels_boots= np.random.choice(indmodels_Set, size=indmodels_Set.size, replace=True)
            for imodel in indmodels_boots:
                sctm=modelSet[imodel]
                inds_model=np.where(inds_trajsteps_models==imodel)[0]
                #x0=x0set[inds_model,:]
                #x1=x1set[inds_model,:]
                indstw=np.where(np.logical_and(sctm.cells_timeSet[inds_trajsteps[inds_model,1]]<tu,sctm.cells_timeSet[inds_trajsteps[inds_model,1]]>tl))[0]
                x0.append(x0set[inds_model[indstw],:])
                x1.append(x1set[inds_model[indstw],:])    
            x0=np.concatenate(x0)
            x1=np.concatenate(x1)
            P=model.get_transition_matrix(x0,x1,clusters=clusters_minima)  
            P_ons=np.zeros(n_SR)
            P_offs=np.zeros(n_SR)
            P_stayons=np.zeros(n_SR)
            P_stayoffs=np.zeros(n_SR)
            ind0=np.where(centers_minima[:,1]==Slow)[0]
            ind1=np.where(centers_minima[:,1]==Shigh)[0]
            for i in range(n_SR):
                P_ons[i]=np.sum(P[i,ind1])
                P_offs[i]=np.sum(P[i+n_SR,ind0])
                P_stayons[i]=np.sum(P[i+n_SR,ind1])
                P_stayoffs[i]=np.sum(P[i,ind0])
            if np.sum(P_ons>0.):
                P_ons_set.append(P_ons)
            if np.sum(P_offs>0.):
                P_offs_set.append(P_offs)
    P_ons_set=np.array(P_ons_set)
    P_offs_set=np.array(P_offs_set)
    P_ons_CR[:,0]=np.nanmean(P_ons_set,axis=0)
    P_offs_CR[:,0]=np.nanmean(P_offs_set,axis=0)
    for i in range(n_SR):
        P_ons_boots=bayesian_bootstrap.mean(P_ons_set[:,i],n_replications=10000)
        P_ons_CR[i,1:]=bayesian_bootstrap.central_credible_interval(P_ons_boots,alpha=.05) #bootstrap.get_CI(P_ons_set[:,i],1000)
        P_offs_boots=bayesian_bootstrap.mean(P_offs_set[:,i],n_replications=10000)
        P_offs_CR[i,1:]=bayesian_bootstrap.central_credible_interval(P_offs_boots,alpha=.05) #bootstrap.get_CI(P_offs_set[:,i],1000)
    spl_on=csaps.CubicSmoothingSpline(SR_clusters, P_ons_CR[:,0], smooth=.999)
    P_ons_smth=spl_on(SR_clusters_smth)    
    spl_off=csaps.CubicSmoothingSpline(SR_clusters, P_offs_CR[:,0], smooth=.999)
    P_offs_smth=spl_off(SR_clusters_smth) 
    indcalc_on=np.where(P_ons_CR>0.)[0]
    indcalc_off=np.where(P_offs_CR>0.)[0]
    ax=axs[iplot]
    ax.fill_between(SR_clusters[indcalc_on],P_ons_CR[indcalc_on,1],P_ons_CR[indcalc_on,2],color='red',alpha=.2)
    ax.fill_between(SR_clusters[indcalc_off],P_offs_CR[indcalc_off,1],P_offs_CR[indcalc_off,2],color='blue',alpha=.2)
    ax.plot(SR_clusters[indcalc_on],P_ons_CR[indcalc_on,0],'ro--',label='p(off->on)',alpha=1)
    ax.plot(SR_clusters[indcalc_off],P_offs_CR[indcalc_off,0],'bo--',label='p(on->off)',alpha=1)
    #for iSet in range(P_ons_set.shape[0]):
    #    ax.plot(SR_clusters,P_ons_set[iSet,:],'ro',alpha=.3)
    #    ax.plot(SR_clusters,P_offs_set[iSet,:],'bo',alpha=.3)
    #ax.plot(SR_clusters_smth,P_ons_smth,'r-')
    #ax.plot(SR_clusters_smth,P_offs_smth,'b-')
    ax.set_xlim(-.1,SR_clusters[-1]+.1)
    ax.set_xlabel('Sd/R')
    ax.set_ylabel('probability')
    #ax.set_xscale('log')
    ax.legend(loc='upper right')
    ax.set_title(tmSet[i_tm])
    #ax.set_ylim(0,.7)
    #if i_tm==1:
    #    ax.set_ylim(-.01,.6)
    #if i_tm==2:
    #    ax.set_ylim(-.01,.6)
    
plt.savefig(f'{date}/SR_jointP_SD_bootstrap_{figid}_{tmSet[i_tm]}.png')
plt.savefig(f'{date}/SR_jointP_SD_bootstrap_{figid}_{tmSet[i_tm]}.svg')
plt.show()

In [ ]:
@ray.remote(num_cpus=2)
def get_clusterids(h5filename,rsmooth=None):
    print(f'calc {h5filename}')
    sctm=trajectory.Trajectory(h5filename=h5filename)
    status=sctm.load_from_h5('/cell_data_m0/')
    try:
        del sctm.fmsks_cancer; del sctm.fmsks_primary; del sctm.boundary_library #delete off these for memory conservation
    except:
        pass
    print(sctm.h5filename)
    rep_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
    if rsmooth is None:
        rsmooth=sctm.cellpose_diam/2.
    zcut=2.0
    cluster_ids=np.zeros(sctm.cells_indSet.size).astype(int)
    cluster_sizes=np.zeros(sctm.cells_indSet.size).astype(int)
    clusterid=1
    for iframe in range(sctm.nt):
        print(f'extracting cluster data for model {sctm.figid.decode()}, frame {iframe}')
        msk=sctm.get_mask_data(iframe);
        msk_cyto=msk[...,cyto_msk_channel];
        #msk_nuc=msk[...,nuc_msk_channel];
        img_rep=sctm.get_image_data(iframe)[...,rep_img_channel];
        img_rep_smth=skimage.filters.gaussian(img_rep,sigma=[rsmooth/sctm.zscale,rsmooth,rsmooth],mode='mirror')
        fmsk=imprep.znorm(img_rep_smth)>zcut
        indframe=np.where(sctm.cells_frameSet==iframe)[0]
        labelids_frame=sctm.cells_labelidSet[indframe]
        msk_clusters=imprep.get_labeled_mask(fmsk,fill_holes=False)
        titlestr=f''
        for icluster in np.unique(msk_clusters)[1:]:
            #print(f'{iframe}, {icluster}')
            labelids_cluster=np.unique(msk_cyto[msk_clusters==icluster])
            labelids_cluster=labelids_cluster[labelids_cluster>0] #indices one less than labeled mask, ignore 0
            labelids_common,inds_common_frame,inds_common_cluster=np.intersect1d(labelids_frame,labelids_cluster,return_indices=True)
            global_inds=indframe[inds_common_frame]
            print(f'frame {iframe} cluster {clusterid} size {global_inds.size} ids {global_inds} meansig {np.nanmean(sctm.x_signal[global_inds]):.2f})')
            if global_inds.size>0:
                cluster_ids[global_inds]=clusterid
                cluster_sizes[global_inds]=global_inds.size
                clusterid=clusterid+1
    sctm.cluster_ids=cluster_ids
    sctm.cluster_sizes=cluster_sizes
    sctm.save_to_h5('/cell_data_m0/',['cluster_ids','cluster_sizes'],overwrite=True)
    return cluster_ids

In [ ]:
h5filename=modelList[indgood_models[-1]]
cluster_ids=get_clusterids(h5filename,rsmooth=rsmooth)

In [ ]:
#parallel version
futures = [get_clusterids.remote(h5filename,rsmooth=rsmooth) for h5filename in np.array(modelList)[indgood_models]] #now run in parallel with ray

In [ ]:
#get initial frame cluster sizes by replicate
#fraction on version
Smean_csize_set=[None]*nf
csize_set=[None]*nf
rep_set=[None]*nf
clusterbins=np.array([0,1,2,4,8,16,32,64,1000])
#clusterbins=np.array([0,1,2,3,4,5,10,1000])
#clusterbins=np.array([0,1,2,3,4,1000])
#clusterbins=np.array([0,1,2,3,4,10,15])
fl=0
fu=1
for i_tm in range(nf):
    csize=[]
    Smean_csize=[]
    rep=[]
    inds_csize_models=np.array([]).astype(int)
    indmodels=np.where(inds_tmSet_models==i_tm)[0]
    indmodels=np.intersect1d(indmodels,indgood_models)
    for imodel in indmodels:
        try:
            sctm=modelSet[imodel]
            print(f'{sctm.figid.decode()}')
            inds_frames=np.where(np.logical_and(sctm.cells_frameSet>=fl,sctm.cells_frameSet<fu))[0]
            ind_clusters,cluster_sizes=np.unique(sctm.cluster_ids,return_counts=True)
            if np.isin(0,ind_clusters):
                ind_clusters=ind_clusters[1:]
                cluster_sizes=cluster_sizes[1:]
            ind_clusterbins=np.digitize(cluster_sizes,clusterbins)-1
            for iclusterbin in np.unique(ind_clusterbins):
                inds_bin=np.where(ind_clusterbins==iclusterbin)[0]
                clusterids=ind_clusters[inds_bin]
                global_ids=np.where(np.isin(sctm.cluster_ids,clusterids))[0]
                global_ids=np.intersect1d(global_ids,inds_frames)
                #Smean=np.nanmean(sctm.x_signal[global_ids])
                indgood=np.where(np.isfinite(sctm.x_signal[global_ids]))[0]
                Smean=np.nanmean(sctm.x_signal[global_ids[indgood]]>midcut)
                csize.append(clusterbins[iclusterbin])
                Smean_csize.append(Smean)
                rep.append(inds_imagingSet_models[imodel])
            inds_csize_models=np.append(inds_csize_models,np.ones_like(csize)*imodel)
        except Exception as e:
            print(e)
    csize=np.array(csize)
    Smean_csize=np.array(Smean_csize)
    rep=np.array(rep)
    Smean_csize_set[i_tm]=Smean_csize.copy()
    csize_set[i_tm]=csize.copy()
    rep_set[i_tm]=rep.copy()

In [ ]:
# Combine across treatments
# Combine across treatments
cluster_sizes_list = csize_set.copy()
replicate_list = rep_set.copy()
sizes_all = np.concatenate(cluster_sizes_list)
reps_all = np.concatenate(replicate_list)

# Unique bins and replicates
unique_sizes = np.sort(np.unique(sizes_all))
unique_reps = np.sort(np.unique(reps_all))

# Colors for each replicate
repColorSet = ['lightgray', 'dimgray', 'whitesmoke', 'gray']

# Plot
width = 0.8
x = np.arange(len(unique_sizes))

fig, ax = plt.subplots()
bottom = np.zeros(len(unique_sizes))

for i, rep in enumerate(unique_reps):
    counts = np.array([np.sum((sizes_all == sz) & (reps_all == rep)) for sz in unique_sizes])
    #counts = np.array([(sizes_all == sz & (reps_all == rep)).sum() for sz in unique_sizes])
    ax.bar(
        x, counts, width,
        bottom=bottom,
        color=repColorSet[i],
        edgecolor='black',
        label=f'{rep+1}'
    )
    bottom += counts

# Labels and legend
ax.set_xticks(x)
ax.set_xticklabels(unique_sizes)
ax.set_xlabel('Cluster size (cells)')
ax.set_ylabel('Number of clusters')

ax.legend(title='replicate', loc='upper right')
plt.tight_layout()
plt.savefig(f'{date}/clustersizes.svg')

### Boundary library

In [ ]:
border_pix=sctm.micron_per_pixel/sctm.boundary_library['border_resolution']
border_scale=[border_pix*sctm.zscale,border_pix,border_pix]

In [ ]:
#calculate contact fractions and mean distances and save as attributes
surface_names={'cancer':1, 'primary':3, 'matrix':4}
contact_radius=1.4 #distance less means in contact
border_resolution=1.0 #microns
#contact_radius=7.52
for imodel in np.intersect1d(inds_tobeloaded,indgood_models): #indgood_models:
    print(f'calculating contact features for {sctm.figid.decode()}')
    sctm=modelSet[imodel]
    #if hasattr(sctm,'cancer_contact_fraction'):
    #if inds_imagingSet_models[imodel]==1:
        #print(f'{sctm.figid.decode()} has cancer contact attribute')
    print(f'{sctm.figid.decode()}')
    #else:
    ncells=sctm.cells_indSet.size
    boundary_library=sctm.boundary_library
    for surfname in ['cancer','primary','matrix']:
        contact_fraction=np.zeros(ncells)
        mean_distance=np.zeros(ncells)
        for ic in range(ncells):
            b_inds=np.where(np.isin(boundary_library['global_index'],[ic]))[0]
            bpts=boundary_library['nn_pts_states'][b_inds,surface_names[surfname],:]
            pts=boundary_library['pts'][b_inds,:]
            dists=np.linalg.norm(bpts-pts,axis=1)/border_resolution
            dists=dists[np.isfinite(dists)]
            contact_fraction[ic]=np.sum(dists<contact_radius)/dists.size
            mean_distance[ic]=np.mean(dists)
        setattr(sctm,f'{surfname}_contact_fraction',contact_fraction)
        setattr(sctm,f'{surfname}_mean_distance',mean_distance)
        sctm.save_to_h5(f'/cell_data_m{mskchannel}/',[f'{surfname}_contact_fraction',f'{surfname}_mean_distance'],overwrite=True)
        

### Trajectory Embedding

In [ ]:
data_id=f'SITE_mm'

In [ ]:
#Map all trajectories
all_trajSet=[None]*nmodels
for i in indgood_models:
    modelSet[i].get_unique_trajectories()
    all_trajSet[i]=modelSet[i].trajectories.copy()

In [ ]:
cancer_contact_fraction=[]
primary_contact_fraction=[]
matrix_contact_fraction=[]
indtreatment=np.array([]).astype(int)
indcellSet=np.array([]).astype(int)
indcells_global_Set=np.array([]).astype(int)
indcells_models=np.array([]).astype(int)
for i in indgood_models:
    sctm=modelSet[i]
    cancer_contact_fraction.append(sctm.cancer_contact_fraction)
    primary_contact_fraction.append(sctm.primary_contact_fraction)
    matrix_contact_fraction.append(sctm.matrix_contact_fraction)
    indtreatment=np.append(indtreatment,inds_tmSet_models[i]*np.ones(sctm.cancer_contact_fraction.size).astype(int))
    indcellSet=np.append(indcellSet,sctm.cells_indSet)
    indcells_global_Set=np.append(indcells_global_Set,np.arange(sctm.cancer_contact_fraction.size).astype(int))
    indcells_models=np.append(indcells_models,i*np.ones(sctm.cancer_contact_fraction.size).astype(int))

cancer_contact_fraction=np.concatenate(cancer_contact_fraction)
primary_contact_fraction=np.concatenate(primary_contact_fraction)
matrix_contact_fraction=np.concatenate(matrix_contact_fraction)

In [ ]:
cancer_mean_distance=[]
primary_mean_distance=[]
matrix_mean_distance=[]
indtreatment=np.array([]).astype(int)
indcellSet=np.array([]).astype(int)
indcells_models=np.array([]).astype(int)
for i in indgood_models:
    sctm=modelSet[i]
    cancer_mean_distance.append(sctm.cancer_mean_distance)
    primary_mean_distance.append(sctm.primary_mean_distance)
    matrix_mean_distance.append(sctm.matrix_mean_distance)
    indtreatment=np.append(indtreatment,inds_tmSet_models[i]*np.ones(sctm.cancer_mean_distance.size).astype(int))
    indcellSet=np.append(indcellSet,sctm.cells_indSet)
    indcells_models=np.append(indcells_models,i*np.ones(sctm.cancer_mean_distance.size).astype(int))

cancer_mean_distance=np.concatenate(cancer_mean_distance)
primary_mean_distance=np.concatenate(primary_mean_distance)
matrix_mean_distance=np.concatenate(matrix_mean_distance)

In [ ]:
#put the features for embedding into Xpca feature vector
feature_list=[]
Xpca=np.zeros((cratio_erk.size,0)) #use this if you want to leave out pca morphology features

indkeep=np.arange(Xf.shape[1]).astype(int); #np.setdiff1d(np.arange(Xf.shape[1]).astype(int),[16]).astype(int)
for i in indkeep: #range(len(Xf_feature_list)):
    feature_list.append(Xf_feature_list[i].decode())
    Xpca=np.append(Xpca,Xf[:,i][:,np.newaxis],axis=1)

print(feature_list)

for i in indgood_models:
   indsf=np.where(indcells_models==i)[0]
   modelSet[i].Xpca=Xpca[indsf,:]

ndim=Xpca.shape[1]

In [ ]:
Xpca.shape

### Trajectory Embedding and State Analysis

In [ ]:
def get_tcf(self, trajectories=None, x=None, minlength=2):
    if x is None:
        x=self.Xpca
    if trajectories is None:
        trajectories = self.trajectories
    ntraj = len(trajectories)
    traj_lengths = np.zeros(ntraj)
    for itraj in range(ntraj):
        traj_lengths[itraj] = trajectories[itraj].size
    nframes = np.max(traj_lengths)
    nt = np.floor(nframes / 2).astype(int)
    dxcorr = np.zeros(nt)
    tnorm = np.zeros(nt)
    for itraj in range(ntraj):
        cell_traj = trajectories[itraj]
        traj_len = cell_traj.size
        nmax = np.floor(traj_len / 2).astype(int)
        if traj_len > minlength:
            xtraj = x[cell_traj,:]
            for it1 in range(nmax):
                for it2 in range(it1, it1 + nmax):
                    it = it2 - it1
                    #dxcorr[it] = dxcorr[it] + np.sum(np.power(xtraj[it1, :]-xtraj[it2, :],2))
                    corr=np.dot(xtraj[it1,:],xtraj[it2,:])
                    if np.isfinite(corr):
                        dxcorr[it]=dxcorr[it]+corr
                        tnorm[it] = tnorm[it] + 1
    for it in range(nt):
        dxcorr[it] = dxcorr[it] / tnorm[it]
    return dxcorr

In [ ]:
tcfSet = [None] * nmodels

for imodel in indgood_models:
    i_tm=inds_tmSet_models[imodel]
    sctm = modelSet[imodel]
    tcf = get_tcf(sctm, x=sctm.Xpca, minlength=2)
    if tcf.size > 0:
        av_sq = np.sum(np.power(np.nanmean(sctm.Xpca, axis=0), 2))
        avsq = np.nanmean(np.power(np.linalg.norm(sctm.Xpca, axis=1), 2))
        tcf = (tcf - av_sq) / (tcf[0]-av_sq)  # Auto-correlation
        tcfSet[imodel] = tcf
        
        # Get the color for the current model's treatment from color mapping
        current_treatment = tmSet[inds_tmSet_models[imodel]]
        color = colorSet[i_tm]  # Default to 'grey' if not found
        
        # Plot with the correct color
        plt.plot(np.arange(1, tcf.size + 1), tcf, color=color, label=current_treatment)

plt.xlabel('Lag')
plt.ylabel('Auto-correlation')
plt.title('Time Correlation Function')
plt.ylim(-1,1)
plt.show()


In [ ]:
n_tm=len(tmSet)
nf = n_tm
tcf_tm = [None] * nf
tmax = 60

for i_tm in range(nf):
    ind_models = indgood_models[inds_tmSet_models[indgood_models] == i_tm]
    print(f'{tmSet[i_tm]} models {ind_models}')
    tcfs = np.ones((tmax, ind_models.size)) * np.nan

    for ii in range(ind_models.size):
        try:
            tmax_model = tcfSet[ind_models[ii]].size
            tmax_model = np.min(np.array([tmax, tmax_model]))
            tcfs[0:tmax_model, ii] = tcfSet[ind_models[ii]][0:tmax_model]
        except Exception as e:
            print(e)

    tcf_tm[i_tm] = np.nanmean(tcfs, axis=1)

    # Get color from the color mapping
    color = colorSet[i_tm]  # Default to 'grey' if not found

    # Plot with the correct color
    plt.plot(np.arange(1, tmax + 1), tcf_tm[i_tm], color=color, label=tmSet[i_tm])

trajl=5
plt.plot([trajl,trajl],[0,np.nanmax(np.array(tcf_tm))],'k--',linewidth=.3)
# Customize plot labels and title
plt.title('Feature autocorrelation')
plt.xlabel('time separation (frames)')
plt.ylim(0,1)
plt.legend(loc='upper right')
plt.savefig(f'{date}/tcf.svg')
plt.show()


In [ ]:
#trajectory embedding
trajl=1
ndim=modelSet[indgood_models[0]].Xpca.shape[1]
Xpcat=np.zeros((0,ndim*trajl))
indtreatment_traj=np.array([]).astype(int)
indstack_traj=np.array([]).astype(int)
indframes_traj=np.array([]).astype(int)
cellinds0_traj=np.array([]).astype(int)
cellinds1_traj=np.array([]).astype(int)
cancer_cf_traj=np.array([])
primary_cf_traj=np.array([])
matrix_cf_traj=np.array([])
cancer_dist_traj=np.array([])
primary_dist_traj=np.array([])
matrix_dist_traj=np.array([])
cratio_erk_traj=np.array([]).astype(float)
cratio_erk_full_traj=np.zeros((0,trajl))
cratio_akt_full_traj=np.zeros((0,trajl))
cancer_cf_full_traj=np.zeros((0,trajl))
primary_cf_full_traj=np.zeros((0,trajl))
matrix_cf_full_traj=np.zeros((0,trajl))
cancer_dist_full_traj=np.zeros((0,trajl))
primary_dist_full_traj=np.zeros((0,trajl))
matrix_dist_full_traj=np.zeros((0,trajl))
dx_full_traj=np.zeros((0,trajl))
dx_traj=np.array([])
Sr_traj=np.array([])
Sr_full_traj=np.zeros((0,trajl))
Srd_traj=np.array([])
Srd_full_traj=np.zeros((0,trajl))
fra1_traj=np.array([])
fra1_full_traj=np.zeros((0,trajl))
for i in indgood_models:
    print('building trajectory data for model {}...'.format(i))
    modelSet[i].trajectories=all_trajSet[i].copy()
    modelSet[i].trajl=trajl
    modelSet[i].traj=modelSet[i].get_traj_segments(trajl)
    data=modelSet[i].Xpca[modelSet[i].traj,:]
    data=data.reshape(modelSet[i].traj.shape[0],modelSet[i].Xpca.shape[1]*trajl)
    indgood=np.where(np.sum(np.isnan(data),axis=1)==0)[0]
    data=data[indgood,:]
    modelSet[i].traj=modelSet[i].traj[indgood,:]
    Xpcat=np.append(Xpcat,data,axis=0)
    indtreatment_traj=np.append(indtreatment_traj,i*np.ones(data.shape[0]).astype(int))
    indstacks=modelSet[i].cells_imgfileSet[modelSet[i].traj[:,0]]
    indstack_traj=np.append(indstack_traj,indstacks)
    indframes=modelSet[i].cells_frameSet[modelSet[i].traj[:,-1]]
    indframes_traj=np.append(indframes_traj,indframes)
    cellinds0=modelSet[i].traj[:,0]
    cellinds0_traj=np.append(cellinds0_traj,cellinds0)
    cellinds1=modelSet[i].traj[:,-1]
    cellinds1_traj=np.append(cellinds1_traj,cellinds1)
    cratio_erk_traj=np.append(cratio_erk_traj,modelSet[i].x_signal[cellinds1])
    cratio_erk_full_traj=np.append(cratio_erk_full_traj,modelSet[i].x_signal[modelSet[i].traj],axis=0)
    cratio_akt_full_traj=np.append(cratio_akt_full_traj,modelSet[i].x_signal_akt[modelSet[i].traj],axis=0)
    cancer_cf_traj=np.append(cancer_cf_traj,modelSet[i].cancer_contact_fraction[cellinds1])
    primary_cf_traj=np.append(primary_cf_traj,modelSet[i].primary_contact_fraction[cellinds1])
    matrix_cf_traj=np.append(matrix_cf_traj,modelSet[i].matrix_contact_fraction[cellinds1])
    cancer_cf_full_traj=np.append(cancer_cf_full_traj,modelSet[i].cancer_contact_fraction[modelSet[i].traj],axis=0)
    primary_cf_full_traj=np.append(primary_cf_full_traj,modelSet[i].primary_contact_fraction[modelSet[i].traj],axis=0)
    matrix_cf_full_traj=np.append(matrix_cf_full_traj,modelSet[i].matrix_contact_fraction[modelSet[i].traj],axis=0)
    cancer_dist_traj=np.append(cancer_dist_traj,modelSet[i].cancer_mean_distance[cellinds1])
    primary_dist_traj=np.append(primary_dist_traj,modelSet[i].primary_mean_distance[cellinds1])
    matrix_dist_traj=np.append(matrix_dist_traj,modelSet[i].matrix_mean_distance[cellinds1])
    cancer_dist_full_traj=np.append(cancer_dist_full_traj,modelSet[i].cancer_mean_distance[modelSet[i].traj],axis=0)
    primary_dist_full_traj=np.append(primary_dist_full_traj,modelSet[i].primary_mean_distance[modelSet[i].traj],axis=0)
    matrix_dist_full_traj=np.append(matrix_dist_full_traj,modelSet[i].matrix_mean_distance[modelSet[i].traj],axis=0)
    dx_traj=np.append(dx_traj,modelSet[i].dx[cellinds1])
    dx_full_traj=np.append(dx_full_traj,modelSet[i].dx[modelSet[i].traj],axis=0)
    Sr_traj=np.append(Sr_traj,modelSet[i].S_r[cellinds1])
    Sr_full_traj=np.append(Sr_full_traj,modelSet[i].S_r[modelSet[i].traj],axis=0)
    Srd_traj=np.append(Srd_traj,modelSet[i].S_r_death[cellinds1])
    Srd_full_traj=np.append(Srd_full_traj,modelSet[i].S_r_death[modelSet[i].traj],axis=0)
    fra1_traj=np.append(fra1_traj,modelSet[i].x_signal_fra1[cellinds1])
    fra1_full_traj=np.append(fra1_full_traj,modelSet[i].x_signal_fra1[modelSet[i].traj],axis=0)

cellinds0_traj=cellinds0_traj.astype(int)
cellinds1_traj=cellinds1_traj.astype(int)

x=Xpcat #x=np.matmul(Xpcat,np.real(M)) #x=np.concatenate((np.real(np.matmul(Xpcat,M)),np.imag(np.matmul(Xpcat,M))),axis=1) #x=np.concatenate((np.real(phi_X[:,indmodes_M1]),np.imag(phi_X[:,indmodes_M1])),axis=1)
for i in indgood_models:
    modelSet[i].trajectories=all_trajSet[i].copy()

neigen=x.shape[1]
inds_conditions=[None]*nf
for imf in range(nf):
    indmodels=np.intersect1d(indgood_models,np.where(inds_tmSet_models==imf)[0])
    indstm=np.array([])
    for imodel in indmodels:
        indtm=np.where(indtreatment_traj==imodel)
        indstm=np.append(indstm,indtm)
    inds_conditions[imf]=indstm.astype(int).copy()


In [ ]:
#get transitions in the trajectory embedding space
x0set=np.zeros((0,neigen))
x1set=np.zeros((0,neigen))
inds_trajsteps_models=np.array([]).astype(int)
inds_trajsteps=np.zeros((0,2)).astype(int)
for i in indgood_models:
    print('getting flows from model: '+str(i))
    indstm=np.where(indtreatment_traj==i)[0]
    if indstm.size>0:
        modelSet[i].Xtraj=x[indstm,0:neigen]
        indstm_model=indstm-np.min(indstm) #index in model
        modelSet[i].get_trajectory_steps(inds=None,get_trajectories=False,traj=modelSet[i].traj[indstm_model,:],Xtraj=modelSet[i].Xtraj[indstm_model,:])
        x0=modelSet[i].Xtraj0
        x1=modelSet[i].Xtraj1
        x0set=np.append(x0set,x0,axis=0)
        x1set=np.append(x1set,x1,axis=0)
        inds_trajsteps_models=np.append(inds_trajsteps_models,np.ones(x0.shape[0]).astype(int)*i)
        inds_trajsteps=np.append(inds_trajsteps,modelSet[i].inds_trajp1,axis=0)

### Date Driven Modeling

In [ ]:
#Analysis identifiers
figid=f'SITE_mml1_trajl{trajl}_'
print(figid)

In [ ]:
n_clusters=150 #int(np.sqrt(x.shape[0]))
clusters=np.load(f'7sep25/clusters_nc{n_clusters}_{figid}.npy',allow_pickle=True)
clusters=clusters.item()

In [ ]:
print(f'{x.shape[0]}, sqrt {np.sqrt(x.shape[0])}')

In [ ]:
plt.scatter(x[:,0],x[:,2],s=5,c='black',marker='.',alpha=.3)
plt.scatter(clusters.clustercenters[:,0],clusters.clustercenters[:,2])

In [ ]:
#get microstate clusters
get_clusters=False
n_clusters=150 #int(np.sqrt(x.shape[0]))
if get_clusters:
    clusters=coor.cluster_kmeans([x],k=n_clusters,metric='euclidean',fixed_seed=2,max_iter=100) #0)
    np.save(f'{date}/clusters_nc{n_clusters}_{figid}.npy',clusters)
else:
    clusters=np.load(f'{date}/clusters_nc{n_clusters}_{figid}.npy',allow_pickle=True)
    #clusters=np.load(f'6may25/clusters_nc{n_clusters}_{figid}.npy',allow_pickle=True)
    clusters=clusters.item()

In [ ]:
#check out a umap of the embedding space
import sklearn
#reducer=umap.UMAP(n_components=2) #, metric='euclidean')
reducer=sklearn.manifold.TSNE(n_components=2)
trans = reducer.fit(x.astype(np.float32))
x_umap=trans.embedding_

In [ ]:
#plot umap of the embedding space
fig,ax=plt.subplots(1,2,figsize=(12,6))

ax[0].scatter(x_umap[:,0],x_umap[:,1],s=20,c=corrc_erk_traj,cmap=plt.cm.seismic,clim=(-2,2))
ax[0].set_title('ERK')
ind_clusters=clusters.assign(x)
x_umap_clusters=np.zeros((n_clusters,2))
for iclust in np.unique(ind_clusters):
    indclust=np.where(ind_clusters==iclust)[0]
    x_umap_clusters[iclust,:]=np.mean(x_umap[indclust,:],axis=0)
    #ax[0].text(x_umap_clusters[iclust,0],x_umap_clusters[iclust,1],str(iclust))
    ax[0].axis('off')

ax[1].scatter(x_umap[:,0],x_umap[:,1],s=20,c=corrc_akt_traj,cmap=plt.cm.seismic,clim=(-2,2))
ax[1].set_title('AKT')
ind_clusters=clusters.assign(x)
x_umap_clusters=np.zeros((n_clusters,2))
for iclust in np.unique(ind_clusters):
    indclust=np.where(ind_clusters==iclust)[0]
    x_umap_clusters[iclust,:]=np.mean(x_umap[indclust,:],axis=0)
    #ax[1].text(x_umap_clusters[iclust,0],x_umap_clusters[iclust,1],str(iclust))
    ax[1].axis('off')

#plt.savefig(f'{date}/dMAP_{figid}.png')
plt.savefig(os.path.join(date, f'_UMAP_{figid}.png'))
plt.savefig(os.path.join(date, f'_UMAP_{figid}.svg'), format='svg')
plt.show()

In [ ]:
#Markov state model
centers_minima=clusters.clustercenters.copy()
clusters_minima=coor.clustering.AssignCenters(centers_minima, metric='euclidean')
P=model.get_transition_matrix(x0set,x1set,clusters=clusters_minima)
graph = csr_matrix(P>0.)
n_components, labels = connected_components(csgraph=graph, directed=False, return_labels=True)
unique, counts = np.unique(labels, return_counts=True)
icc=unique[np.argmax(counts)]
indcc=np.where(labels==icc)[0]
centers_minima=centers_minima[indcc,:];stateCenters=centers_minima
clusters_minima=coor.clustering.AssignCenters(centers_minima, metric='euclidean')
P=model.get_transition_matrix(x0set,x1set,clusters=clusters_minima)
n_clusters=indcc.size

H=.5*(P+np.transpose(P))+.5j*(P-np.transpose(P))
w,v=np.linalg.eig(H)
w=np.real(w)
indsort=np.argsort(w)
w=w[indsort]
v=v[:,indsort]

plt.clf()
plt.plot(np.arange(n_clusters),w,'ko',label='eigenvalues')
plt.ylim(0.,np.max(w)+.05*np.max(w))
plt.xlabel('eig index')
plt.ylabel('H eigenvalue')
plt.pause(.1)

In [ ]:
ncomp=3 #number of eigencomponents to keep
vr=np.multiply(w[-ncomp:],np.real(v[:,-ncomp:]))
vi=np.multiply(w[-ncomp:],np.imag(v[:,-ncomp:]))
vkin=np.append(vr,vi,axis=1)
va=np.abs(v[:,-ncomp:])
vkin=np.append(vr,vi,axis=1)
vth=np.arctan2(vr,vi)
plt.clf()
plt.plot(np.arange(n_clusters),w,'ko',label='eigenvalues')
#plt.plot(dense_range,wdense,'k-',label='spline')
plt.plot(np.arange(n_clusters)[-ncomp:],w[-ncomp:],'ro')
plt.ylim(0.,np.max(w)+.05*np.max(w))
plt.xlabel('eig index')
plt.ylabel('H eigenvalue')


#plt.savefig(f'{date}/eigenvalues_{figid}.png')
plt.savefig(os.path.join(date, f'_eigenvalues_{figid}.png'))
plt.savefig(os.path.join(date, f'_eigenvalues_{figid}.svg'), format='svg')
plt.pause(.1)

In [ ]:
get_cluster_embedding=False
if get_cluster_embedding:
    reducer=umap.UMAP(n_components=2) #n_neighbors=int(n_clusters),min_dist=0.1, n_components=2, metric='euclidean')
    #reducer=sklearn.manifold.TSNE(n_components=2)
    trans = reducer.fit(vkin)
    x_clusters=trans.embedding_
    np.save(f'{date}/cluster_embedding_nc{n_clusters}_ncomp{ncomp}_{figid}.npy',x_clusters)
else:
    #x_clusters=np.load(f'{date}/cluster_embedding_nc{n_clusters}_ncomp{ncomp}_{figid}.npy')
    x_clusters=np.load(f'6may25/cluster_embedding_nc{n_clusters}_ncomp{ncomp}_{figid}.npy')

ind_clusters=clusters_minima.assign(x)
x_clusters_all=x_clusters[ind_clusters,:]

In [ ]:
#to visualize in a different basis
ind_clusters=clusters_minima.assign(x)
#x_clusters=centers_minima.copy()
x_clusters_all=np.array([2.*((1./sctm.boundary_library['border_resolution'])*(1./cancer_dist_traj)-.5),2.*((1./sctm.boundary_library['border_resolution'])*(1./primary_dist_traj)-.5)]).T
x_clusters=np.zeros((n_clusters,2))
for iclust in range(n_clusters):
    indclust=np.where(ind_clusters==iclust)[0]
    x_clusters[iclust,:]=np.nanmean(x_clusters_all[indclust,:],axis=0)

In [ ]:
plt.close('all')
for icomp in range(ncomp-1,0-1,-1): #range(ncomp):
    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1);plt.scatter(x_clusters[:,0],x_clusters[:,1],s=30,c=va[:,icomp],cmap=plt.cm.seismic)
    plt.title('absolute value '+str(ncomp-icomp))
    plt.subplot(1,2,2);plt.scatter(x_clusters[:,0],x_clusters[:,1],s=30,c=vth[:,icomp],cmap=plt.cm.seismic)
    plt.title('theta '+str(ncomp-icomp))
    #plt.savefig(date+'/componentrth_'+str(ncomp-icomp)+'_H_nc'+str(n_clusters)+'_ncomp'+str(ncomp)+'_'+figid+'.png')
    plt.savefig(os.path.join(date, f'componentrth_'+str(ncomp-icomp)+'_H_nc'+str(n_clusters)+'_ncomp'+str(ncomp)+'_'+figid+'.png'))
    plt.savefig(os.path.join(date, f'componentrth_'+str(ncomp-icomp)+'_H_nc'+str(n_clusters)+'_ncomp'+str(ncomp)+'_'+figid+'.svg'), format='svg')

In [ ]:
dx_clusters=np.zeros((nf,n_clusters,x_clusters.shape[1]))
MtSet=[None]*nf
for i in range(nf):
    indmodels=np.where(inds_tmSet_models==i)[0]
    indstm=np.array([]).astype(int)
    for imodel in indmodels:
        indstm=np.append(indstm,np.where(inds_trajsteps_models==imodel)[0].astype(int))
    x0=x0set[indstm,:]
    x1=x1set[indstm,:]
    Mt=model.get_transition_matrix(x0,x1,clusters_minima)
    MtSet[i]=Mt.copy()
    dxmatrix=np.zeros((n_clusters,n_clusters,x_clusters.shape[1]))
    for ii in range(n_clusters):
        for jj in range(n_clusters):
            dxmatrix[ii,jj]=(x_clusters[jj,:]-x_clusters[ii,:])*Mt[ii,jj]
    dx_clusters[i,:,:]=np.sum(dxmatrix,axis=1)

dx_clusters_smooth=np.zeros_like(dx_clusters)
knn=20
for i in range(nf):
    for iclust in range(n_clusters):
        xc=np.array([clusters_minima.clustercenters[iclust,:]])
        dmatr=utilities.get_dmat(clusters_minima.clustercenters,xc) #get closest cells to cluster center
        indr=np.argsort(dmatr[:,0])
        indr=indr[0:knn]
        dx_clusters_smooth[i,iclust,:]=np.mean(dx_clusters[i,indr,:],axis=0)

In [ ]:
dxsav=np.mean(dx_clusters_smooth,axis=0)
ascale=.8
ax=plt.gca()
for ic in range(n_clusters):
    ax.arrow(x_clusters[ic,0],x_clusters[ic,1],ascale*dxsav[ic,0],ascale*dxsav[ic,1],head_width=.02,linewidth=.5,color='black',alpha=1.0)

#plt.xlim(-1,1)
#plt.ylim(-1,1)
#plt.axis('equal')
#plt.savefig(date+'/dx_clusters_'+figid+'.png')
#plt.savefig(date+'/dx_clusters_'+figid+'.pdf')
plt.savefig(os.path.join(date, f'dx_clusters'+figid+'.png'))
plt.savefig(os.path.join(date, f'dx_clusters'+figid+'.svg'), format='svg')

plt.pause(.1)
plt.show()

In [ ]:
times_models=np.zeros(indframes_models.size)
rep_models=np.zeros(indframes_models.size)
for imodel in indgood_models:
    start_frame=initial_frames_model[imodel]
    timestep=timestep_model[imodel]
    indmodel=np.where(indcells_models==imodel)[0]
    times_model=timestep*(indframes_models[indmodel]-start_frame)
    times_models[indmodel]=times_model
    rep_models[indmodel]=inds_imagingSet_models[imodel]
    sctm=modelSet[imodel]
    sctm.cells_timeSet=times_model.copy()
    print(f'rep {inds_imagingSet_models[imodel]} model {imodel} nframes {np.max(sctm.cells_frameSet)} start {start_frame} {sctm.figid.decode()}')

In [ ]:
times_traj=np.zeros(indframes_traj.size)
rep_traj=np.zeros(indframes_traj.size)
for imodel in indgood_models:
    start_frame=initial_frames_model[imodel]
    timestep=timestep_model[imodel]
    indmodel=np.where(indtreatment_traj==imodel)[0]
    times_traj_model=timestep*(indframes_traj[indmodel]-start_frame)
    times_traj[indmodel]=times_traj_model
    rep_traj[indmodel]=inds_imagingSet_models[imodel]
    sctm=modelSet[imodel]
    print(f'rep {inds_imagingSet_models[imodel]} model {imodel} nframes {np.max(sctm.cells_frameSet)} start {start_frame} {sctm.figid.decode()}')
    
    

In [ ]:
plt.close('all')
head_width=.04
nbins=20
tl=12
tu=16 #hours for time window from times_traj
indstw=np.where(np.logical_and(times_traj<tu,times_traj>tl))[0]
sml=1.
nplots=int(np.ceil(np.sqrt(nf+1)))
plt.figure(figsize=(12,12))
#indstw=np.where(np.logical_and(indframes_traj<fu,indframes_traj>fl))[0]
probSet=[None]*nf
plt.subplot(nplots,nplots,1)
prob1,xedges1,yedges1=np.histogram2d(x_clusters_all[indstw,0],x_clusters_all[indstw,1],bins=nbins,density=True)
prob1=scipy.ndimage.gaussian_filter(prob1,sigma=sml)
prob1c=utilities.get_cdist2d(prob1)
xx,yy=np.meshgrid(.5*xedges1[1:]+.5*xedges1[0:-1],.5*yedges1[1:]+.5*yedges1[0:-1])
levels=np.linspace(0,1,21)
cs=plt.contourf(xx,yy,prob1c.T,levels=levels,cmap=plt.cm.jet_r)
cbar=utilities.colorbar(cs)
#cbar.set_label('cumulative probability')
#plt.title('combined cumulative distribution')
plt.axis('off')
for ic in range(n_clusters):
    ax=plt.gca()
    ax.arrow(x_clusters[ic,0],x_clusters[ic,1],ascale*dxsav[ic,0],ascale*dxsav[ic,1],head_width=head_width,linewidth=.4,color='white',alpha=1.0)
    
for imf in range(nf):
    tm=tmSet[imf]
    indstm=inds_conditions[imf]
    indstwm=np.intersect1d(indstm,indstw)
    prob,xedges2,yedges2=np.histogram2d(x_clusters_all[indstwm,0],x_clusters_all[indstwm,1],bins=[xedges1,yedges1],density=True)
    #prob=prob/np.sum(prob)
    prob=scipy.ndimage.gaussian_filter(prob,sigma=sml)
    probc=utilities.get_cdist2d(prob)
    probSet[imf]=prob.copy()
    plt.subplot(nplots,nplots,imf+2)
    #levels=np.linspace(0,np.max(prob),100)
    cs=plt.contourf(xx,yy,probc.T,levels=levels,cmap=plt.cm.jet_r,extend='both')
    dxf=dx_clusters_smooth[i,:,:]
    ax=plt.gca()
    #for ic in range(n_clusters):
    #    ax.arrow(x_clusters[ic,0],x_clusters[ic,1],ascale*dxf[ic,0],ascale*dxf[ic,1],head_width=head_width,linewidth=.4,color='white',alpha=1.0)
    plt.title(tmSet[imf])
    cs.cmap.set_over('darkred')
    plt.axis('off')

#plt.savefig(date+'/probflows_'+figid+'.png')
plt.savefig(os.path.join(date, f'prob_final'+figid+'.png'))
plt.savefig(os.path.join(date, f'prob_final'+figid+'.svg'), format='svg')

In [ ]:
plt.close('all')
head_width=.04
nbins=20
tl=0
tu=.5*trajl+1 #hours for time window from times_traj
indstw=np.where(np.logical_and(times_traj<tu,times_traj>tl))[0]
sml=1.
nplots=int(np.ceil(np.sqrt(nf+1)))
plt.figure(figsize=(12,12))
probSet=[None]*nf
plt.subplot(nplots,nplots,1)
prob1,xedges2,yedges2=np.histogram2d(x_clusters_all[indstw,0],x_clusters_all[indstw,1],bins=[xedges1,yedges1],density=True)
prob1=scipy.ndimage.gaussian_filter(prob1,sigma=sml)
prob1c=utilities.get_cdist2d(prob1)
xx,yy=np.meshgrid(.5*xedges1[1:]+.5*xedges1[0:-1],.5*yedges1[1:]+.5*yedges1[0:-1])
levels=np.linspace(0,1,21)
cs=plt.contourf(xx,yy,prob1c.T,levels=levels,cmap=plt.cm.jet_r)
cbar=utilities.colorbar(cs)
#cbar.set_label('cumulative probability')
#plt.title('combined cumulative distribution')
plt.axis('off')
for ic in range(n_clusters):
    ax=plt.gca()
    ax.arrow(x_clusters[ic,0],x_clusters[ic,1],ascale*dxsav[ic,0],ascale*dxsav[ic,1],head_width=head_width,linewidth=.4,color='white',alpha=1.0)
    
for imf in range(nf):
    tm=tmSet[imf]
    indstm=inds_conditions[imf]
    indstwm=np.intersect1d(indstm,indstw)
    prob,xedges2,yedges2=np.histogram2d(x_clusters_all[indstwm,0],x_clusters_all[indstwm,1],bins=[xedges1,yedges1],density=True)
    #prob=prob/np.sum(prob)
    prob=scipy.ndimage.gaussian_filter(prob,sigma=sml)
    probc=utilities.get_cdist2d(prob)
    probSet[imf]=prob.copy()
    plt.subplot(nplots,nplots,imf+2)
    #levels=np.linspace(0,np.max(prob),100)
    cs=plt.contourf(xx,yy,probc.T,levels=levels,cmap=plt.cm.jet_r,extend='both')
    dxf=dx_clusters_smooth[i,:,:]
    ax=plt.gca()
    #for ic in range(n_clusters):
    #    ax.arrow(x_clusters[ic,0],x_clusters[ic,1],ascale*dxf[ic,0],ascale*dxf[ic,1],head_width=head_width,linewidth=.4,color='white',alpha=1.0)
    plt.title(tmSet[imf])
    cs.cmap.set_over('darkred')
    plt.axis('off')

#plt.savefig(date+'/probflows_'+figid+'.png')
plt.savefig(os.path.join(date, f'prob_init'+figid+'.png'))
plt.savefig(os.path.join(date, f'prob_init'+figid+'.svg'), format='svg')

In [ ]:
#plot initial, persistor, sensitive populations
sml=1.
head_width=.04
indsetList=[]
captionset=['initial','sensitive','resistant']
nf=len(tmSet)
nbins=20
fw=trajl+7
levels=np.linspace(0,1,21)
for i_tm in range(nf):
        fig,axs=plt.subplots(1,3,figsize=(14,4))
        #vset=vsetList[iv]
        indmodels=np.where(inds_tmSet_models==i_tm)[0]
        inds_death_traj=[]
        inds_initial_traj=[]
        inds_resistant_traj=[]
        times_death_traj=[]
        for imodel in indmodels:
            fl=initial_frames_model[imodel]
            fu=final_frames_model[imodel]-8 #frames for time window
            indstw=np.where(np.logical_and(indframes_traj>fl,indframes_traj<fu))[0]
            indcells_traj_model=np.where(indtreatment_traj==imodel)[0]
            indcells_traj_model=np.intersect1d(indcells_traj_model,indstw)
            cellinds1_traj_model=cellinds1_traj[indcells_traj_model]
            sctm=modelSet[imodel]
            if hasattr(sctm,'inds_death'):
                cellinds_death=sctm.inds_death.copy()
            else:
                cellinds_death=np.array([]).astype(int)
            inds_death_traj.append(indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)])
            times_death_traj.append(timestep_model[imodel]*(indframes_traj[indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)]-initial_frames_model[imodel]]))
            #fl=initial_frames_model[imodel]
            #fu=final_frames_model[imodel] #frames for time window
            inds_initial_traj.append(np.intersect1d(np.where(np.logical_and(indframes_traj<fl+fw,indframes_traj>fl))[0],indcells_traj_model))
            inds_resistant_traj.append(np.intersect1d(np.where(np.logical_and(indframes_traj<fu,indframes_traj>fu-fw))[0],indcells_traj_model))
        inds_death_traj=np.concatenate(inds_death_traj)
        times_death_traj=np.concatenate(times_death_traj)
        inds_initial_traj=np.concatenate(inds_initial_traj)
        #print(inds_initial_traj)
        inds_resistant_traj=np.concatenate(inds_resistant_traj)
        prob_initial,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_initial_traj,0],x_clusters_all[inds_initial_traj,1],bins=[xedges1,yedges1],density=True)
        prob_initial=scipy.ndimage.gaussian_filter(prob_initial,sigma=sml)
        probc_initial=utilities.get_cdist2d(prob_initial)
        cs=axs[0].contourf(xx,yy,probc_initial.T,levels=levels,cmap=plt.cm.binary_r,extend='both')
        axs[0].set_title(f'{tmSet[i_tm]} initial')
        axs[0].axis('off')
        prob_death,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_death_traj,0],x_clusters_all[inds_death_traj,1],bins=[xedges1,yedges1],density=True)
        prob_death=scipy.ndimage.gaussian_filter(prob_death,sigma=sml)
        probc_death=utilities.get_cdist2d(prob_death)
        cs=axs[1].contourf(xx,yy,prob_death.T,levels=None,cmap=plt.cm.binary,extend='both')
        axs[1].scatter(x_clusters_all[inds_death_traj,0],x_clusters_all[inds_death_traj,1],s=30,c=times_death_traj,marker='x',cmap=plt.cm.autumn,alpha=.4,clim=(0,20))
        axs[1].set_title(f'{tmSet[i_tm]} sensitive')
        axs[1].axis('off')
        prob_resistant,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_resistant_traj,0],x_clusters_all[inds_resistant_traj,1],bins=[xedges1,yedges1],density=True)
        prob_resistant=scipy.ndimage.gaussian_filter(prob_resistant,sigma=sml)
        probc_resistant=utilities.get_cdist2d(prob_resistant)
        #cs=axs[2].contourf(xx,yy,probc_resistant.T,levels=levels,cmap=plt.cm.binary_r,extend='both')
        cs=axs[2].contourf(xx,yy,prob_resistant.T-prob_initial.T,levels=2.*(levels-.5),cmap=plt.cm.seismic,extend='both')
        axs[2].set_title(f'{tmSet[i_tm]} resistant')
        axs[2].axis('off')
        plt.savefig(os.path.join(date, f'distributions_{tmSet[i_tm]}_{figid}.png'))
        plt.savefig(os.path.join(date, f'distributions_{tmSet[i_tm]}_{figid}.svg'), format='svg')   
        plt.show()

In [ ]:
#plot initial, persistor, sensitive populations
sml=2.
head_width=.04
indsetList=[]
captionset=['initial','sensitive','resistant']
nf=len(tmSet)
nbins=20
fw=trajl+2
levels=np.linspace(0,1,21)
pinitialSet=[None]*nf
psensitiveSet=[None]*nf
presistantSet=[None]*nf
inds_death_traj_set=[None]*nf
times_death_traj_set=[None]*nf
i_tmr=1
for i_tm in range(nf):
        #vset=vsetList[iv]
        indmodels=np.where(inds_tmSet_models==i_tm)[0]
        inds_death_traj=[]
        inds_initial_traj=[]
        inds_resistant_traj=[]
        times_death_traj=[]
        for imodel in indmodels:
            fl=initial_frames_model[imodel]
            fu=final_frames_model[imodel]-8 #frames for time window
            indstw=np.where(np.logical_and(indframes_traj>fl,indframes_traj<fu))[0]
            indcells_traj_model=np.where(indtreatment_traj==imodel)[0]
            indcells_traj_model=np.intersect1d(indcells_traj_model,indstw)
            indserkon=np.where(cratio_erk_traj>midcut)[0]
            indcells_traj_model=np.intersect1d(indcells_traj_model,indserkon)
            cellinds1_traj_model=cellinds1_traj[indcells_traj_model]
            sctm=modelSet[imodel]
            if hasattr(sctm,'inds_death'):
                cellinds_death=sctm.inds_death.copy()
            else:
                cellinds_death=np.array([]).astype(int)
            inds_death_traj.append(indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)])
            times_death_traj.append(times_traj[indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)]])#timestep_model[imodel]*(indframes_traj[indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)]-initial_frames_model[imodel]]))
            inds_initial_traj.append(np.intersect1d(np.where(np.logical_and(indframes_traj<fl+fw,indframes_traj>fl))[0],indcells_traj_model))
            inds_resistant_traj.append(np.intersect1d(np.where(np.logical_and(indframes_traj<fu,indframes_traj>fu-fw))[0],indcells_traj_model))
        inds_death_traj=np.concatenate(inds_death_traj)
        inds_death_traj_set[i_tm]=inds_death_traj.copy()
        times_death_traj=np.concatenate(times_death_traj)
        times_death_traj_set[i_tm]=times_death_traj.copy()
        if i_tm==i_tmr:
            inds_death_mcl1i_traj=inds_death_traj.copy()
            times_death_mcl1i_traj=times_death_traj.copy()
        inds_initial_traj=np.concatenate(inds_initial_traj)
        inds_resistant_traj=np.concatenate(inds_resistant_traj)
        prob_initial,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_initial_traj,0],x_clusters_all[inds_initial_traj,1],bins=[xedges1,yedges1],density=True)
        prob_initial=scipy.ndimage.gaussian_filter(prob_initial,sigma=sml)
        probc_initial=utilities.get_cdist2d(prob_initial)
        prob_death,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_death_traj,0],x_clusters_all[inds_death_traj,1],bins=[xedges1,yedges1],density=True)
        prob_death=scipy.ndimage.gaussian_filter(prob_death,sigma=sml)
        probc_death=utilities.get_cdist2d(prob_death)
        prob_resistant,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_resistant_traj,0],x_clusters_all[inds_resistant_traj,1],bins=[xedges1,yedges1],density=True)
        prob_resistant=scipy.ndimage.gaussian_filter(prob_resistant,sigma=sml)
        probc_resistant=utilities.get_cdist2d(prob_resistant)
        pinitialSet[i_tm]=prob_initial.copy()
        psensitiveSet[i_tm]=prob_death.copy()
        presistantSet[i_tm]=prob_resistant.copy()

for i_tmr in range(nf):
    fig,axs=plt.subplots(1,3,figsize=(14,4))
    cs=axs[0].contourf(xx,yy,utilities.get_cdist2d(pinitialSet[i_tmr]).T,levels=levels,cmap=plt.cm.binary_r,extend='both')
    axs[0].set_title(f't=0 hrs')
    axs[0].axis('off')
    
    vdist1,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_death_traj_set[i_tmr],0],x_clusters_all[inds_death_traj_set[i_tmr],1],bins=[xedges1,yedges1],weights=times_death_traj_set[i_tmr])
    vdist1=scipy.ndimage.gaussian_filter(vdist1,sigma=sml)
    norm1,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_death_traj_set[i_tmr],0],x_clusters_all[inds_death_traj_set[i_tmr],1],bins=[xedges1,yedges1])
    norm1=scipy.ndimage.gaussian_filter(norm1,sigma=sml)
    vdist1=np.divide(vdist1,norm1)
    indnan=np.where(np.isnan(vdist1))
    indgood=np.where(np.logical_and(np.logical_not(np.isnan(vdist1)),np.logical_not(np.isinf(vdist1))))[0]
    xedges1c=.5*(xedges1[1:]+xedges1[0:-1])
    yedges1c=.5*(yedges1[1:]+yedges1[0:-1])
    xx,yy=np.meshgrid(xedges1c,yedges1c)
    if indgood.size>0:
        levels_f=np.linspace(np.nanmin(vdist1[indgood]),np.nanmax(vdist1[indgood]),20)
    #cs=axs[1].contourf(xx,yy,vdist1.T,cmap=plt.cm.binary,levels=levels_f)
    cs=axs[1].contourf(xx,yy,utilities.get_cdist2d(presistantSet[i_tmr]).T,levels=levels,cmap=plt.cm.binary_r,extend='both')
    axs[1].scatter(x_clusters_all[inds_death_traj_set[i_tmr],0],x_clusters_all[inds_death_traj_set[i_tmr],1],s=40,c='orange',marker='x',alpha=.6,cmap=plt.cm.autumn)
    #cs=axs[1].contourf(xx,yy,utilities.get_cdist2d(psensitiveSet[0]).T,levels=levels,cmap=plt.cm.binary_r,extend='both')
    axs[1].set_title(f't=16 hrs')
    axs[1].axis('off')
    
    cs=axs[2].contourf(xx,yy,presistantSet[i_tmr].T-pinitialSet[i_tmr].T,levels=2.*(levels-.5),cmap=plt.cm.seismic,extend='both')
    #for istate in range(nstates):
    #    axs[0].text(state_centers_minima[istate,0],state_centers_minima[istate,1],str(state_labels[istate]),fontsize=16,fontstyle='normal',color='white')  
    #    axs[1].text(state_centers_minima[istate,0],state_centers_minima[istate,1],str(state_labels[istate]),fontsize=16,fontstyle='normal',color='white')  
    #    axs[2].text(state_centers_minima[istate,0],state_centers_minima[istate,1],str(state_labels[istate]),fontsize=16,fontstyle='normal')  
    #axs[2].scatter(x_clusters[:,0],x_clusters[:,1],s=20,c='white',alpha=.2)
    #axs[2].scatter(x_clusters_all[inds_death_mcl1i_traj,0],x_clusters_all[inds_death_mcl1i_traj,1],s=30,c=times_death_mcl1i_traj,marker='x',cmap=plt.cm.binary,alpha=.4)
    axs[2].set_title(f'final-initial {tmSet[i_tmr]}')
    axs[2].axis('off')
    plt.savefig(os.path.join(date, f'distributions_erkon_wpoints_{tmSet[i_tmr]}_{figid}.png'))
    plt.savefig(os.path.join(date, f'distributions_erkon_wpoints_{tmSet[i_tmr]}_{figid}.svg'), format='svg')   
    plt.show()

In [ ]:
#plot initial, persistor, sensitive populations
sml=2.
head_width=.04
indsetList=[]
captionset=['initial','sensitive','resistant']
nf=len(tmSet)
nbins=20
fw=trajl+2
levels=np.linspace(0,1,21)
pinitialSet=[None]*nf
psensitiveSet=[None]*nf
presistantSet=[None]*nf
inds_death_traj_set=[None]*nf
times_death_traj_set=[None]*nf
i_tmr=1
for i_tm in range(nf):
        #vset=vsetList[iv]
        indmodels=np.where(inds_tmSet_models==i_tm)[0]
        inds_death_traj=[]
        inds_initial_traj=[]
        inds_resistant_traj=[]
        times_death_traj=[]
        for imodel in indmodels:
            fl=initial_frames_model[imodel]
            fu=final_frames_model[imodel]-8 #frames for time window
            indstw=np.where(np.logical_and(indframes_traj>fl,indframes_traj<fu))[0]
            indcells_traj_model=np.where(indtreatment_traj==imodel)[0]
            indcells_traj_model=np.intersect1d(indcells_traj_model,indstw)
            indserkoff=np.where(cratio_erk_traj<midcut)[0]
            indcells_traj_model=np.intersect1d(indcells_traj_model,indserkoff)
            cellinds1_traj_model=cellinds1_traj[indcells_traj_model]
            sctm=modelSet[imodel]
            if hasattr(sctm,'inds_death'):
                cellinds_death=sctm.inds_death.copy()
            else:
                cellinds_death=np.array([]).astype(int)
            inds_death_traj.append(indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)])
            times_death_traj.append(times_traj[indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)]])#timestep_model[imodel]*(indframes_traj[indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)]-initial_frames_model[imodel]]))
            inds_initial_traj.append(np.intersect1d(np.where(np.logical_and(indframes_traj<fl+fw,indframes_traj>fl))[0],indcells_traj_model))
            inds_resistant_traj.append(np.intersect1d(np.where(np.logical_and(indframes_traj<fu,indframes_traj>fu-fw))[0],indcells_traj_model))
        inds_death_traj=np.concatenate(inds_death_traj)
        inds_death_traj_set[i_tm]=inds_death_traj.copy()
        times_death_traj=np.concatenate(times_death_traj)
        times_death_traj_set[i_tm]=times_death_traj.copy()
        if i_tm==i_tmr:
            inds_death_mcl1i_traj=inds_death_traj.copy()
            times_death_mcl1i_traj=times_death_traj.copy()
        inds_initial_traj=np.concatenate(inds_initial_traj)
        inds_resistant_traj=np.concatenate(inds_resistant_traj)
        prob_initial,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_initial_traj,0],x_clusters_all[inds_initial_traj,1],bins=[xedges1,yedges1],density=True)
        prob_initial=scipy.ndimage.gaussian_filter(prob_initial,sigma=sml)
        probc_initial=utilities.get_cdist2d(prob_initial)
        prob_death,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_death_traj,0],x_clusters_all[inds_death_traj,1],bins=[xedges1,yedges1],density=True)
        prob_death=scipy.ndimage.gaussian_filter(prob_death,sigma=sml)
        probc_death=utilities.get_cdist2d(prob_death)
        prob_resistant,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_resistant_traj,0],x_clusters_all[inds_resistant_traj,1],bins=[xedges1,yedges1],density=True)
        prob_resistant=scipy.ndimage.gaussian_filter(prob_resistant,sigma=sml)
        probc_resistant=utilities.get_cdist2d(prob_resistant)
        pinitialSet[i_tm]=prob_initial.copy()
        psensitiveSet[i_tm]=prob_death.copy()
        presistantSet[i_tm]=prob_resistant.copy()

for i_tmr in range(nf):
    fig,axs=plt.subplots(1,3,figsize=(14,4))
    cs=axs[0].contourf(xx,yy,utilities.get_cdist2d(pinitialSet[i_tmr]).T,levels=levels,cmap=plt.cm.binary_r,extend='both')
    axs[0].set_title(f't=0 hrs')
    axs[0].axis('off')
    
    vdist1,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_death_traj_set[i_tmr],0],x_clusters_all[inds_death_traj_set[i_tmr],1],bins=[xedges1,yedges1],weights=times_death_traj_set[i_tmr])
    vdist1=scipy.ndimage.gaussian_filter(vdist1,sigma=sml)
    norm1,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_death_traj_set[i_tmr],0],x_clusters_all[inds_death_traj_set[i_tmr],1],bins=[xedges1,yedges1])
    norm1=scipy.ndimage.gaussian_filter(norm1,sigma=sml)
    vdist1=np.divide(vdist1,norm1)
    indnan=np.where(np.isnan(vdist1))
    indgood=np.where(np.logical_and(np.logical_not(np.isnan(vdist1)),np.logical_not(np.isinf(vdist1))))[0]
    xedges1c=.5*(xedges1[1:]+xedges1[0:-1])
    yedges1c=.5*(yedges1[1:]+yedges1[0:-1])
    xx,yy=np.meshgrid(xedges1c,yedges1c)
    if indgood.size>0:
        levels_f=np.linspace(np.nanmin(vdist1[indgood]),np.nanmax(vdist1[indgood]),20)
    #cs=axs[1].contourf(xx,yy,vdist1.T,cmap=plt.cm.binary,levels=levels_f)
    cs=axs[1].contourf(xx,yy,utilities.get_cdist2d(presistantSet[i_tmr]).T,levels=levels,cmap=plt.cm.binary_r,extend='both')
    #axs[1].scatter(x_clusters_all[inds_death_traj_set[i_tmr],0],x_clusters_all[inds_death_traj_set[i_tmr],1],s=40,c='orange',marker='x',alpha=.6,cmap=plt.cm.autumn)
    #cs=axs[1].contourf(xx,yy,utilities.get_cdist2d(psensitiveSet[0]).T,levels=levels,cmap=plt.cm.binary_r,extend='both')
    axs[1].set_title(f't=16 hrs')
    axs[1].axis('off')
    
    cs=axs[2].contourf(xx,yy,presistantSet[i_tmr].T-pinitialSet[i_tmr].T,levels=2.*(levels-.5),cmap=plt.cm.seismic,extend='both')
    #for istate in range(nstates):
    #    axs[0].text(state_centers_minima[istate,0],state_centers_minima[istate,1],str(state_labels[istate]),fontsize=16,fontstyle='normal',color='white')  
    #    axs[1].text(state_centers_minima[istate,0],state_centers_minima[istate,1],str(state_labels[istate]),fontsize=16,fontstyle='normal',color='white')  
    #    axs[2].text(state_centers_minima[istate,0],state_centers_minima[istate,1],str(state_labels[istate]),fontsize=16,fontstyle='normal')  
    #axs[2].scatter(x_clusters[:,0],x_clusters[:,1],s=20,c='white',alpha=.2)
    #axs[2].scatter(x_clusters_all[inds_death_mcl1i_traj,0],x_clusters_all[inds_death_mcl1i_traj,1],s=30,c=times_death_mcl1i_traj,marker='x',cmap=plt.cm.binary,alpha=.4)
    axs[2].set_title(f'final-initial {tmSet[i_tmr]}')
    axs[2].axis('off')
    plt.savefig(os.path.join(date, f'distributions_erkoff_{tmSet[i_tmr]}_{figid}.png'))
    plt.savefig(os.path.join(date, f'distributions_erkoff_{tmSet[i_tmr]}_{figid}.svg'), format='svg')   
    plt.show()

In [ ]:
#plot initial, persistor, sensitive populations
sml=2.
head_width=.04
indsetList=[]
captionset=['initial','sensitive','resistant']
nf=len(tmSet)
nbins=20
fw=trajl+2
levels=np.linspace(0,1,21)
pinitialSet=[None]*nf
psensitiveSet=[None]*nf
presistantSet=[None]*nf
inds_death_traj_set=[None]*nf
times_death_traj_set=[None]*nf
i_tmr=1
for i_tm in range(nf):
        #vset=vsetList[iv]
        indmodels=np.where(inds_tmSet_models==i_tm)[0]
        inds_death_traj=[]
        inds_initial_traj=[]
        inds_resistant_traj=[]
        times_death_traj=[]
        for imodel in indmodels:
            fl=initial_frames_model[imodel]
            fu=final_frames_model[imodel]-8 #frames for time window
            indstw=np.where(np.logical_and(indframes_traj>fl,indframes_traj<fu))[0]
            indcells_traj_model=np.where(indtreatment_traj==imodel)[0]
            indcells_traj_model=np.intersect1d(indcells_traj_model,indstw)
            #indserkoff=np.where(cratio_erk_traj<midcut)[0]
            #indcells_traj_model=np.intersect1d(indcells_traj_model,indserkoff)
            cellinds1_traj_model=cellinds1_traj[indcells_traj_model]
            sctm=modelSet[imodel]
            if hasattr(sctm,'inds_death'):
                cellinds_death=sctm.inds_death.copy()
            else:
                cellinds_death=np.array([]).astype(int)
            inds_death_traj.append(indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)])
            times_death_traj.append(times_traj[indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)]])#timestep_model[imodel]*(indframes_traj[indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)]-initial_frames_model[imodel]]))
            inds_initial_traj.append(np.intersect1d(np.where(np.logical_and(indframes_traj<fl+fw,indframes_traj>fl))[0],indcells_traj_model))
            inds_resistant_traj.append(np.intersect1d(np.where(np.logical_and(indframes_traj<fu,indframes_traj>fu-fw))[0],indcells_traj_model))
        inds_death_traj=np.concatenate(inds_death_traj)
        inds_death_traj_set[i_tm]=inds_death_traj.copy()
        times_death_traj=np.concatenate(times_death_traj)
        times_death_traj_set[i_tm]=times_death_traj.copy()
        if i_tm==i_tmr:
            inds_death_mcl1i_traj=inds_death_traj.copy()
            times_death_mcl1i_traj=times_death_traj.copy()
        inds_initial_traj=np.concatenate(inds_initial_traj)
        inds_resistant_traj=np.concatenate(inds_resistant_traj)
        prob_initial,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_initial_traj,0],x_clusters_all[inds_initial_traj,1],bins=[xedges1,yedges1],density=True)
        prob_initial=scipy.ndimage.gaussian_filter(prob_initial,sigma=sml)
        probc_initial=utilities.get_cdist2d(prob_initial)
        prob_death,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_death_traj,0],x_clusters_all[inds_death_traj,1],bins=[xedges1,yedges1],density=True)
        prob_death=scipy.ndimage.gaussian_filter(prob_death,sigma=sml)
        probc_death=utilities.get_cdist2d(prob_death)
        prob_resistant,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_resistant_traj,0],x_clusters_all[inds_resistant_traj,1],bins=[xedges1,yedges1],density=True)
        prob_resistant=scipy.ndimage.gaussian_filter(prob_resistant,sigma=sml)
        probc_resistant=utilities.get_cdist2d(prob_resistant)
        pinitialSet[i_tm]=prob_initial.copy()
        psensitiveSet[i_tm]=prob_death.copy()
        presistantSet[i_tm]=prob_resistant.copy()

for i_tmr in range(nf):
    fig,axs=plt.subplots(1,3,figsize=(14,4))
    cs=axs[0].contourf(xx,yy,utilities.get_cdist2d(pinitialSet[i_tmr]).T,levels=levels,cmap=plt.cm.binary_r,extend='both')
    axs[0].set_title(f't=0 hrs')
    axs[0].axis('off')
    
    vdist1,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_death_traj_set[i_tmr],0],x_clusters_all[inds_death_traj_set[i_tmr],1],bins=[xedges1,yedges1],weights=times_death_traj_set[i_tmr])
    vdist1=scipy.ndimage.gaussian_filter(vdist1,sigma=sml)
    norm1,xedges2,yedges2=np.histogram2d(x_clusters_all[inds_death_traj_set[i_tmr],0],x_clusters_all[inds_death_traj_set[i_tmr],1],bins=[xedges1,yedges1])
    norm1=scipy.ndimage.gaussian_filter(norm1,sigma=sml)
    vdist1=np.divide(vdist1,norm1)
    indnan=np.where(np.isnan(vdist1))
    indgood=np.where(np.logical_and(np.logical_not(np.isnan(vdist1)),np.logical_not(np.isinf(vdist1))))[0]
    xedges1c=.5*(xedges1[1:]+xedges1[0:-1])
    yedges1c=.5*(yedges1[1:]+yedges1[0:-1])
    xx,yy=np.meshgrid(xedges1c,yedges1c)
    if indgood.size>0:
        levels_f=np.linspace(np.nanmin(vdist1[indgood]),np.nanmax(vdist1[indgood]),20)
    #cs=axs[1].contourf(xx,yy,vdist1.T,cmap=plt.cm.binary,levels=levels_f)
    cs=axs[1].contourf(xx,yy,utilities.get_cdist2d(presistantSet[i_tmr]).T,levels=levels,cmap=plt.cm.binary_r,extend='both')
    #axs[1].scatter(x_clusters_all[inds_death_mcl1i_traj,0],x_clusters_all[inds_death_mcl1i_traj,1],s=40,c=times_death_mcl1i_traj,marker='x',alpha=.3,cmap=plt.cm.autumn)
    #cs=axs[1].contourf(xx,yy,utilities.get_cdist2d(psensitiveSet[0]).T,levels=levels,cmap=plt.cm.binary_r,extend='both')
    axs[1].set_title(f't=16 hrs')
    axs[1].axis('off')
    
    cs=axs[2].contourf(xx,yy,presistantSet[i_tmr].T-pinitialSet[i_tmr].T,levels=2.*(levels-.5),cmap=plt.cm.seismic,extend='both')
    #for istate in range(nstates):
    #    axs[0].text(state_centers_minima[istate,0],state_centers_minima[istate,1],str(state_labels[istate]),fontsize=16,fontstyle='normal',color='white')  
    #    axs[1].text(state_centers_minima[istate,0],state_centers_minima[istate,1],str(state_labels[istate]),fontsize=16,fontstyle='normal',color='white')  
    #    axs[2].text(state_centers_minima[istate,0],state_centers_minima[istate,1],str(state_labels[istate]),fontsize=16,fontstyle='normal')  
    #axs[2].scatter(x_clusters[:,0],x_clusters[:,1],s=20,c='white',alpha=.2)
    #axs[2].scatter(x_clusters_all[inds_death_mcl1i_traj,0],x_clusters_all[inds_death_mcl1i_traj,1],s=30,c=times_death_mcl1i_traj,marker='x',cmap=plt.cm.binary,alpha=.4)
    axs[1].scatter(x_clusters_all[inds_death_traj_set[i_tmr],0],x_clusters_all[inds_death_traj_set[i_tmr],1],s=40,c='orange',marker='x',alpha=.6,cmap=plt.cm.autumn)
    axs[2].set_title(f'final-initial {tmSet[i_tmr]}')
    axs[2].axis('off')
    plt.savefig(os.path.join(date, f'distributions_erkall_wpoints_{tmSet[i_tmr]}_{figid}.png'))
    plt.savefig(os.path.join(date, f'distributions_erkall_wpoints_{tmSet[i_tmr]}_{figid}.svg'), format='svg')   
    plt.show()

In [ ]:
plt.close('all')
plt.clf()
nstateSet=[2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,18,20,22,24,28,30] #,30,40]
nS=len(nstateSet)
scoreSet_states=[None]*nS
nstates_initial=nstateSet[0]
pcut_final=.05
indc_x0set=clusters_minima.assign(x0set);indc_x1set=clusters_minima.assign(x1set)
for istate in range(len(nstateSet)):
    MtSet_test=[None]*nf
    nstates=nstateSet[istate]
    scoreSet=np.zeros(nf)
    #clusters_v = KMeans(n_clusters=nstates,init='k-means++',n_init=1000,max_iter=1000)
    #clusters_v.fit(vkin)
    #states=clusters_v.labels_
    states,nstates_initial=model.get_kineticstates(vkin,nstates,nstates_initial=nstates_initial,pcut_final=pcut_final,seed=0,return_nstates_initial=True,cluster_ninit=100)
    for imf in range(nf):
        print('building likelihood model for '+tmSet[imf])
        indmodels=np.where(inds_tmSet_models==imf)[0]
        indmodels=np.intersect1d(indmodels,indgood_models)
        indstm=np.array([]).astype(int)
        for i in indmodels: #[:-1]:
            indstm=np.append(indstm,np.where(inds_trajsteps_models==i)[0])
        MtSet_test[imf]=model.get_transition_matrix_CG(x0set[indstm,:],x1set[indstm,:],clusters=clusters_minima,states=states)
        scoreSet[imf]=model.get_kscore(MtSet_test[imf])
        #scoreSet[imf]=model.get_path_entropy_2point(states[indc_x0set],states[indc_x1set],MtSet_test[imf])
    print(str(nstates)+' states')
    print(scoreSet)
    nstates_initial=nstates_initial+1
    #np.savetxt(f'{date}_scoreSet_{figid}_nS{nstates}.dat',scoreSet)
    scoreSet_states[istate]=scoreSet
    plt.clf()
    plt.contourf(xx,yy,-np.log(prob1.T),cmap=plt.cm.gray,alpha=0.8)
    plt.scatter(x_clusters[:,0],x_clusters[:,1],s=100,c=states,cmap=plt.cm.jet)
    plt.title(str(nstates))
    plt.pause(2)
    #plt.savefig(f'{date}/clustertest_{figid}_{nstates}.png')
    #plt.savefig(os.path.join(date, f'clustertest_{figid}_{nstates}.png'))
    #plt.savefig(os.path.join(date, f'clustertest_{figid}_{nstates}.svg'), format='svg')   


In [ ]:
scoreSet_statearray = np.zeros((nS, nf))

# Populate scoreSet_statearray with data from scoreSet_states
for istate in range(len(nstateSet)):
    try:
        scoreSet_statearray[istate, :] = scoreSet_states[istate]
    except:
        pass  # Skip if there is an issue

# Close any existing plots and create a new figure
plt.close('all')
plt.figure(figsize=(6, 5))
plt.clf()

# Plot data for each treatment with the appropriate color
for imf in range(nf):
    color = colorSet[imf]  # Use color mapping or default to 'grey'
    plt.plot(nstateSet, scoreSet_statearray[:, imf], 'o-', color=color, label=tmSet[imf], alpha=0.2)

# Plot the mean score across all treatments
plt.plot(nstateSet, np.nanmean(scoreSet_statearray, axis=1), 'o--', color='black', linewidth=2)

# Set y-axis to a logarithmic scale and add labels
plt.yscale('log')
plt.ylabel(r'kinetic score ($\sum_l \tau_l$)')
plt.xlabel('number of states')
plt.pause(0.1)

# Save the figure
plt.savefig(os.path.join(date, f'scoreset_{figid}.png'))
plt.savefig(os.path.join(date, f'scoreset_{figid}.svg'), format='svg')

# Display the plot
plt.show()


In [ ]:
get_kstates=True
nstates_final=3 #for production
pcut_final=.05 #for prediction
#pcut_final=.02 #for production
nstates_initial=nstates_final
stateCenters=x_clusters
if get_kstates:
    stateSet=model.get_kineticstates(vkin,nstates_final,nstates_initial=nstates_initial,pcut_final=pcut_final,seed=0,return_nstates_initial=False,cluster_ninit=100)
    nstates=np.unique(stateSet).size
    objFile=f'{date}/stateSet_{figid}_nS{nstates}.obj'
    objFileHandler=open(objFile,'wb')
    states_object=[clusters_minima,stateSet]
    pickle.dump(states_object,objFileHandler,protocol=4)
    objFileHandler.close()
    #stateSet=get_kineticstates(wctm,nstates_final,P=MtSet[4],clusters_minima=clusters_minima,pcut_final=.01,ncomp=ncomp) #treatment specific
else:
    objFile=f'{date}/stateSet_{figid}_nS{nstates_final}.obj'
    objFileHandler=open(objFile,'rb')
    states_object=pickle.load(objFileHandler)
    clusters_minima=states_object[0]
    stateSet=states_object[1]
    objFileHandler.close()

nstates=np.unique(stateSet).size; n_states=nstates
state_centers_minima=np.zeros((n_states,x_clusters.shape[1]))
for i in range(n_states):
    indstate=np.where(stateSet==i)[0]
    state_centers_minima[i,:]=np.median(stateCenters[indstate,:],axis=0)



In [ ]:
#load from specific file
nstates=3
objFile=f'11sep25/stateSet_SITE_mml1_trajl1__nS{nstates}.obj'
objFileHandler=open(objFile,'rb')
states_object=pickle.load(objFileHandler)
clusters_minima=states_object[0]
stateSet=states_object[1]
objFileHandler.close()

nstates=np.unique(stateSet).size; n_states=nstates
n_clusters=clusters_minima.clustercenters.shape[0]
#to visualize in a different basis
ind_clusters=clusters_minima.assign(x)
x_clusters_all=np.array([2.*((1./sctm.boundary_library['border_resolution'])*(1./cancer_dist_traj)-.5),2.*((1./sctm.boundary_library['border_resolution'])*(1./primary_dist_traj)-.5)]).T
x_clusters=np.zeros((n_clusters,2))
for iclust in range(n_clusters):
    indclust=np.where(ind_clusters==iclust)[0]
    #indclust=np.intersect1d(indclust,inds_training_traj)
    x_clusters[iclust,:]=np.nanmean(x_clusters_all[indclust,:],axis=0)
    print(indclust)

#set cell locations by cluster
#x_clusters_all=x_clusters[ind_clusters,:]
#x_clusters[:,0]=clusters_minima.clustercenters[:,0]
#x_clusters[:,1]=clusters_minima.clustercenters[:,2]
stateCenters=x_clusters
#set cell locations by cluster
#x_clusters_all=x_clusters[ind_clusters,:]
state_centers_minima=np.zeros((n_states,x_clusters.shape[1]))
for i in range(n_states):
    indstate=np.where(stateSet==i)[0]
    state_centers_minima[i,:]=np.nanmedian(x_clusters[indstate,:],axis=0)

In [ ]:
#state_order=np.argsort(state_centers_minima[:,0])
inds_states=[None]*n_states
for i in range(n_states):
    indstate=np.where(stateSet[ind_clusters]==i)[0]
    inds_states[i]=indstate
    
# Step 1: Calculate mean ERK activity and determine sorted order
activity = np.nanmean(primary_cf_full_traj,axis=1)
mean_activity = []
tm_include=[0,1]
for i in range(n_states):
    inds_traj=inds_states[i]
    inds_traj=inds_traj[np.isin(inds_tmSet_models[indtreatment_traj[inds_traj]],tm_include)]
    setv = activity[inds_traj]
    indgood=np.where(np.isfinite(setv))[0]
    setv = setv[indgood]
    mean_activity.append(np.mean(setv))

# Get sorted indices based on mean ERK activity
state_order = np.argsort(mean_activity)
#state_order=np.array([6,4,5,3,2,1,0])
#state_order=np.array([9,16,15,13,10,4,6,7,3,14,12,11,1,0,2,5,8]).astype(int)
#state_order=np.array([0,1,2,7,18,17,15,4,11,8,3,14,16,12,13,5,6,9,10])
#state_order=np.array([0,2,1,7,18,17,15,8,11,4,3,16,14,5,12,13,6,9,10])
stateSet_ordered=np.ones_like(stateSet)*-1
for istate in range(nstates):
    stateSet_ordered[stateSet==state_order[istate]]=istate

stateSet=stateSet_ordered

nstates=np.unique(stateSet).size; n_states=nstates
state_centers_minima=np.zeros((n_states,x_clusters.shape[1]))
for i in range(n_states):
    indstate=np.where(stateSet==i)[0]
    state_centers_minima[i,:]=np.nanmedian(stateCenters[indstate,:],axis=0)

inds_states=[None]*n_states
for i in range(n_states):
    indstate=np.where(stateSet[ind_clusters]==i)[0]
    inds_states[i]=indstate

In [ ]:
import string
nbins=prob1.shape[0]
state_labels=np.array(list(string.ascii_uppercase))[0:nstates]
plt.close('all');plt.figure(figsize=(6,5))
plt.clf()
pts=np.array([xx.flatten(),yy.flatten()]).T
#indpts=clusters_minima.assign(pts)
#states=stateSet[indpts]
#states=states[prob1.flatten()>np.min(prob1[prob1>0])]
#pts=pts[prob1.flatten()>np.min(prob1[prob1>0]),:]
plt.contourf(xx,yy,prob1.T,cmap=plt.cm.gray,levels=20,alpha=.3)
#plt.scatter(tateSet=model.get_kineticstates(vkin,nstates_final,nstates_initial=nstates_initial,pcut_final=pcut_final,seed=0,return_nstates_initial=False,cluster_ninit=500)ts[:,0],pts[:,1],s=10,c=states,cmap=plt.cm.jet,marker='.',alpha=0.5)
plt.scatter(x_clusters[:,0],x_clusters[:,1],s=100,c=stateSet,cmap=plt.cm.jet)
#for ic in range(n_clusters):
#    ax=plt.gca()
#    ax.arrow(x_clusters[ic,0],x_clusters[ic,1],dxsav[ic,0],dxsav[ic,1],head_width=.1,linewidth=.2,color='white',alpha=1.0)

for istate in range(nstates):
    plt.text(state_centers_minima[istate,0],state_centers_minima[istate,1],str(state_labels[istate]))

plt.axis('off')
#plt.savefig(date+'/kineticstates_'+figid+'_nS_'+str(nstates)+'.png')
#plt.savefig(date+'/kineticstates_'+figid+'_nS_'+str(nstates)+'.pdf')
plt.savefig(os.path.join(date, f'nstates_{figid}.png'))
plt.savefig(os.path.join(date, f'nstates_{figid}.svg'), format='svg')   
plt.pause(.1)

In [ ]:
#plt.contourf(xx,yy,prob1.T,cmap=plt.cm.gray,levels=20,alpha=.3)
plt.scatter(x_clusters_all[:,0],x_clusters_all[:,1],s=2,c=stateSet[ind_clusters],cmap=plt.cm.jet,alpha=.3)
for istate in range(nstates):
    plt.text(state_centers_minima[istate,0],state_centers_minima[istate,1],str(state_labels[istate]),size='large')

plt.axis('off')
#plt.savefig(date+'/kineticstates_'+figid+'_nS_'+str(nstates)+'.png')
#plt.savefig(date+'/kineticstates_'+figid+'_nS_'+str(nstates)+'.pdf')
plt.savefig(os.path.join(date, f'states_all_{figid}.png'))
plt.savefig(os.path.join(date, f'states_all_{figid}.svg'), format='svg')   
plt.pause(.1)

In [ ]:
levels=np.linspace(0,1,41)
plt.figure(figsize=(8,6))
plt.gca()
i_tmr=np.where(np.array(tmSet)=='Mcl1i')[0][0]
cs=plt.contourf(xx,yy,presistantSet[i_tmr].T-pinitialSet[i_tmr].T,levels=2.*(levels-.5),cmap=plt.cm.seismic,extend='both')
#plt.title(f'resistant')
plt.axis('off')
cbar=plt.colorbar()
cbar.set_label(r'$P_{\mathrm{resistant}}-P_{\mathrm{initial}}$')
#plt.scatter(x_clusters_all[inds_death_mcl1i_traj,0],x_clusters_all[inds_death_mcl1i_traj,1],s=30,c='black',marker='x',cmap=plt.cm.prism,alpha=.1)
#dxf=dx_clusters_smooth[0,:,:]
#ax=plt.gca()
#for ic in range(n_clusters):
#    ax.arrow(x_clusters[ic,0],x_clusters[ic,1],ascale*dxf[ic,0],ascale*dxf[ic,1],head_width=head_width,linewidth=.4,color='black',alpha=1.0)
plt.scatter(x_clusters[:,0],x_clusters[:,1],s=100,c=stateSet,cmap=plt.cm.jet,alpha=.4)
for istate in range(nstates):
    plt.text(state_centers_minima[istate,0],state_centers_minima[istate,1],str(state_labels[istate]),color='black')  
    
plt.savefig(os.path.join(date, f'resistant_dist_{tmSet[i_tmr]}_{figid}.png'))
plt.savefig(os.path.join(date, f'resistant_dist_{tmSet[i_tmr]}_{figid}.svg'), format='svg')   

In [ ]:
levels=np.linspace(0,1,41)
for i_tmr in range(len(tmSet)):
    plt.figure(figsize=(7,5))
    plt.gca()
    cs=plt.contourf(xx,yy,presistantSet[i_tmr].T-pinitialSet[i_tmr].T,levels=4.*(levels-.5),cmap=plt.cm.seismic,extend='both')
    #plt.title(f'resistant')
    plt.axis('off')
    cbar=plt.colorbar()
    cbar.set_label(r'$P_{\mathrm{initial}}-P_{\mathrm{initial}}$')
    #plt.scatter(x_clusters_all[inds_death_mcl1i_traj,0],x_clusters_all[inds_death_mcl1i_traj,1],s=30,c='black',marker='x',cmap=plt.cm.prism,alpha=.1)
    #dxf=dx_clusters_smooth[0,:,:]
    #ax=plt.gca()
    #for ic in range(n_clusters):
    #    ax.arrow(x_clusters[ic,0],x_clusters[ic,1],ascale*dxf[ic,0],ascale*dxf[ic,1],head_width=head_width,linewidth=.4,color='black',alpha=1.0)
    #plt.scatter(x_clusters[:,0],x_clusters[:,1],s=100,c=stateSet,cmap=plt.cm.tab20,alpha=.4)
    for istate in range(nstates):
        plt.text(state_centers_minima[istate,0],state_centers_minima[istate,1],str(state_labels[istate]),fontsize=16,fontstyle='normal')  
        
    plt.savefig(os.path.join(date, f'resistant_dist_{tmSet[i_tmr]}_{figid}.png'))
    plt.savefig(os.path.join(date, f'resistant_dist_{tmSet[i_tmr]}_{figid}.svg'), format='svg')   

In [ ]:
def show_cell_activation(self,icell,cyto_msk_channel=None,rep_img_channel=None,nuc_img_channel=None,bf_img_channel=None,viewsize=50,npix=100,sigma=3.,zoom=2,nuchigh=None,nuclow=None,rephigh=None,replow=None,bflow=None,bfhigh=None,projection=True,show_neighbor_signals=False):
    #plt.figure(figsize=(6,6))
    if cyto_msk_channel is None:
        cyto_msk_channel=self.mskchannel
        if rep_img_channel is None:
            rep_img_channel=self.rep_channel
        if nuc_img_channel is None:
            nuc_img_channel=self.nuc_channel
        if bf_img_channel is None:
            bf_img_channel=self.bf_channel
    if True: #try:
        imgc,mskc=self.get_cell_data(icell,relabel_mskchannels=[0],boundary_expansion=[1,viewsize,viewsize])
        imgc2,mskc2=self.get_cell_data(icell,relabel_mskchannels=[],boundary_expansion=[1,viewsize,viewsize])
        if not projection:
            mslices=np.where(np.sum(mskc[...,cyto_msk_channel],axis=(1,2))>0)[0]
            hslice=np.argmax(np.sum(imgc[mslices,:,:,nuc_img_channel],axis=(1,2)))
            islice=mslices[hslice]
            img_rep=imprep.znorm(imgc[islice,:,:,rep_img_channel]).astype(float)
            #img_rep=imprep.znorm(np.max(imgc[...,rep_img_channel],axis=0))
            img_nuc=imprep.znorm(imgc[islice,:,:,nuc_img_channel]).astype(float)
            #img_nuc=imprep.znorm(np.max(imgc[...,nuc_img_channel],axis=0))
            img_bf=skimage.filters.difference_of_gaussians(imgc[...,bf_img_channel],.5,1.5)
            img_bf=imprep.znorm(img_bf[islice,:,:]).astype(float)
            #img_bf=imprep.znorm(np.max(img_bf,axis=0))
            msk_cyto=mskc[islice,:,:,cyto_msk_channel]
            #msk_cyto=np.max(mskc[...,cyto_msk_channel],axis=0)
            msk2_cyto=mskc2[islice,:,:,cyto_msk_channel]
        else:
            img_rep=np.max(imprep.znorm(imgc[:,:,:,rep_img_channel]).astype(float),axis=0)
            img_nuc=np.max(imprep.znorm(imgc[:,:,:,nuc_img_channel]).astype(float),axis=0)
            img_bf=skimage.filters.difference_of_gaussians(imgc[...,bf_img_channel],.5,1.5)
            img_bf=np.max(imprep.znorm(img_bf[:,:,:]).astype(float),axis=0)
            msk_cyto=np.max(mskc[:,:,:,cyto_msk_channel],axis=0)
            msk2_cyto=np.max(mskc2[:,:,:,cyto_msk_channel],axis=0)
        if nuchigh is None:
            nuclow=np.percentile(img_nuc,5);nuchigh=np.percentile(img_nuc,95)
        #img_nuc[msk_nuc==0]=-1000.
        #img_rep[msk_cyto==0]=-1000.1
        img_bf=np.abs(img_bf)
        #img_bf[np.logical_not(msk_f)]=0
        img_rep=scipy.ndimage.zoom(img_rep,zoom,order=1)
        img_rep=scipy.ndimage.gaussian_filter(img_rep,sigma)
        img_nuc=scipy.ndimage.zoom(img_nuc,zoom,order=1)
        img_nuc=scipy.ndimage.gaussian_filter(img_nuc,sigma)
        img_bf=scipy.ndimage.zoom(img_bf,zoom,order=1)
        if rephigh is None:
            replow=np.percentile(img_rep,5);rephigh=np.percentile(img_rep,95)
        if bfhigh is None:
            bflow=np.percentile(img_bf,5);bfhigh=np.percentile(img_bf,95)
        #img_bf=scipy.ndimage.gaussian_filter(img_bf,sigma)
        msk_cyto=scipy.ndimage.zoom(msk_cyto,zoom,order=1)
        msk2_cyto=scipy.ndimage.zoom(msk2_cyto,zoom,order=1)
        plt.imshow(img_bf,cmap=plt.cm.binary,clim=(bflow,bfhigh),alpha=1.)
        #csr=plt.contour(img_rep,levels=np.linspace(rep_min,rep_max,201),cmap=plt.cm.binary,extend='both')
        #csr.cmap.set_over('black')
        #csr.cmap.set_under('white')
        if show_neighbor_signals:
            plt.imshow(np.ma.masked_where(msk2_cyto==0,img_rep),cmap=plt.cm.inferno,alpha=.3)
        #else:
        plt.imshow(np.ma.masked_where(msk_cyto==0,img_rep),cmap=plt.cm.inferno,alpha=.7)
        #####plt.imshow(img_rep,cmap=plt.cm.YlOrBr,clim=(replow,3*rephigh),alpha=.3)
        #plt.contour(img_rep,levels=np.linspace(0,10,11),cmap=plt.cm.inferno,alpha=.3)
        #csn=plt.contour(img_nuc,levels=np.linspace(nuclow,nuchigh,4),cmap=plt.cm.Reds,alpha=0.6,linewidths=1)
        ##csn=plt.contour(msk2_cyto>0,levels=np.unique(msk_cyto>0),colors='orange',linewidths=1.2,alpha=.4)
        csn=plt.contour(msk_cyto>0,levels=np.unique(msk_cyto>0),colors='darkorange',linewidths=2)
        csn.cmap.set_over('darkred')
        csn.cmap.set_under('white')
        #cs.cmap.set_under('white')
        #plt.imshow(np.ma.masked_where(msk_f==0,np.ones_like(msk_f)),cmap=plt.cm.binary_r,clim=(0,1),alpha=0.15)
        #plt.xlim(viewsize*zoom/2-1*viewsize*zoom,viewsize*zoom/2+1*viewsize*zoom)
        #plt.ylim(viewsize*zoom/2-1*viewsize*zoom,viewsize*zoom/2+1.3*viewsize*zoom)
        #plt.text(zoom*npix/2,zoom*npix/4,r'central slice \textbf{YFP} \textcolor{red}{\textbf{Cy5}}')
        #plt.text(zoom*npix/2,zoom*npix/4,'central slice, YFP black, Cy5 red')
        plt.axis('off')
        return 0
    else: #except Exception as e:
        print(e)
    return 1

In [ ]:
nrounds=5
nr=1
maxl=trajl
stordir='cellstatepics'
os.system(f'mkdir -p {date}/{stordir}')
nuclow=-1.;nuchigh=9.;replow=0.;rephigh=5.;bflow=0.;bfhigh=5.
npics=1
for i_tm in range(nf):
    indmodels_tm=np.where(inds_tmSet_models==i_tm)[0]
    for iround in range(nrounds):
        plt.close('all')
        for istate in range(nstates):
            indcells_traj=inds_states[istate]
            indmodels=indtreatment_traj[indcells_traj]
            indtm=np.where(np.isin(indmodels,indmodels_tm))[0]
            indcells_model=cellinds1_traj[indcells_traj[indtm]]
            indmodels=indmodels[indtm]
            indr=np.random.choice(indcells_model.size,nr,replace=False)
            for ic in indr: #range(indcells_traj.size):
                plt.figure(figsize=(7,6))
                model_sctm=modelSet[indmodels[ic]]
                try:
                    bf_img_channel=np.where(model_sctm.channels==np.array(['BF']).astype('S32'))[0][0]
                except:
                    bf_img_channel=np.where(model_sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
                erk_img_channel=np.where(model_sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
                celltraj=model_sctm.get_cell_trajectory(indcells_model[ic])
                if celltraj.size>maxl:
                    celltraj=celltraj[-maxl:]
                iplot=0
                for icell in np.linspace(0,celltraj.size-1,npics).astype(int):
                    indcell=celltraj[icell]
                    plt.subplot(1,npics+1,iplot+1)
                    show_cell_activation(model_sctm,indcell,projection=True,bf_img_channel=bf_img_channel,nuc_img_channel=None,rep_img_channel=erk_img_channel,nuclow=nuclow,nuchigh=nuchigh,replow=replow,rephigh=rephigh,bflow=bflow,bfhigh=bfhigh,show_neighbor_signals=True)
                    iplot=iplot+1
                    plt.title(f'state {state_labels[istate]} {tmSet[i_tm]}')
                plt.savefig(f'{date}/{stordir}/state{state_labels[istate]}_cell{ic}_model{indmodels[ic]}_{model_sctm.figid.decode()}_{figid}.png')
                plt.pause(.1)

In [ ]:
import matplotlib.gridspec as gridspec
def notrans(arr):
    return arr

def rot_cw(arr):
    return np.fliplr(arr.T)

def rot2(arr):
    return np.flipud(arr)

def pcoord(x,n):
    return x

def ncoord(x,n):
    return n-x
    
transforms=[notrans,rot2,rot_cw]
coord_orders=[[0,1],[0,1],[1,0]]
htrans=[[pcoord,pcoord],[ncoord,pcoord],[pcoord,ncoord]]

In [ ]:
#new version rendered as image
vizpath=f'{date}/boundarystate_pics'
os.system(f'mkdir {vizpath}')
state_labels=np.array(list(string.ascii_uppercase))[0:nstates]
from adjustText import adjust_text
minc=2.
maxc=10.
import cmap
from cmap import Colormap
berlin=Colormap('berlin').to_mpl()
berlinlow=berlin(.5);berlinhigh=berlin(1.)
dxcmap=berlin=Colormap([berlinlow,berlinhigh]).to_mpl()
for imodel in [5]: #indgood_models: #[89,78,83,62,51,32,33,34,36]: #indgood_models:
    sctm=modelSet[imodel]
    #iframe=final_frames_model[imodel]-trajl
    for iS in [1,40]: #range(trajl+1,sctm.nt):
        iframe=initial_frames_model[imodel]+trajl+iS
        boundary_library=sctm.boundary_library.copy()
        cyto_msk_channel=np.where(sctm.mask_channels==np.array(['cyto']).astype('S32'))[0][0]
        nuc_img_channel=np.where(sctm.channels==np.array(['DAPI']).astype('S32'))[0][0]
        rep_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
        #bf_img_channel=np.where(sctm.channels==np.array(['BF']).astype('S32'))[0][0]
        erk_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
        akt_img_channel=np.where(sctm.channels==np.array(['TRITC']).astype('S32'))[0][0]
        #fra1_img_channel=np.where(sctm.channels==np.array(['CFP']).astype('S32'))[0][0]
        foreground_fmskchannel=np.where(sctm.fmask_channels==np.array(['foreground']).astype('S32'))[0][0]
        nuc_fmskchannel=np.where(sctm.fmask_channels==np.array(['nuc']).astype('S32'))[0][0]
        mskchannel=cyto_msk_channel; sctm.mskchannel=mskchannel
        cellinds=np.where(sctm.cells_frameSet==iframe)[0]
        img=sctm.get_image_data(iframe)
        msk=sctm.get_mask_data(iframe)
        #if len(sctm.fmask_channels)==2:
        #    fmsk_lung=sctm.get_fmask_data(iframe,channel=2)
        #    fmsk_cancer=sctm.get_fmask_data(iframe,channel=3)
        #if len(sctm.fmask_channels)==4:
        fmsk_lung=sctm.get_fmask_data(iframe,channel=4)
        fmsk_cancer=sctm.get_fmask_data(iframe,channel=5)
        surfaces=[fmsk_lung,fmsk_cancer];np.array(surfaces).shape
        surface_label0=np.all(np.logical_not(np.array(surfaces)).astype(bool),axis=0)
        fmsk_airway=surface_label0.copy()
        border_scale=(sctm.micron_per_pixel/sctm.boundary_library['border_resolution'])*np.array([sctm.zscale,1,1])
        #fmsk_lung=scipy.ndimage.zoom(test1,zoom=border_scale,order=0)
        #fmsk_cancer=scipy.ndimage.zoom(test2,zoom=border_scale,order=0)
        #fmsk_airway=scipy.ndimage.zoom(fmsk_airway,zoom=border_scale,order=0)
        #img_bf=img[...,bf_img_channel]
        img_rep=img[...,erk_img_channel]
        msk_cyto=msk[...,cyto_msk_channel]
        #img_bf=scipy.ndimage.zoom(img_bf,zoom=[sctm.zscale,1,1])
        #img_rep=scipy.ndimage.zoom(img[...,erk_img_channel],zoom=[sctm.zscale,1,1])
        msk_cyto=scipy.ndimage.zoom(msk[...,cyto_msk_channel],zoom=[sctm.zscale,1,1],order=0)
        fmsk_lung=scipy.ndimage.zoom(fmsk_lung,zoom=[sctm.zscale,1,1],order=0)
        #img_bf=skimage.filters.difference_of_gaussians(imprep.znorm(img_bf),.5,1.5)
        #img_bf=scipy.ndimage.zoom(img_bf,zoom=border_scale)
        #img_rep=scipy.ndimage.zoom(img[...,erk_img_channel],zoom=border_scale)
        #msk_cyto=scipy.ndimage.zoom(msk[...,cyto_msk_channel],zoom=border_scale,order=0)
        #plt.imshow(np.min(fmsk,axis=0).T)
        #fig,axs=plt.subplots(1,3,figsize=(15,5))
        fig = plt.figure(figsize=(10, 10))
        gs = gridspec.GridSpec(2, 2,width_ratios=[1,msk_cyto.shape[0]/msk_cyto.shape[1]],height_ratios=[1,msk_cyto.shape[0]/msk_cyto.shape[2]],
                               wspace=0.02, hspace=0.02)
        ax_z = fig.add_subplot(gs[0, 0])
        ax_y = fig.add_subplot(gs[0, 1]) #, sharey=ax_z)
        ax_x = fig.add_subplot(gs[1, 0]) #, sharex=ax_z)
        ax_blank = fig.add_subplot(gs[1, 1])  # unused
        ax_blank.axis('off')
        #axs.flatten()
        axs=[ax_z,ax_y,ax_x]
        for i_ax in range(3):
            ax=axs[i_ax]
            h_axs=np.setdiff1d(np.arange(3),[i_ax]).astype(int)
            #ax.contour(np.max(msk_cyto==sctm.cells_labelidSet[ic],axis=i_ax).T,colors='green',linewidth=3.,alpha=1.)
            nz=fmsk_lung.shape[i_ax]
            #ax.imshow(np.sum(fmsk_lung,axis=i_ax).T,cmap=plt.cm.gray_r,alpha=.8,clim=(0,nz))
            ax.imshow(transforms[i_ax](np.sum(fmsk_lung,axis=i_ax)).T,cmap=plt.cm.gray_r,alpha=.8,clim=(0,nz*.6))
            #ax.contour(np.max(msk_cyto,axis=i_ax).T,colors='black',linewidths=.6,alpha=1.)
            ax.contour(transforms[i_ax](np.max(msk_cyto,axis=i_ax)).T,colors='black',linewidths=.6,alpha=1.)
            #ax.imshow(np.ma.masked_where(np.max(msk_cyto,axis=i_ax).T==0,np.max(img_rep,axis=i_ax).T),cmap=plt.cm.hot,alpha=.8)
            #ax.imshow(np.ma.masked_where(np.max(msk_cyto,axis=i_ax).T==0,np.sum(msk_cyto>0,axis=i_ax).T),cmap=plt.cm.Greens,alpha=.5)
            #indboundary=np.where(np.isin(sctm.boundary_library['global_index'],cellinds))[0]
            #indc_boundary=sctm.boundary_library['global_index'][indboundary]
            cf=sctm.cancer_contact_fraction[cellinds]
            pf=sctm.primary_contact_fraction[cellinds]
            #mm=sctm.Xpca[cellinds,3] #0-ccl0,1-ccl1,2-ccl0,3-ccl1
            erkcn=sctm.x_signal[cellinds]
            indgood_erk=np.where(np.isfinite(erkcn))[0]
            erkcn[np.isnan(erkcn)]=0.
            dxcn=sctm.dx[cellinds]
            indgood_dx=np.where(np.isfinite(dxcn))[0]
            dxcn[np.isnan(dxcn)]=0.
            SD=sctm.S_r_death[cellinds]
            #ax.scatter(pts[:,h_axs[0]],pts[:,h_axs[1]],s=8,c=cf,cmap=plt.cm.Oranges,alpha=.2,vmin=0.,vmax=1.)
            #ax.scatter(pts[:,h_axs[0]],pts[:,h_axs[1]],s=8,c=pf,cmap=plt.cm.Greens,alpha=.2,vmin=0.,vmax=1.)
            #ax.scatter(pts[indgood_erk,h_axs[0]],pts[indgood_erk,h_axs[1]],s=8,c=erkcn[indgood_erk],cmap=plt.cm.hot,alpha=.2,vmin=0.3,vmax=2.5)
            #ax.scatter(pts[indgood_dx,h_axs[0]],pts[indgood_dx,h_axs[1]],s=8,c=dxcn[indgood_dx],cmap=dxcmap,alpha=.2,vmin=-2.,vmax=12.)
            msk_cyto_proj=np.max(msk_cyto,axis=i_ax)
            msk_cyto_proj_blank=np.ones_like(msk_cyto_proj)==0
            img=np.zeros_like(msk_cyto_proj).astype('float')
            #cellinds=cellinds[indgood_erk]
            for icell in range(cellinds.size):
                ic=cellinds[icell]
                indc=np.where(msk_cyto_proj==sctm.cells_labelidSet[ic])
                #img[indc]=cf[icell]
                #img[indc]=pf[icell]
                #img[indc]=SD[icell]
                #img[indc]=mm[icell]
                img[indc]=erkcn[icell]
                #img[indc]=erkcn[indgood_erk[icell]]
                #img[indc]=dxcn[indgood_dx[icell]]
                msk_cyto_proj_blank[indc]=True
            #ax.imshow(np.ma.masked_where(msk_cyto_proj==0,img).T,cmap=plt.cm.Oranges,alpha=.75,vmin=0.,vmax=1.)
            #ax.imshow(np.ma.masked_where(msk_cyto_proj_blank==0,img).T,cmap=plt.cm.Greens,alpha=.75,vmin=-2.,vmax=2.)
            #ax.imshow(transforms[i_ax](np.ma.masked_where(msk_cyto_proj_blank==0,img)).T,cmap=plt.cm.Oranges,alpha=.75,vmin=0.,vmax=1.)
            #ax.imshow(transforms[i_ax](np.ma.masked_where(msk_cyto_proj_blank==0,img)).T,cmap=plt.cm.hot,alpha=.75,vmin=0.,vmax=10.)
            ax.imshow(transforms[i_ax](np.ma.masked_where(msk_cyto_proj_blank==0,img)).T,cmap=plt.cm.hot,alpha=.75,vmin=.75,vmax=1.5)
            #ax.imshow(np.ma.masked_where(msk_cyto_proj_blank==0,img).T,cmap=dxcmap,alpha=.75,vmin=-2.,vmax=12.)
            #texts=[]
            testimg=transforms[i_ax](np.ma.masked_where(msk_cyto_proj_blank==0,img)).T
            nx=testimg.shape[0];ny=testimg.shape[1]
            for istate in range(nstates):
                indcells_traj=inds_states[istate]
                indcells_model=np.where(indtreatment_traj==imodel)[0]
                indcells_state_traj=np.intersect1d(indcells_traj,indcells_model)
                indcells_state_model=cellinds1_traj[indcells_state_traj]
                cellinds_state=np.intersect1d(cellinds,indcells_state_model)
                indboundary=np.where(np.isin(sctm.boundary_library['global_index'],cellinds_state))[0]
                pts=sctm.boundary_library['pts'][indboundary,:]
                pts[...,0]=(pts[...,0]/border_scale[0])*sctm.zscale
                pts[...,1]=pts[...,1]/border_scale[1]
                pts[...,2]=pts[...,2]/border_scale[2]
                #ax.scatter(pts[:,h_axs[0]],pts[:,h_axs[1]],s=3,c=plt.cm.Reds(istate/nstates),alpha=.2)
                for ic in cellinds_state:
                    indc=np.where(sctm.boundary_library['global_index'][indboundary]==ic)[0]
                    cpt=np.nanmean(pts[indc,:],axis=0)
                    ax.text(htrans[i_ax][0](cpt[h_axs[coord_orders[i_ax][0]]],ny),htrans[i_ax][1](cpt[h_axs[coord_orders[i_ax][1]]],nx),state_labels[istate],color='black')
                    #texts.append(ax.text(cpt[h_axs[0]],cpt[h_axs[1]],state_labels[istate],color='black'))
            #adjust_text(texts)
            ax.axis('off')
        plt.savefig(f'{vizpath}/boundarymap_ERK_m{imodel}_frame{iS:04d}.svg')
        plt.show()

In [ ]:
plt.imshow(img,cmap=plt.cm.hot,alpha=.75,vmin=.75,vmax=1.5);plt.colorbar();
plt.savefig(f'{date}/colorbar_ERK.svg')

In [ ]:
#new version rendered as image
vizpath=f'{date}/boundarystate_movies'
os.system(f'mkdir {vizpath}')
state_labels=np.array(list(string.ascii_uppercase))[0:nstates]
from adjustText import adjust_text
minc=2.
maxc=10.
import cmap
from cmap import Colormap
berlin=Colormap('berlin').to_mpl()
berlinlow=berlin(.5);berlinhigh=berlin(1.)
dxcmap=berlin=Colormap([berlinlow,berlinhigh]).to_mpl()
for imodel in [5]: #indgood_models: #[30,36,63,67,73,76,81]: #indgood_models: #[62,51,32,33,34,36]: #[89,78,83,#indgood_models:
    sctm=modelSet[imodel]
    for iS in [1,40]: #range(trajl+1,sctm.nt):
        iframe=initial_frames_model[imodel]+trajl+iS
        iframe=initial_frames_model[imodel]+trajl+iS
        boundary_library=sctm.boundary_library.copy()
        cyto_msk_channel=np.where(sctm.mask_channels==np.array(['cyto']).astype('S32'))[0][0]
        nuc_img_channel=np.where(sctm.channels==np.array(['DAPI']).astype('S32'))[0][0]
        rep_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
        #bf_img_channel=np.where(sctm.channels==np.array(['BF']).astype('S32'))[0][0]
        erk_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
        akt_img_channel=np.where(sctm.channels==np.array(['TRITC']).astype('S32'))[0][0]
        #fra1_img_channel=np.where(sctm.channels==np.array(['CFP']).astype('S32'))[0][0]
        foreground_fmskchannel=np.where(sctm.fmask_channels==np.array(['foreground']).astype('S32'))[0][0]
        nuc_fmskchannel=np.where(sctm.fmask_channels==np.array(['nuc']).astype('S32'))[0][0]
        mskchannel=cyto_msk_channel; sctm.mskchannel=mskchannel
        cellinds=np.where(sctm.cells_frameSet==iframe)[0]
        img=sctm.get_image_data(iframe)
        msk=sctm.get_mask_data(iframe)
        fmsk_lung=sctm.get_fmask_data(iframe,channel=4)
        fmsk_cancer=sctm.get_fmask_data(iframe,channel=5)
        surfaces=[fmsk_lung,fmsk_cancer];np.array(surfaces).shape
        surface_label0=np.all(np.logical_not(np.array(surfaces)).astype(bool),axis=0)
        fmsk_airway=surface_label0.copy()
        border_scale=(sctm.micron_per_pixel/sctm.boundary_library['border_resolution'])*np.array([sctm.zscale,1,1])
        #fmsk_lung=scipy.ndimage.zoom(test1,zoom=border_scale,order=0)
        #fmsk_cancer=scipy.ndimage.zoom(test2,zoom=border_scale,order=0)
        #fmsk_airway=scipy.ndimage.zoom(fmsk_airway,zoom=border_scale,order=0)
        #img_bf=img[...,bf_img_channel]
        img_rep=img[...,erk_img_channel]
        msk_cyto=msk[...,cyto_msk_channel]
        #img_bf=scipy.ndimage.zoom(img_bf,zoom=[sctm.zscale,1,1])
        #img_rep=scipy.ndimage.zoom(img[...,erk_img_channel],zoom=[sctm.zscale,1,1])
        msk_cyto=scipy.ndimage.zoom(msk[...,cyto_msk_channel],zoom=[sctm.zscale,1,1],order=0)
        fmsk_lung=scipy.ndimage.zoom(fmsk_lung,zoom=[sctm.zscale,1,1],order=0)
        #img_bf=skimage.filters.difference_of_gaussians(imprep.znorm(img_bf),.5,1.5)
        #img_bf=scipy.ndimage.zoom(img_bf,zoom=border_scale)
        #img_rep=scipy.ndimage.zoom(img[...,erk_img_channel],zoom=border_scale)
        #msk_cyto=scipy.ndimage.zoom(msk[...,cyto_msk_channel],zoom=border_scale,order=0)
        #plt.imshow(np.min(fmsk,axis=0).T)
        #fig,axs=plt.subplots(1,3,figsize=(15,5))
        fig = plt.figure(figsize=(10, 10))
        gs = gridspec.GridSpec(2, 2,width_ratios=[1,msk_cyto.shape[0]/msk_cyto.shape[1]],height_ratios=[1,msk_cyto.shape[0]/msk_cyto.shape[2]],
                               wspace=0.02, hspace=0.02)
        ax_z = fig.add_subplot(gs[0, 0])
        ax_y = fig.add_subplot(gs[0, 1]) #, sharey=ax_z)
        ax_x = fig.add_subplot(gs[1, 0]) #, sharex=ax_z)
        ax_blank = fig.add_subplot(gs[1, 1])  # unused
        ax_blank.axis('off')
        #axs.flatten()
        axs=[ax_z,ax_y,ax_x]
        for i_ax in range(3):
            ax=axs[i_ax]
            h_axs=np.setdiff1d(np.arange(3),[i_ax]).astype(int)
            #ax.contour(np.max(msk_cyto==sctm.cells_labelidSet[ic],axis=i_ax).T,colors='green',linewidth=3.,alpha=1.)
            nz=fmsk_lung.shape[i_ax]
            #ax.imshow(np.sum(fmsk_lung,axis=i_ax).T,cmap=plt.cm.gray_r,alpha=.8,clim=(0,nz))
            ax.imshow(transforms[i_ax](np.sum(fmsk_lung,axis=i_ax)).T,cmap=plt.cm.gray_r,alpha=.8,clim=(0,nz*.6))
            #ax.contour(np.max(msk_cyto,axis=i_ax).T,colors='black',linewidths=.6,alpha=1.)
            ax.contour(transforms[i_ax](np.max(msk_cyto,axis=i_ax)).T,colors='black',linewidths=.6,alpha=1.)
            #ax.imshow(np.ma.masked_where(np.max(msk_cyto,axis=i_ax).T==0,np.max(img_rep,axis=i_ax).T),cmap=plt.cm.hot,alpha=.8)
            #ax.imshow(np.ma.masked_where(np.max(msk_cyto,axis=i_ax).T==0,np.sum(msk_cyto>0,axis=i_ax).T),cmap=plt.cm.Greens,alpha=.5)
            #indboundary=np.where(np.isin(sctm.boundary_library['global_index'],cellinds))[0]
            #indc_boundary=sctm.boundary_library['global_index'][indboundary]
            cf=sctm.cancer_contact_fraction[cellinds]
            pf=sctm.primary_contact_fraction[cellinds]
            erkcn=sctm.x_signal[cellinds]
            indgood_erk=np.where(np.isfinite(erkcn))[0]
            erkcn[np.isnan(erkcn)]=0.
            dxcn=sctm.dx[cellinds]
            indgood_dx=np.where(np.isfinite(dxcn))[0]
            dxcn[np.isnan(dxcn)]=0.
            #ax.scatter(pts[:,h_axs[0]],pts[:,h_axs[1]],s=8,c=cf,cmap=plt.cm.Oranges,alpha=.2,vmin=0.,vmax=1.)
            #ax.scatter(pts[:,h_axs[0]],pts[:,h_axs[1]],s=8,c=pf,cmap=plt.cm.Greens,alpha=.2,vmin=0.,vmax=1.)
            #ax.scatter(pts[indgood_erk,h_axs[0]],pts[indgood_erk,h_axs[1]],s=8,c=erkcn[indgood_erk],cmap=plt.cm.hot,alpha=.2,vmin=0.3,vmax=2.5)
            #ax.scatter(pts[indgood_dx,h_axs[0]],pts[indgood_dx,h_axs[1]],s=8,c=dxcn[indgood_dx],cmap=dxcmap,alpha=.2,vmin=-2.,vmax=12.)
            msk_cyto_proj=np.max(msk_cyto,axis=i_ax)
            msk_cyto_proj_blank=np.ones_like(msk_cyto_proj)==0
            img=np.zeros_like(msk_cyto_proj).astype('float')
            #cellinds=cellinds[indgood_erk]
            #for icell in range(cellinds.size):
            for istate in range(nstates):
                indcells_traj=inds_states[istate]
                indcells_model=np.where(indtreatment_traj==imodel)[0]
                indcells_state_traj=np.intersect1d(indcells_traj,indcells_model)
                indcells_state_model=cellinds1_traj[indcells_state_traj]
                cellinds_state=np.intersect1d(cellinds,indcells_state_model)
                for icell in range(cellinds_state.size):
                    ic=cellinds_state[icell]
                    indc=np.where(msk_cyto_proj==sctm.cells_labelidSet[ic])
                    #img[indc]=cf[icell]
                    img[indc]=istate #pf[icell]
                    #img[indc]=erkcn[indgood_erk[icell]]
                    #img[indc]=dxcn[indgood_dx[icell]]
                    msk_cyto_proj_blank[indc]=True
            ax.imshow(transforms[i_ax](np.ma.masked_where(msk_cyto_proj_blank==0,img)).T,cmap=plt.cm.jet,alpha=.75,vmin=0,vmax=nstates-1)
            #ax.imshow(np.ma.masked_where(msk_cyto_proj==0,img).T,cmap=plt.cm.Oranges,alpha=.75,vmin=0.,vmax=1.)
            #ax.imshow(np.ma.masked_where(msk_cyto_proj_blank==0,img).T,cmap=plt.cm.Greens,alpha=.75,vmin=0.,vmax=1.)
            #ax.imshow(transforms[i_ax](np.ma.masked_where(msk_cyto_proj_blank==0,img)).T,cmap=plt.cm.Greens,alpha=.75,vmin=0.,vmax=1.)
            #ax.imshow(np.ma.masked_where(msk_cyto_proj_blank==0,img).T,cmap=plt.cm.hot,alpha=.75,vmin=0.3,vmax=2.)
            #ax.imshow(np.ma.masked_where(msk_cyto_proj_blank==0,img).T,cmap=dxcmap,alpha=.75,vmin=-2.,vmax=12.)
            #texts=[]
            testimg=transforms[i_ax](np.ma.masked_where(msk_cyto_proj_blank==0,img)).T
            nx=testimg.shape[0];ny=testimg.shape[1]
            for istate in range(nstates):
                indcells_traj=inds_states[istate]
                indcells_model=np.where(indtreatment_traj==imodel)[0]
                indcells_state_traj=np.intersect1d(indcells_traj,indcells_model)
                indcells_state_model=cellinds1_traj[indcells_state_traj]
                cellinds_state=np.intersect1d(cellinds,indcells_state_model)
                indboundary=np.where(np.isin(sctm.boundary_library['global_index'],cellinds_state))[0]
                pts=sctm.boundary_library['pts'][indboundary,:]
                pts[...,0]=(pts[...,0]/border_scale[0])*sctm.zscale
                pts[...,1]=pts[...,1]/border_scale[1]
                pts[...,2]=pts[...,2]/border_scale[2]
                #ax.scatter(pts[:,h_axs[0]],pts[:,h_axs[1]],s=3,c=plt.cm.Reds(istate/nstates),alpha=.2)
                for ic in cellinds_state:
                    indc=np.where(sctm.boundary_library['global_index'][indboundary]==ic)[0]
                    cpt=np.nanmean(pts[indc,:],axis=0)
                    ax.text(htrans[i_ax][0](cpt[h_axs[coord_orders[i_ax][0]]],ny),htrans[i_ax][1](cpt[h_axs[coord_orders[i_ax][1]]],nx),state_labels[istate],color='black')
                    #texts.append(ax.text(cpt[h_axs[0]],cpt[h_axs[1]],state_labels[istate],color='black'))
            #adjust_text(texts)
            ax.axis('off')
        ax_z.set_title(f'm{imodel}: {tmSet[inds_tmSet_models[imodel]]} {iframe*timestep:.1f} hrs')
        plt.savefig(f'{vizpath}/boundarymap_states_m{imodel}_{figid}_f{iframe:04d}.png')
        plt.savefig(f'{vizpath}/boundarymap_states_m{imodel}_{figid}_f{iframe:04d}.svg')
        plt.show()

In [ ]:
nf=len(tmSet)
state_probs=np.zeros((nf,n_states))
twindow=2 #hours
tu=16
tstep=1
tstart=int(np.ceil(.5*(trajl))) #+twindow
time_inds=np.arange(tstart,tu,tstep).astype(int)
state_probs_times=np.zeros((nf,n_states,time_inds.size))
ntimes=time_inds.size
itimes=0
counts0=np.ones(nf)*np.inf
for itt in time_inds:
    fl=itt
    fu=itt+twindow
    indstw=np.where(np.logical_and(times_traj<fu,times_traj>fl))[0]
    for i in range(nf):
        indstm=inds_conditions[i]
        indstwm=np.intersect1d(indstm,indstw)
        #indstwm=np.intersect1d(indstwm,indsrep)
        x0=x[indstwm,:]
        indc0=stateSet[clusters_minima.assign(x0)]
        statesc,counts=np.unique(indc0,return_counts=True)
        if itt==time_inds[0]:
            counts0[i]=np.sum(counts)
        #state_probs[i,statesc]=counts/np.sum(counts)
        state_probs[i,statesc]=counts/counts0[i]
    state_probs_times[:,:,itimes]=state_probs
    itimes=itimes+1

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
nplots=int(np.ceil(np.sqrt(nf)))
plt.figure(figsize=(10,20))
colors = ['white'] + plt.cm.Reds(np.linspace(0, 1, 256)).tolist()  # 'white' for 0, 'Reds' for others
cmap = ListedColormap(colors)
for i_tm in range(nf):
    plt.subplot(nplots,nplots,i_tm+1)
    plt.imshow(state_probs_times[i_tm,:,:].T,origin='lower',cmap=cmap,clim=(0,.65))
    ax=plt.gca()
    ax.set_yticks(np.arange(time_inds.size))
    ax.set_xticks(np.arange(nstates))
    ax.set_xticklabels(np.array(state_labels))
    ax.set_yticklabels(time_inds)
    plt.xlabel('state');
    plt.ylabel('time (hrs)')

    plt.title(tmSet[i_tm])
    plt.ylim(0-.5,time_inds.size-.5)
    
#cbar=plt.colorbar()
#cbar.set_label('state fraction')

#plt.savefig(date+f'/stateprobs_times_{figid}_nS{n_states}.png')
plt.savefig(os.path.join(date, f'stateprobs_times_{figid}.png'))
plt.savefig(os.path.join(date, f'stateprobs_times_{figid}.svg'), format='svg')   

In [ ]:
nf=len(tmSet)
state_feat=np.zeros((nf,n_states))
twindow=2 #hours
tu=16
tstep=1
time_inds=np.arange(1,tu,tstep).astype(int)

vsetList=[np.nanmean(cratio_erk_full_traj,axis=1),dx_traj]
captionList=['ERKactivity','motility']
#cmapList=[plt.cm.hot,plt.cm.
state_feat_times=np.zeros((nf,n_states,time_inds.size))
ntimes=time_inds.size
itimes=0
counts0=np.ones(nf)*np.inf
for itt in time_inds:
    fl=itt-twindow
    fu=itt+twindow
    indstw=np.where(np.logical_and(times_traj<fu,times_traj>fl))[0]
    for i in range(nf):
        indstm=inds_conditions[i]
        indstwm=np.intersect1d(indstm,indstw)
        #indstwm=np.intersect1d(indstwm,indsrep)
        x0=x[indstwm,:]
        #feat=np.nanmean(cratio_erk_full_traj,axis=1)[indstwm]
        feat=np.nanmean(dx_full_traj,axis=1)[indstwm]
        indc0=stateSet[clusters_minima.assign(x0)]
        state_feat=np.zeros((n_states))
        for istate in range(nstates):
            indstate=np.where(indc0==istate)[0]
            state_feat[istate]=np.nanmean(feat[indstate])
        state_feat_times[i,:,itimes]=state_feat
    itimes=itimes+1
    
nplots=int(np.ceil(np.sqrt(nf)))
colors = ['white'] + plt.cm.Reds(np.linspace(0, 1, 256)).tolist()  # 'white' for 0, 'Reds' for others
cmap = ListedColormap(colors)
for i_tm in range(nf):
    plt.figure(figsize=(6,6))
    plt.imshow(state_feat_times[i_tm,:,:].T,origin='lower',cmap=plt.cm.hot,clim=(1.,6.))#clim=(.75,1.25))
    ax=plt.gca()
    ax.set_yticks(np.arange(time_inds.size))
    ax.set_xticks(np.arange(nstates))
    ax.set_xticklabels(np.array(state_labels))
    ax.set_yticklabels(time_inds)
    plt.xlabel('state');
    plt.ylabel('time (hrs)')
    plt.title(tmSet[i_tm])
    plt.ylim(0-.5,time_inds.size-.5)
    cbar=plt.colorbar()
    cbar.set_label('motility')#ERK activity')
    plt.savefig(os.path.join(date, f'motility_times_{figid}_{tmSet[i_tm]}.png'))
    plt.savefig(os.path.join(date, f'motility_times_{figid}_{tmSet[i_tm]}.svg'), format='svg')   

#plt.savefig(date+f'/stateprobs_times_{figid}_nS{n_states}.png')


In [ ]:
nf=len(tmSet)
state_feat=np.zeros((nf,n_states))
twindow=2 #hours
tu=16
tstep=1
time_inds=np.arange(1,tu,tstep).astype(int)

vsetList=[np.nanmean(cratio_erk_full_traj,axis=1),dx_traj]
captionList=['ERKactivity','motility']
#cmapList=[plt.cm.hot,plt.cm.
state_feat_times=np.zeros((nf,n_states,time_inds.size))
ntimes=time_inds.size
itimes=0
counts0=np.ones(nf)*np.inf
for itt in time_inds:
    fl=itt-twindow
    fu=itt+twindow
    indstw=np.where(np.logical_and(times_traj<fu,times_traj>fl))[0]
    for i in range(nf):
        indstm=inds_conditions[i]
        indstwm=np.intersect1d(indstm,indstw)
        #indstwm=np.intersect1d(indstwm,indsrep)
        x0=x[indstwm,:]
        feat=np.nanmean(cratio_erk_full_traj,axis=1)[indstwm]
        #feat=np.nanmean(dx_full_traj,axis=1)[indstwm]
        indc0=stateSet[clusters_minima.assign(x0)]
        state_feat=np.zeros((n_states))
        for istate in range(nstates):
            indstate=np.where(indc0==istate)[0]
            state_feat[istate]=np.nanmean(feat[indstate])
        state_feat_times[i,:,itimes]=state_feat
    itimes=itimes+1
    
nplots=int(np.ceil(np.sqrt(nf)))
colors = ['white'] + plt.cm.Reds(np.linspace(0, 1, 256)).tolist()  # 'white' for 0, 'Reds' for others
cmap = ListedColormap(colors)
for i_tm in range(nf):
    plt.figure(figsize=(6,6))
    plt.imshow(state_feat_times[i_tm,:,:].T,origin='lower',cmap=plt.cm.hot,clim=(.8,1.15))
    ax=plt.gca()
    ax.set_yticks(np.arange(time_inds.size))
    ax.set_xticks(np.arange(nstates))
    ax.set_xticklabels(np.array(state_labels))
    ax.set_yticklabels(time_inds)
    plt.xlabel('state');
    plt.ylabel('time (hrs)')
    plt.title(tmSet[i_tm])
    plt.ylim(0-.5,time_inds.size-.5)
    cbar=plt.colorbar()
    cbar.set_label('ERK activity')
    plt.savefig(os.path.join(date, f'ERK_times_{figid}_{tmSet[i_tm]}.png'))
    plt.savefig(os.path.join(date, f'ERK_times_{figid}_{tmSet[i_tm]}.svg'), format='svg')   

#plt.savefig(date+f'/stateprobs_times_{figid}_nS{n_states}.png')

In [ ]:
nf=len(tmSet)
state_feat=np.zeros((nf,n_states))
twindow=2 #hours
tu=16
tstep=1
time_inds=np.arange(1,tu,tstep).astype(int)

vsetList=[np.nanmean(cratio_akt_full_traj,axis=1),dx_traj]
captionList=['AKTactivity','motility']
#cmapList=[plt.cm.hot,plt.cm.
state_feat_times=np.zeros((nf,n_states,time_inds.size))
ntimes=time_inds.size
itimes=0
counts0=np.ones(nf)*np.inf
for itt in time_inds:
    fl=itt-twindow
    fu=itt+twindow
    indstw=np.where(np.logical_and(times_traj<fu,times_traj>fl))[0]
    for i in range(nf):
        indstm=inds_conditions[i]
        indstwm=np.intersect1d(indstm,indstw)
        #indstwm=np.intersect1d(indstwm,indsrep)
        x0=x[indstwm,:]
        feat=np.nanmean(cratio_akt_full_traj,axis=1)[indstwm]
        #feat=np.nanmean(dx_full_traj,axis=1)[indstwm]
        indc0=stateSet[clusters_minima.assign(x0)]
        state_feat=np.zeros((n_states))
        for istate in range(nstates):
            indstate=np.where(indc0==istate)[0]
            state_feat[istate]=np.nanmean(feat[indstate])
        state_feat_times[i,:,itimes]=state_feat
    itimes=itimes+1
    
nplots=int(np.ceil(np.sqrt(nf)))
colors = ['white'] + plt.cm.Reds(np.linspace(0, 1, 256)).tolist()  # 'white' for 0, 'Reds' for others
cmap = ListedColormap(colors)
for i_tm in range(nf):
    plt.figure(figsize=(6,6))
    plt.imshow(state_feat_times[i_tm,:,:].T,origin='lower',cmap=plt.cm.hot,clim=(.8,1.15))
    ax=plt.gca()
    ax.set_yticks(np.arange(time_inds.size))
    ax.set_xticks(np.arange(nstates))
    ax.set_xticklabels(np.array(state_labels))
    ax.set_yticklabels(time_inds)
    plt.xlabel('state');
    plt.ylabel('time (hrs)')
    plt.title(tmSet[i_tm])
    plt.ylim(0-.5,time_inds.size-.5)
    cbar=plt.colorbar()
    cbar.set_label('AKT activity')
    plt.savefig(os.path.join(date, f'AKT_times_{figid}_{tmSet[i_tm]}.png'))
    plt.savefig(os.path.join(date, f'AKT_times_{figid}_{tmSet[i_tm]}.svg'), format='svg')   

#plt.savefig(date+f'/stateprobs_times_{figid}_nS{n_states}.png')

In [ ]:
nf=len(tmSet)
state_feat=np.zeros((nf,n_states))
twindow=2 #hours
tu=16
tstep=1
time_inds=np.arange(1,tu,tstep).astype(int)

vsetList=[np.nanmean(fra1_full_traj,axis=1),dx_traj]
captionList=['FRA1activity','motility']
#cmapList=[plt.cm.hot,plt.cm.
state_feat_times=np.zeros((nf,n_states,time_inds.size))
ntimes=time_inds.size
itimes=0
counts0=np.ones(nf)*np.inf
for itt in time_inds:
    fl=itt-twindow
    fu=itt+twindow
    indstw=np.where(np.logical_and(times_traj<fu,times_traj>fl))[0]
    for i in range(nf):
        indstm=inds_conditions[i]
        indstwm=np.intersect1d(indstm,indstw)
        #indstwm=np.intersect1d(indstwm,indsrep)
        x0=x[indstwm,:]
        feat=np.nanmean(fra1_full_traj,axis=1)[indstwm]
        feat[np.isinf(feat)]=np.nan
        #feat=np.nanmean(dx_full_traj,axis=1)[indstwm]
        indc0=stateSet[clusters_minima.assign(x0)]
        state_feat=np.zeros((n_states))
        for istate in range(nstates):
            indstate=np.where(indc0==istate)[0]
            state_feat[istate]=np.nanmean(feat[indstate])
        state_feat_times[i,:,itimes]=state_feat
    itimes=itimes+1
    
nplots=int(np.ceil(np.sqrt(nf)))
colors = ['white'] + plt.cm.Reds(np.linspace(0, 1, 256)).tolist()  # 'white' for 0, 'Reds' for others
cmap = ListedColormap(colors)
for i_tm in range(nf):
    plt.figure(figsize=(6,6))
    plt.imshow(state_feat_times[i_tm,:,:].T-np.nanmin(state_feat_times),origin='lower',cmap=plt.cm.hot,clim=(0.,3.5))
    ax=plt.gca()
    ax.set_yticks(np.arange(time_inds.size))
    ax.set_xticks(np.arange(nstates))
    ax.set_xticklabels(np.array(state_labels))
    ax.set_yticklabels(time_inds)
    plt.xlabel('state');
    plt.ylabel('time (hrs)')
    plt.title(tmSet[i_tm])
    plt.ylim(0-.5,time_inds.size-.5)
    cbar=plt.colorbar()
    cbar.set_label('FRA1 activity')
    plt.savefig(os.path.join(date, f'FRA1_times_{figid}_{tmSet[i_tm]}.png'))
    plt.savefig(os.path.join(date, f'FRA1_times_{figid}_{tmSet[i_tm]}.svg'), format='svg')   

#plt.savefig(date+f'/stateprobs_times_{figid}_nS{n_states}.png')

In [ ]:
import pandas as pd
import seaborn as sns
tminds=np.array([0,1,2,3]).astype(int)
stateprob_df = pd.DataFrame(data=state_probs[tminds,:], index=np.array(tmSet)[tminds], columns=state_labels)

# Plot the clustermap without row clustering to preserve the treatment order
hmap = sns.clustermap(stateprob_df, cmap="PuOr_r", cbar_kws={'label': 'state probability'}, row_cluster=True,col_cluster=False)

# Show the plot
plt.savefig(os.path.join(date, f'stateprobs_seaborn_{figid}.png'))
plt.show()


In [ ]:
cratio_erk_var_traj=np.std(cratio_erk_full_traj,axis=1)
outcut=4.
indstm_show=np.array([0,1,2,4]).astype(int) #only show controls
nshow=indstm_show.size
plt.clf()
plt.close('all')
imm=0
#vsetList=[Xpcat[:,imm] for imm in range(len(sctm.mm_property_names))]
#captionset=[sctm.mm_property_names[imm] for imm in range(len(sctm.mm_property_names))]
tl=0
tu=20
indstw=np.where(np.logical_and(times_traj<tu,times_traj>tl))[0]
inds_rep=np.where(rep_traj<1000)[0]
indstw=np.intersect1d(inds_rep,indstw)
vsetList=[cratio_erk_traj,np.nanmean(cratio_erk_full_traj,axis=1),np.nanmean(fra1_full_traj,axis=1),np.nanmean(cratio_akt_full_traj,axis=1),np.nanmean(Srd_full_traj,axis=1),np.nanmean(cancer_cf_full_traj,axis=1),np.nanmean(primary_cf_full_traj,axis=1),np.nanmean(matrix_cf_full_traj,axis=1),times_traj,cratio_erk_var_traj,dx_traj] #np.nanmean(dx_full_traj,axis=1)]#,dx_traj,beta_traj,alpha_traj,erk_akt_crosscorr]
captionset=['ERK activity last','ERK activity mean','fra1','AKT activity mean','deathproximity','cancer contact fraction','primary contact fraction','matrix contact fraction','time','ERKvariability','dx (pixels)'] #,'motility alignment (beta)','contact avoidance (alpha)','ERK-AKT timec ross-corr']
#vset=cb_ratio_traj

## Step 1: Calculate mean ERK activity and determine sorted order
#activity = np.nanmean(cancer_cf_full_traj,axis=1)
#mean_activity = []
#for i in range(n_states):
#    setv = activity[inds_states[i]]
#    indgood=np.where(np.isfinite(setv))[0]
#    setv = setv[indgood]
#    mean_activity.append(np.mean(setv))

# Get sorted indices based on mean ERK activity
sorted_indices = np.arange(n_states).astype(int) #np.argsort(mean_activity)
# Step 2: Use this sorted order for all variables in vsetList
for iv in range(len(vsetList)):
    if True: #try:
        vset = vsetList[iv]
        plt.figure(figsize=(8, 4))
        ax = plt.gca()
    
        # Plot violin plots in the order of sorted mean ERK activity
        for sorted_index in range(n_states):
            i = sorted_indices[sorted_index]  # Get the correct state index from the sorted order
            indstate=np.intersect1d(inds_states[i],indstw)
            #setv = vset[inds_states[i]]
            setv = vset[indstate]
            setv[np.isinf(setv)]=np.nan
            indgood=np.where(np.isfinite(setv))[0]
            indin=np.where(np.abs(imprep.znorm(setv))<outcut)[0]
            indgood=np.intersect1d(indgood,indin)
            setv = setv[indgood]
            indstate=indstate[indgood]
            ind_tm=inds_tmSet_models[indtreatment_traj[indstate]]
            indin_tm=np.where(np.isin(ind_tm,indstm_show))[0]
            ind_tm=ind_tm[indin_tm]
            setv = setv[indin_tm]
            for ishow in range(indstm_show.size):
                i_tm=indstm_show[ishow]
                if setv[ind_tm==i_tm].size>1:
                    vplot = ax.violinplot(setv[ind_tm==i_tm], positions=[nshow*sorted_index + 1+.5*ishow], showmedians=True, showextrema=False)
                    vplot['cmedians'].set_color('black')    
                    for pc in vplot['bodies']:
                        pc.set_facecolor(colorSet[i_tm])
                    xpoints=np.array(utilities.generate_xcoords_violin(vplot,[setv[ind_tm==i_tm]]))
                    plt.scatter(xpoints,setv[ind_tm==i_tm],s=1,c=colorSet[i_tm],marker='.',alpha=.2)

        # Reorder state labels based on the sorted order
        sorted_labels = [state_labels[i] for i in sorted_indices]
    
        ax.set_xticks(range(1, nshow*n_states + 1,nshow))
        ax.set_xticklabels(sorted_labels)
    
        plt.ylabel(captionset[iv])
        plt.xlabel('states')
        if captionset[iv][0:3]=='ERK':
            plt.ylim(.3,1.6)
            plt.plot([-100,100],[1,1],'k--',linewidth=.5,alpha=.2)
        if captionset[iv][0:2]=='dx':
            plt.ylim(0,14.)
        #if captionset[iv][0:2]=='fr':
        #    plt.ylim(-1.,3.5)
        # Save or display the figure
        #plt.savefig(date + '/' + captionset[iv] + '_tearplot_' + figid + '.png')
        # plt.savefig(date + '/' + captionset[iv] + '_tearplot_' + figid + '.pdf')
        plt.xlim(0,nshow*n_states)
        plt.savefig(os.path.join(date + '/' + captionset[iv] + f'_tearplot_2tm_tl{tl}tu{tu}_' + figid + '.png'))
        plt.savefig(os.path.join(date + '/' + captionset[iv] + f'_tearplot_2tm_tl{tl}tu{tu}_' + figid + '.svg'), format='svg')
        plt.pause(0.5)
    #except Exception as e:
    #    print(e)

In [ ]:
from scipy import stats
outcut=4.
i_tm_ctrl=0
indstm_show=np.array([0,1,2,3,4]).astype(int) #only show controls
nshow=indstm_show.size
plt.clf()
plt.close('all')
imm=0
#vsetList=[Xpcat[:,imm] for imm in range(len(sctm.mm_property_names))]
#captionset=[sctm.mm_property_names[imm] for imm in range(len(sctm.mm_property_names))]
tl=4
tu=12
indstw=np.where(np.logical_and(times_traj<tu,times_traj>tl))[0]
inds_rep=np.where(rep_traj<1000)[0]
indstw=np.intersect1d(inds_rep,indstw)
vsetList=[cratio_erk_traj,np.nanmean(cratio_erk_full_traj,axis=1),np.nanmean(fra1_full_traj,axis=1),np.nanmean(cratio_akt_full_traj,axis=1),np.nanmean(Srd_full_traj,axis=1),np.nanmean(cancer_cf_full_traj,axis=1),np.nanmean(primary_cf_full_traj,axis=1),np.nanmean(matrix_cf_full_traj,axis=1),times_traj,cratio_erk_var_traj,dx_traj] #np.nanmean(dx_full_traj,axis=1)]#,dx_traj,beta_traj,alpha_traj,erk_akt_crosscorr]
captionset=['ERK activity last','ERK activity mean','fra1','AKT activity mean','deathproximity','cancer contact fraction','primary contact fraction','matrix contact fraction','time','ERKvariability','dx (pixels)'] #,'motility alignment (beta)','contact avoidance (alpha)','ERK-AKT timec ross-corr']
#vset=cb_ratio_traj

## Step 1: Calculate mean ERK activity and determine sorted order
#activity = np.nanmean(cancer_cf_full_traj,axis=1)
#mean_activity = []
#for i in range(n_states):
#    setv = activity[inds_states[i]]
#    indgood=np.where(np.isfinite(setv))[0]
#    setv = setv[indgood]
#    mean_activity.append(np.mean(setv))

# Get sorted indices based on mean ERK activity
sorted_indices = np.arange(n_states).astype(int) #np.argsort(mean_activity)
# Step 2: Use this sorted order for all variables in vsetList
for iv in range(len(vsetList)):
    if True: #try:
        vset = vsetList[iv]
        plt.figure(figsize=(8, 4))
        ax = plt.gca()
    
        # Plot violin plots in the order of sorted mean ERK activity
        for sorted_index in range(n_states):
            i = sorted_indices[sorted_index]  # Get the correct state index from the sorted order
            indstate=np.intersect1d(inds_states[i],indstw)
            #setv = vset[inds_states[i]]
            setv = vset[indstate]
            setv[np.isinf(setv)]=np.nan
            indgood=np.where(np.isfinite(setv))[0]
            indin=np.where(np.abs(imprep.znorm(setv))<outcut)[0]
            indgood=np.intersect1d(indgood,indin)
            setv = setv[indgood]
            indstate=indstate[indgood]
            ind_tm=inds_tmSet_models[indtreatment_traj[indstate]]
            indin_tm=np.where(np.isin(ind_tm,indstm_show))[0]
            ind_tm=ind_tm[indin_tm]
            setv = setv[indin_tm]
            setv0=setv[ind_tm==i_tm_ctrl]
            for ishow in range(indstm_show.size):
                i_tm=indstm_show[ishow]
                if setv[ind_tm==i_tm].size>1:
                    vplot = ax.violinplot(setv[ind_tm==i_tm], positions=[nshow*sorted_index + 1+.5*ishow], showmedians=True, showextrema=False)
                    vplot['cmedians'].set_color('black')    
                    for pc in vplot['bodies']:
                        pc.set_facecolor(colorSet[i_tm])
                    #xpoints=np.array(utilities.generate_xcoords_violin(vplot,[setv[ind_tm==i_tm]]))
                    #plt.scatter(xpoints,setv[ind_tm==i_tm],s=1,c=colorSet[i_tm],marker='.',alpha=.2)
                    t_statistic, p_value = stats.ttest_ind(setv0, setv[ind_tm==i_tm], equal_var=False)
                    if i_tm!=i_tm_ctrl:
                        ax.text(nshow*sorted_index + 1+.5*ishow,.5*np.nanmax(setv)+.2*(1.-ishow/float(nshow))*np.nanmax(setv),f'{p_value:.2e}')

        # Reorder state labels based on the sorted order
        sorted_labels = [state_labels[i] for i in sorted_indices]
    
        ax.set_xticks(range(1, nshow*n_states + 1,nshow))
        ax.set_xticklabels(sorted_labels)
    
        plt.ylabel(captionset[iv])
        plt.xlabel('states')
        if captionset[iv][0:3]=='ERK':
            plt.ylim(.3,1.6)
            plt.plot([-100,100],[1,1],'k--',linewidth=.5,alpha=.2)
        if captionset[iv][0:2]=='dx':
            plt.ylim(0,14.)
        #if captionset[iv][0:2]=='fr':
        #    plt.ylim(-1.,3.5)
        # Save or display the figure
        #plt.savefig(date + '/' + captionset[iv] + '_tearplot_' + figid + '.png')
        # plt.savefig(date + '/' + captionset[iv] + '_tearplot_' + figid + '.pdf')
        plt.xlim(0,nshow*n_states)
        plt.savefig(os.path.join(date + '/' + captionset[iv] + f'_tearplot_pvals_tl{tl}tu{tu}_' + figid + '.png'))
        plt.savefig(os.path.join(date + '/' + captionset[iv] + f'_tearplot_pvals_tl{tl}tu{tu}_' + figid + '.svg'), format='svg')
        plt.pause(0.5)
    #except Exception as e:
    #    print(e)

In [ ]:
i_tm0=0
i_tm1=1
tl=4
tu=12
outcut=4.
indstw=np.where(np.logical_and(times_traj<tu,times_traj>tl))[0]
vset=np.nanmean(cratio_erk_full_traj,axis=1)
for i_tm0 in [0,1]:
    for i_tm1 in [0,1]:
        for istate0 in range(n_states):
            indstate=np.intersect1d(inds_states[istate0],indstw)
            setv = vset[indstate]
            setv[np.isinf(setv)]=np.nan
            indgood=np.where(np.isfinite(setv))[0]
            indin=np.where(np.abs(imprep.znorm(setv))<outcut)[0]
            indgood=np.intersect1d(indgood,indin)
            setv = setv[indgood]
            indstate=indstate[indgood]
            ind_tm=inds_tmSet_models[indtreatment_traj[indstate]]
            indstm=np.where(ind_tm==i_tm0)[0]
            setv0=setv[indstm]
            indmodels=indtreatment_traj[indstate[indstm]]
            means0=[]
            for imodel in np.unique(indmodels):
                means0.append(np.mean(setv0[indmodels==imodel]))

            states1=np.setdiff1d(np.arange(n_states).astype(int),[istate0])
            for istate1 in states1: #range(n_states):
                indstate=np.intersect1d(inds_states[istate1],indstw)
                setv = vset[indstate]
                setv[np.isinf(setv)]=np.nan
                indgood=np.where(np.isfinite(setv))[0]
                indin=np.where(np.abs(imprep.znorm(setv))<outcut)[0]
                indgood=np.intersect1d(indgood,indin)
                setv = setv[indgood]
                indstate=indstate[indgood]
                ind_tm=inds_tmSet_models[indtreatment_traj[indstate]]
                indstm=np.where(ind_tm==i_tm1)[0]
                setv=setv[indstm]
                indmodels=indtreatment_traj[indstate[indstm]]
                means=[]
                for imodel in np.unique(indmodels):
                    means.append(np.mean(setv[indmodels==imodel]))
                t_statistic, p_value = stats.ttest_ind(means0, means, equal_var=False)
                print(f'{tmSet[i_tm0]} {state_labels[istate0]} to {tmSet[i_tm1]} {state_labels[istate1]}: p-value: {p_value:.3e}')


In [ ]:
mm_properties_traj=[]
for imodel in indgood_models:
    sctm=modelSet[imodel]
    indmodel_traj=np.where(indtreatment_traj==imodel)[0]
    cellinds_indmodel_traj=cellinds1_traj[indmodel_traj]
    mm_properties_traj.append(sctm.mm_properties[cellinds_indmodel_traj,:])

mm_properties_traj=np.concatenate(mm_properties_traj,axis=0)

In [ ]:
outcut=5.
indstm_show=np.array([0,1,2]).astype(int) #only show controls
plt.clf()
plt.close('all')
imm=0
vsetList=[mm_properties_traj[:,imm] for imm in range(mm_properties_traj.shape[1])]
captionset=[sctm.mm_property_names[imm].decode() for imm in range(len(sctm.mm_property_names))]
#vsetList=[cratio_erk_traj,np.nanmean(cratio_erk_full_traj,axis=1),np.nanmean(cancer_cf_full_traj,axis=1),np.nanmean(primary_cf_full_traj,axis=1),np.nanmean(matrix_cf_full_traj,axis=1),indframes_traj,cratio_erk_var_traj,np.nanmean(dx_full_traj,axis=1)]#,dx_traj,beta_traj,alpha_traj,erk_akt_crosscorr]
#captionset=['ERK activity last','ERK activity mean','cancer contact fraction','primary contact fraction','matrix contact fraction','frame','ERKvariability','dx (pixels)'] #,'motility alignment (beta)','contact avoidance (alpha)','ERK-AKT timec ross-corr']
#vset=cb_ratio_traj

# Step 1: Calculate mean ERK activity and determine sorted order
activity = np.nanmean(cancer_cf_full_traj,axis=1)
mean_activity = []
for i in range(n_states):
    setv = activity[inds_states[i]]
    indgood=np.where(np.isfinite(setv))[0]
    setv = setv[indgood]
    mean_activity.append(np.mean(setv))

# Get sorted indices based on mean ERK activity
sorted_indices = np.arange(n_states).astype(int) #np.argsort(mean_activity)

# Step 2: Use this sorted order for all variables in vsetList
for iv in range(len(vsetList)):
    try:
        vset = vsetList[iv]
        plt.figure(figsize=(10, 4))
        ax = plt.gca()
    
        # Plot violin plots in the order of sorted mean ERK activity
        for sorted_index in range(n_states):
            i = sorted_indices[sorted_index]  # Get the correct state index from the sorted order
            setv = vset[inds_states[i]]
            indgood=np.where(np.isfinite(setv))[0]
            indin=np.where(imprep.znorm(setv)<outcut)[0]
            indgood=np.intersect1d(indgood,indin)
            setv = setv[indgood]
            ind_tm=inds_tmSet_models[indtreatment_traj[inds_states[i][indgood]]]
            indin_tm=np.where(np.isin(ind_tm,indstm_show))[0]
            ind_tm=ind_tm[indin_tm]
            setv = setv[indin_tm]
            for ishow in range(indstm_show.size):
                i_tm=indstm_show[ishow]
                vplot = ax.violinplot(setv[ind_tm==i_tm], positions=[2*sorted_index + 1+.5*(ishow-.5)], showmedians=True, showextrema=False)
                vplot['cmedians'].set_color('black')    
                for pc in vplot['bodies']:
                    pc.set_facecolor(colorSet[i_tm])
                xpoints=np.array(utilities.generate_xcoords_violin(vplot,[setv[ind_tm==i_tm]]))
                plt.scatter(xpoints,setv[ind_tm==i_tm],s=1,c=colorSet[i_tm],marker='.',alpha=.2)

        # Reorder state labels based on the sorted order
        sorted_labels = [state_labels[i] for i in sorted_indices]
    
        ax.set_xticks(range(1, 2*n_states + 1,2))
        ax.set_xticklabels(sorted_labels)
    
        plt.ylabel(captionset[iv])
        plt.xlabel('states')
        if captionset[iv][0:3]=='ERK':
            plt.ylim(.2,1.9)
            plt.plot([-100,100],[1,1],'k--',linewidth=.5,alpha=.2)
        if captionset[iv][0:2]=='dx':
            plt.ylim(0.,15.)
        # Save or display the figure
        #plt.savefig(date + '/' + captionset[iv] + '_tearplot_' + figid + '.png')
        # plt.savefig(date + '/' + captionset[iv] + '_tearplot_' + figid + '.pdf')
        plt.xlim(0,2*n_states)
        plt.savefig(os.path.join(date + '/' + captionset[iv] + '_tearplot_2tm_' + figid + '.png'))
        plt.savefig(os.path.join(date + '/' + captionset[iv] + '_tearplot_2tm_' + figid + '.svg'), format='svg')
        plt.pause(0.5)
    except Exception as e:
        print(e)

In [ ]:
plt.clf()
plt.close('all')
imm=0
i_tmr=1
vsetList=[times_death_mcl1i_traj]
captionset=['time of death (hrs)']
#vsetList=[cratio_erk_traj,np.nanmean(cratio_erk_full_traj,axis=1),np.nanmean(cancer_cf_full_traj,axis=1),np.nanmean(primary_cf_full_traj,axis=1),np.nanmean(matrix_cf_full_traj,axis=1),indframes_traj,cratio_erk_var_traj,np.nanmean(dx_full_traj,axis=1)]#,dx_traj,beta_traj,alpha_traj,erk_akt_crosscorr]
#captionset=['ERK activity last','ERK activity mean','cancer contact fraction','primary contact fraction','matrix contact fraction','frame','ERKvariability','dx (pixels)'] #,'motility alignment (beta)','contact avoidance (alpha)','ERK-AKT timec ross-corr']
#vset=cb_ratio_traj

states_death=stateSet[ind_clusters[inds_death_mcl1i_traj]]
for iv in range(len(vsetList)):
        vset = vsetList[iv]
        plt.figure(figsize=(8, 4))
        ax = plt.gca()
    
        # Plot violin plots in the order of sorted mean ERK activity
        for sorted_index in range(n_states):
            i = sorted_index  # Get the correct state index from the sorted order
            inds_death=states_death==i
            setv=vset[inds_death]
            setv=setv[setv>0]
            i_tm=0
            if setv.size>1:
                #plt.bar(2*sorted_index + 1,np.median(setv),color=colorSet[i_tmr],alpha=.3)
                vplot = ax.violinplot(setv, positions=[2*sorted_index + 1], showmedians=True, showextrema=False)
                vplot['cmedians'].set_color('black')    
                for pc in vplot['bodies']:
                    pc.set_facecolor('gray')
                    pc.set_alpha(.4)
                xpoints=np.array(utilities.generate_xcoords_violin(vplot,[setv]))
                plt.scatter(xpoints,setv,s=10,c=colorSet[i_tmr],marker='o',alpha=.8)

        # Reorder state labels based on the sorted order
        sorted_labels = [state_labels[i] for i in sorted_indices]
    
        ax.set_xticks(range(1, 2*n_states + 1,2))
        ax.set_xticklabels(sorted_labels)
    
        plt.ylabel(captionset[iv])
        plt.xlabel('states')
        plt.xlim(0,2*n_states)
        plt.ylim(0,20)
        plt.savefig(os.path.join(date + '/timeofdeath_tearplot_2tm_' + figid + '.png'))
        plt.savefig(os.path.join(date + '/timeofdeath_tearplot_2tm_' + figid + '.svg'), format='svg')
        plt.pause(0.5)

In [ ]:
i_tm=1
indmodels=np.where(inds_tmSet_models==i_tm)[0]
indmodels=np.intersect1d(indmodels,indgood_models)
inds_death_traj=[]
times_death_traj=[]
inds_initial_traj=[]
inds_resistant_traj=[]
fw=4
for imodel in indmodels:
    indcells_traj_model=np.where(indtreatment_traj==imodel)[0]
    cellinds1_traj_model=cellinds1_traj[indcells_traj_model]
    sctm=modelSet[imodel]
    cellinds_death=sctm.inds_death.copy()
    print(sctm.cells_frameSet[sctm.inds_death])
    inds_death_traj.append(indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)])
    times_death_traj.append(times_traj[indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)]])#timestep_model[imodel]*(indframes_traj[indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)]-initial_frames_model[imodel]]))
    fl=initial_frames_model[imodel]+trajl+1
    fu=final_frames_model[imodel] #frames for time window
    inds_initial_traj.append(np.intersect1d(np.where(np.logical_and(indframes_traj<fl+fw,indframes_traj>fl))[0],indcells_traj_model))
    inds_resistant_traj.append(np.intersect1d(np.where(np.logical_and(indframes_traj<fu,indframes_traj>fu-fw))[0],indcells_traj_model))
    
inds_death_traj=np.concatenate(inds_death_traj)
times_death_traj=np.concatenate(times_death_traj)
inds_initial_traj=np.concatenate(inds_initial_traj)
inds_resistant_traj=np.concatenate(inds_resistant_traj)

In [ ]:
plt.clf()
plt.close('all')
imm=0
for i_tmr in range(nf):
    i_tm=i_tmr
    indmodels=np.where(inds_tmSet_models==i_tm)[0]
    indmodels=np.intersect1d(indmodels,indgood_models)
    inds_death_traj=[]
    times_death_traj=[]
    inds_initial_traj=[]
    inds_resistant_traj=[]
    fw=4
    times_all_traj=[]
    inds_all_traj=[]
    for imodel in indmodels:
        indcells_traj_model=np.where(indtreatment_traj==imodel)[0]
        cellinds1_traj_model=cellinds1_traj[indcells_traj_model]
        sctm=modelSet[imodel]
        fl=initial_frames_model[imodel] #+trajl+1
        fu=final_frames_model[imodel]-8 #frames for time window
        cellinds_death=sctm.inds_death.copy()
        inds_death=indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)]
        inds_death=np.intersect1d(np.where(np.logical_and(indframes_traj<fu,indframes_traj>fl))[0],inds_death)
        inds_death_traj.append(inds_death)
        times_death_traj.append(times_traj[inds_death])#timestep_model[imodel]*(indframes_traj[indcells_traj_model[np.isin(cellinds1_traj_model,cellinds_death)]-initial_frames_model[imodel]]))
        inds_initial_traj.append(np.intersect1d(np.where(np.logical_and(indframes_traj<fl+fw,indframes_traj>fl))[0],indcells_traj_model))
        inds_resistant_traj.append(np.intersect1d(np.where(np.logical_and(indframes_traj<fu,indframes_traj>fu-fw))[0],indcells_traj_model))
        inds_traj=np.intersect1d(np.where(np.logical_and(indframes_traj<fu,indframes_traj>fl))[0],indcells_traj_model)
        inds_all_traj.append(inds_traj)
        times_all_traj.append(times_traj[inds_traj])
    inds_all_traj=np.concatenate(inds_all_traj)
    times_all_traj=np.concatenate(times_all_traj)
    inds_death_traj=np.concatenate(inds_death_traj)
    times_death_traj=np.concatenate(times_death_traj)
    inds_initial_traj=np.concatenate(inds_initial_traj)
    inds_resistant_traj=np.concatenate(inds_resistant_traj)
    vsetList=[times_death_traj]
    caption='time of death (hrs)'
    #vsetList=[cratio_erk_traj,np.nanmean(cratio_erk_full_traj,axis=1),np.nanmean(cancer_cf_full_traj,axis=1),np.nanmean(primary_cf_full_traj,axis=1),np.nanmean(matrix_cf_full_traj,axis=1),indframes_traj,cratio_erk_var_traj,np.nanmean(dx_full_traj,axis=1)]#,dx_traj,beta_traj,alpha_traj,erk_akt_crosscorr]
    #captionset=['ERK activity last','ERK activity mean','cancer contact fraction','primary contact fraction','matrix contact fraction','frame','ERKvariability','dx (pixels)'] #,'motility alignment (beta)','contact avoidance (alpha)','ERK-AKT timec ross-corr']
    #vset=cb_ratio_traj

    states_death=stateSet[ind_clusters[inds_death_traj]]
    states_all=stateSet[ind_clusters[inds_all_traj]]
    #erk_death=np.nanmean(cratio_erk_full_traj[inds_death_traj],axis=1)
    erk_death=np.ones(inds_death_traj.size)*np.nan
    for ideath in range(inds_death_traj.size):
        imodel=indtreatment_traj[inds_death_traj[ideath]]
        sctm=modelSet[imodel]
        ic=cellinds1_traj[inds_death_traj[ideath]]
        erktraj=sctm.x_signal[sctm.get_cell_trajectory(ic)][-8:]
        good1=np.isfinite(erktraj)
        if np.sum(good1)>0:
            #erk_model1[i]=erktraj1[np.where(np.isfinite(erktraj1))[0][-1]]
            erk_death[ideath]=erktraj[np.where(np.isfinite(erktraj))[0][-1]]
    erk_all=np.nanmean(cratio_erk_full_traj[inds_all_traj],axis=1)
    #for iv in range(len(vsetList)):
    vset = times_death_traj
    plt.figure(figsize=(8, 4))
    ax = plt.gca()
    
    # Plot violin plots in the order of sorted mean ERK activity
    for sorted_index in range(n_states):
        i = sorted_index  # Get the correct state index from the sorted order
        inds_death=np.where(states_death==i)[0]
        inds_all=np.where(states_all==i)[0]
        setv=vset[inds_death]
        setv_erk=erk_death[inds_death[setv>0]]
        erk_state=erk_all[inds_all]
        setv=setv[setv>0]
        i_tm=0
        if setv.size>1:
            #plt.bar(2*sorted_index + 1,np.median(setv),color=colorSet[i_tmr],alpha=.2)
            vplot = ax.violinplot(setv, positions=[2*sorted_index + 1], showmedians=False, showextrema=False,widths=state_probs_times[i_tmr,i,0])
            #vplot['cmedians'].set_color('black')    
            for pc in vplot['bodies']:
                pc.set_facecolor('white')
                pc.set_alpha(0.)
            xpoints=np.array(utilities.generate_xcoords_violin(vplot,[setv]))
            ind_erkhigh=setv_erk>midcut
            ind_erklow=setv_erk<midcut
            ind_erknan=np.isnan(setv_erk)
            ax.scatter(xpoints[ind_erknan],setv[ind_erknan],s=10,c='black',marker='o',alpha=.3)
            ax.scatter(xpoints[ind_erkhigh]-.25,setv[ind_erkhigh],s=10,c='orangered',marker='x',alpha=.8)
            ax.scatter(xpoints[ind_erklow]+.25,setv[ind_erklow],s=10,c='darkblue',marker='x',alpha=.8)
            #plt.scatter(xpoints[np.isfinite(setv_erk)],setv[np.isfinite(setv_erk)],s=10,c=setv_erk[np.isfinite(setv_erk)],cmap=plt.cm.hot,clim=(lowcut,highcut),marker='o',alpha=.8)
        widths_on=np.sum(erk_state>midcut)
        widths_off=np.sum(erk_state<midcut)
        widths_sum=widths_off+widths_on
        widths_on=widths_on/widths_sum
        widths_off=widths_off/widths_sum
        vplot = ax.violinplot(times_all_traj[inds_all[erk_state>midcut]], positions=[2*sorted_index + .75], showmedians=False, widths=2.*widths_on*state_probs_times[i_tmr,i,0],showextrema=False)
        #vplot['cmedians'].set_color('black')    
        for pc in vplot['bodies']:
            pc.set_facecolor('orange')
            pc.set_alpha(.4)
        vplot = ax.violinplot(times_all_traj[inds_all[erk_state<midcut]], positions=[2*sorted_index + 1.25], widths=2.*widths_off*state_probs_times[i_tmr,i,0],showmedians=False, showextrema=False)
        #vplot['cmedians'].set_color('black')    
        for pc in vplot['bodies']:
            pc.set_facecolor('gray')
            pc.set_alpha(.4)

    # Reorder state labels based on the sorted order
    sorted_labels = [state_labels[i] for i in range(n_states)]

    ax.set_xticks(range(1, 2*n_states + 1,2))
    ax.set_xticklabels(sorted_labels)

    plt.ylabel(caption)

    plt.xlabel('states')
    plt.xlim(0,2*n_states)
    plt.ylim(0,16)
    plt.title(tmSet[i_tmr])
    plt.savefig(os.path.join(date + f'/timeofdeath_tearplot_erk_{tmSet[i_tmr]}_' + figid + '.png'))
    plt.savefig(os.path.join(date + f'/timeofdeath_tearplot_erk_{tmSet[i_tmr]}_' + figid + '.svg'))
    plt.pause(0.5)

In [ ]:
times_death_set=[None]*nf
inds_death_set=[None]*nf
erk_activity_set=[None]*nf
times_set=[None]*nf
for i_tm in range(nf):
        indmodels=np.where(inds_tmSet_models==i_tm)[0]
        indmodels=np.intersect1d(indmodels,indgood_models)
        inds_death=[]
        times_death=[]
        times_live=[]
        erk_activity=[]
        for imodel in indmodels:
            sctm=modelSet[imodel]
            cellinds_death=sctm.inds_death.copy()
            if np.isscalar(cellinds_death):
                cellinds_death=np.array([cellinds_death])
            fl=initial_frames_model[imodel]
            timestep=timestep_model[imodel]
            if cellinds_death.size>0:
                times_death.append(timestep*(sctm.cells_frameSet[cellinds_death]-fl))
                inds_death.append(sctm.cells_frameSet[cellinds_death])
            times_live.append(timestep*(sctm.cells_frameSet-fl))
            erk_activity.append(sctm.x_signal)
        if len(times_death)>0:
            times_death=np.concatenate(times_death)
            inds_death=np.concatenate(inds_death)
        times_live=np.concatenate(times_live)
        erk_activity=np.concatenate(erk_activity)
        times_death_set[i_tm]=times_death.copy()
        inds_death_set[i_tm]=inds_death.copy()
        times_set[i_tm]=times_live.copy()
        erk_activity_set[i_tm]=erk_activity.copy()

In [ ]:
tf=20.0
tl=0.0
dt=0.5
indtm=2 #np.where(np.array(tmSet)=='Mcl1i')[0][0]
times_death=times_death_set[indtm].copy()
erk_activity=erk_activity_set[indtm].copy()
times_death=times_death[np.logical_and(times_death<tf,times_death>tl)]
times_live=times_set[indtm].copy()
inds_live=np.where(np.logical_and(times_live<tf,times_live>tl))[0]
times_live=times_live[inds_live]
erk_activity=erk_activity[inds_live]
times_death=np.sort(times_death)
time_set=np.arange(tl,tf,dt)
times_death_fraction=np.zeros_like(time_set)
erk_mean=np.zeros_like(time_set)
fraction_on=np.zeros_like(time_set)
for itt in range(time_set.size):
    tu=time_set[itt]
    times_death_fraction[itt]=np.sum(times_death<=tu)
    indstw=np.where(np.logical_and(times_live>tu-dt,times_live<=tu))[0]
    erk_mean[itt]=np.nanmean(erk_activity[indstw])
    fraction_on[itt]=np.sum(erk_activity[indstw]>midcut)/np.sum(np.isfinite(erk_activity[indstw]))
times_death_fraction=times_death_fraction/times_death_fraction[-1]
death_rate=np.gradient(times_death_fraction,time_set)

In [ ]:
tf=18.0
tl=0.
dt=1.0
indtm=np.where(np.array(tmSet)=='Mcl1i')[0][0]
times_death=times_death_set[indtm].copy()
erk_activity=erk_activity_set[indtm].copy()
times_death=times_death[np.logical_and(times_death<tf,times_death>tl)]
times_live=times_set[indtm].copy()
inds_live=np.where(np.logical_and(times_live<tf,times_live>=tl))[0]
times_live=times_live[inds_live]
erk_activity=erk_activity[inds_live]
times_death=np.sort(times_death)
time_set=np.arange(tl,tf,dt)
times_death_fraction=np.zeros_like(time_set)
erk_mean=np.zeros_like(time_set)
fraction_on=np.zeros_like(time_set)
for itt in range(time_set.size):
    tu=time_set[itt]
    times_death_fraction[itt]=np.sum(times_death<=tu)
    indstw=np.where(np.logical_and(times_live>tu-dt,times_live<=tu))[0]
    erk_mean[itt]=np.nanmean(erk_activity[indstw])
    fraction_on[itt]=np.sum(erk_activity[indstw]>midcut)/np.sum(np.isfinite(erk_activity[indstw]))
erk_mean0=erk_mean.copy()
times_death_fraction=times_death_fraction/times_death_fraction[-1]
death_rate=np.gradient(times_death_fraction,time_set)
death_rate0=death_rate.copy()
death_rate_m=(death_rate-np.nanmin(death_rate))/(np.nanmax(death_rate)-np.nanmin(death_rate))
erk_mean_m=erk_mean #(erk_mean-np.nanmin(erk_mean))/(np.nanmax(erk_mean)-np.nanmin(erk_mean))
time_set_smth=np.linspace(time_set[0],time_set[-1],201)
spl_death=csaps.CubicSmoothingSpline(time_set[np.isfinite(death_rate_m)], death_rate_m[np.isfinite(death_rate_m)], smooth=.7)
death_rate_smth=spl_death(time_set_smth)    
spl_erk=csaps.CubicSmoothingSpline(time_set[np.isfinite(erk_mean_m)], erk_mean_m[np.isfinite(erk_mean_m)], smooth=.7)
erk_mean_smth=spl_erk(time_set_smth)
fig,axs=plt.subplots(2,1,sharex=True,figsize=(8,5))
label0=tmSet[indtm]
axs[0].scatter(time_set,death_rate_m,s=30,c=colorSet[indtm],alpha=.3,marker='o',label=label0)
axs[0].plot(time_set_smth,death_rate_smth,'-',color=colorSet[indtm],linewidth=1.5)
axs[0].set_ylabel('death rate (min-max)')
axs[0].legend(loc='upper right')
label1=tmSet[indtm]
axs[1].scatter(time_set,erk_mean_m,s=30,c=colorSet[indtm],alpha=.3,marker='o',label=label1)
axs[1].plot(time_set_smth,erk_mean_smth,'-',color=colorSet[indtm],linewidth=1.5)
axs[1].set_ylabel('ERK activity (min-max)')
axs[1].legend(loc='lower right')
axs[1].set_xlabel('time (hrs)')
fig.tight_layout()

indtm=np.where(np.array(tmSet)=='Pazo+Mcl1i')[0][0]
times_death=times_death_set[indtm].copy()
erk_activity=erk_activity_set[indtm].copy()
times_death=times_death[np.logical_and(times_death<tf,times_death>tl)]
times_live=times_set[indtm].copy()
inds_live=np.where(np.logical_and(times_live<tf,times_live>=tl))[0]
times_live=times_live[inds_live]
erk_activity=erk_activity[inds_live]
times_death=np.sort(times_death)
time_set=np.arange(tl,tf,dt)
times_death_fraction=np.zeros_like(time_set)
erk_mean=np.zeros_like(time_set)
fraction_on=np.zeros_like(time_set)
for itt in range(time_set.size):
    tu=time_set[itt]
    times_death_fraction[itt]=np.sum(times_death<=tu)
    indstw=np.where(np.logical_and(times_live>tu-dt,times_live<=tu))[0]
    erk_mean[itt]=np.nanmean(erk_activity[indstw])
    fraction_on[itt]=np.sum(erk_activity[indstw]>midcut)/np.sum(np.isfinite(erk_activity[indstw]))
times_death_fraction=times_death_fraction/times_death_fraction[-1]
death_rate=np.gradient(times_death_fraction,time_set)
death_rate_m=(death_rate-np.nanmin(death_rate))/(np.nanmax(death_rate)-np.nanmin(death_rate))
erk_mean_m=erk_mean #(erk_mean-np.nanmin(erk_mean0))/(np.nanmax(erk_mean0)-np.nanmin(erk_mean0))
time_set_smth=np.linspace(time_set[0],time_set[-1],201)
spl_death=csaps.CubicSmoothingSpline(time_set[np.isfinite(death_rate_m)], death_rate_m[np.isfinite(death_rate_m)], smooth=.7)
death_rate_smth=spl_death(time_set_smth)    
spl_erk=csaps.CubicSmoothingSpline(time_set[np.isfinite(erk_mean_m)], erk_mean_m[np.isfinite(erk_mean_m)], smooth=.7)
erk_mean_smth=spl_erk(time_set_smth)
label0=tmSet[indtm]
axs[0].scatter(time_set,death_rate_m,s=30,c=colorSet[indtm],alpha=.3,marker='o',label=label0)
axs[0].plot(time_set_smth,death_rate_smth,'-',color=colorSet[indtm],linewidth=1.5)
axs[0].set_ylabel('death rate (min-max)')
axs[0].legend(loc='upper right')
axs[0].set_xlim(-.2,16.5)
label1=tmSet[indtm]
axs[1].scatter(time_set,erk_mean_m,s=30,c=colorSet[indtm],alpha=.3,marker='o',label=label1)
axs[1].plot(time_set_smth,erk_mean_smth,'-',color=colorSet[indtm],linewidth=1.5)

indtm=np.where(np.array(tmSet)=='AZD4547+Mcl1i')[0][0]
times_death=times_death_set[indtm].copy()
erk_activity=erk_activity_set[indtm].copy()
times_death=times_death[np.logical_and(times_death<tf,times_death>tl)]
times_live=times_set[indtm].copy()
inds_live=np.where(np.logical_and(times_live<tf,times_live>=tl))[0]
times_live=times_live[inds_live]
erk_activity=erk_activity[inds_live]
times_death=np.sort(times_death)
time_set=np.arange(tl,tf,dt)
times_death_fraction=np.zeros_like(time_set)
erk_mean=np.zeros_like(time_set)
fraction_on=np.zeros_like(time_set)
for itt in range(time_set.size):
    tu=time_set[itt]
    times_death_fraction[itt]=np.sum(times_death<=tu)
    indstw=np.where(np.logical_and(times_live>tu-dt,times_live<=tu))[0]
    erk_mean[itt]=np.nanmean(erk_activity[indstw])
    fraction_on[itt]=np.sum(erk_activity[indstw]>midcut)/np.sum(np.isfinite(erk_activity[indstw]))
times_death_fraction=times_death_fraction/times_death_fraction[-1]
death_rate=np.gradient(times_death_fraction,time_set)
death_rate_m=(death_rate-np.nanmin(death_rate))/(np.nanmax(death_rate)-np.nanmin(death_rate))
erk_mean_m=erk_mean #(erk_mean-np.nanmin(erk_mean0))/(np.nanmax(erk_mean0)-np.nanmin(erk_mean0))
time_set_smth=np.linspace(time_set[0],time_set[-1],201)
spl_death=csaps.CubicSmoothingSpline(time_set[np.isfinite(death_rate_m)], death_rate_m[np.isfinite(death_rate_m)], smooth=.7)
death_rate_smth=spl_death(time_set_smth)    
spl_erk=csaps.CubicSmoothingSpline(time_set[np.isfinite(erk_mean_m)], erk_mean_m[np.isfinite(erk_mean_m)], smooth=.7)
erk_mean_smth=spl_erk(time_set_smth)
label0=tmSet[indtm]
axs[0].scatter(time_set,death_rate_m,s=30,c=colorSet[indtm],alpha=.3,marker='o',label=label0)
axs[0].plot(time_set_smth,death_rate_smth,'-',color=colorSet[indtm],linewidth=1.5)
axs[0].set_ylabel('death rate (min-max)')
axs[0].legend(loc='upper right')
axs[0].set_xlim(-.2,16.5)
label1=tmSet[indtm]
axs[1].scatter(time_set,erk_mean_m,s=30,c=colorSet[indtm],alpha=.3,marker='o',label=label1)
axs[1].plot(time_set_smth,erk_mean_smth,'-',color=colorSet[indtm],linewidth=1.5)

axs[1].set_ylabel('ERK activity (c/n ratio)')
axs[1].legend(loc='center right')
axs[1].set_xlabel('time (hrs)')
axs[1].set_xlim(-.2,16.5)
plt.savefig(f'{date}/combo_comp_deaths_ERK_time.svg')

In [ ]:
def get_erk_death_info(i_tm,random_shuffle=False,tl=0,tf=20,norm_death=True):
    indmodels=np.where(inds_tmSet_models==i_tm)[0]
    indmodels=np.intersect1d(indmodels,indgood_models)
    inds_death=[]
    times_death=[]
    times_live=[]
    erk_activity=[]
    if random_shuffle:
        indmodels = np.random.choice(indmodels, size=indmodels.size, replace=True)
    for imodel in indmodels:
        sctm=modelSet[imodel]
        cellinds_death=sctm.inds_death.copy()
        if np.isscalar(cellinds_death):
            cellinds_death=np.array([cellinds_death])
        fl=initial_frames_model[imodel]
        timestep=timestep_model[imodel]
        if cellinds_death.size>0:
            times_death.append(timestep*(sctm.cells_frameSet[cellinds_death]-fl))
            inds_death.append(sctm.cells_frameSet[cellinds_death])
        times_live.append(timestep*(sctm.cells_frameSet-fl))
        erk_activity.append(sctm.x_signal)
    if len(times_death)>0:
        times_death=np.concatenate(times_death)
        inds_death=np.concatenate(inds_death)
    times_live=np.concatenate(times_live)
    erk_activity=np.concatenate(erk_activity)
    times_death=times_death[np.logical_and(times_death<tf,times_death>tl)]
    inds_live=np.where(np.logical_and(times_live<tf,times_live>=tl))[0]
    times_live=times_live[inds_live]
    erk_activity=erk_activity[inds_live]
    times_death=np.sort(times_death)
    time_set=np.arange(tl,tf,dt)
    times_death_fraction=np.zeros_like(time_set)
    erk_mean=np.zeros_like(time_set)
    fraction_on=np.zeros_like(time_set)
    for itt in range(time_set.size):
        tu=time_set[itt]
        times_death_fraction[itt]=np.sum(times_death<=tu)
        indstw=np.where(np.logical_and(times_live>tu-dt,times_live<=tu))[0]
        erk_mean[itt]=np.nanmean(erk_activity[indstw])
        fraction_on[itt]=np.sum(erk_activity[indstw]>midcut)/np.sum(np.isfinite(erk_activity[indstw]))
    times_death_fraction=times_death_fraction/times_death_fraction[-1]
    death_rate=np.gradient(times_death_fraction,time_set)
    if norm_death:
        death_rate=(death_rate-np.nanmin(death_rate))/(np.nanmax(death_rate)-np.nanmin(death_rate))
    return erk_mean,death_rate

In [ ]:
tf=20.0
tl=0.
dt=1.0
time_set=np.arange(tl,tf,dt)
tms=[1,2,4]
fig,axs=plt.subplots(2,1,sharex=True,figsize=(8,5))

for i_tm in tms:
    label0=tmSet[i_tm]
    nsample=100
    erk_means=[]
    death_rates=[]
    for isample in range(nsample):
        erk_mean,death_rate=get_erk_death_info(i_tm,tl=0.,tf=tf,random_shuffle=True,norm_death=False)
        #axs[0].scatter(time_set,death_rate,s=30,c=colorSet[i_tm],alpha=.3,marker='o')
        #axs[1].scatter(time_set,erk_mean,s=30,c=colorSet[i_tm],alpha=.3,marker='o')
        erk_means.append(erk_mean)
        death_rates.append(death_rate)
    
    erk_means=np.array(erk_means)
    death_rates=np.array(death_rates)
    nrd=death_rates.shape[1]
    erk_means_CR=np.zeros((nrd,3))
    death_rates_CR=np.zeros((nrd,3))
    erk_means_CR[:,0]=np.nanmean(erk_means,axis=0)
    death_rates_CR[:,0]=np.nanmean(death_rates,axis=0)
    for ird in range(nrd):
        erk_means_CR[ird,1:]=bootstrap.get_CI(erk_means[:,ird],1000)
        death_rates_CR[ird,1:]=bootstrap.get_CI(death_rates[:,ird],1000)
    
    death_rate_m=death_rates_CR[:,0]
    erk_mean_m=erk_means_CR[:,0]
    time_set_smth=np.linspace(time_set[0],time_set[-1],201)
    spl_death=csaps.CubicSmoothingSpline(time_set[np.isfinite(death_rate_m)], death_rate_m[np.isfinite(death_rate_m)], smooth=.7)
    death_rate_smth=spl_death(time_set_smth)    
    spl_erk=csaps.CubicSmoothingSpline(time_set[np.isfinite(erk_mean_m)], erk_mean_m[np.isfinite(erk_mean_m)], smooth=.7)
    erk_mean_smth=spl_erk(time_set_smth)
    axs[0].fill_between(time_set,death_rates_CR[:,1],death_rates_CR[:,2],color=colorSet[i_tm],alpha=.25)
    axs[0].scatter(time_set,death_rate_m,s=30,c=colorSet[i_tm],alpha=.3,marker='o',label=label0)
    axs[1].scatter(time_set,erk_mean_m,s=30,c=colorSet[i_tm],alpha=.3,marker='o',label=label0)
    axs[1].fill_between(time_set,erk_means_CR[:,1],erk_means_CR[:,2],color=colorSet[i_tm],alpha=.25)
    #axs[0].plot(time_set_smth,death_rate_smth,'-',color=colorSet[i_tm],linewidth=1.5,label=label0)
    #axs[1].plot(time_set_smth,erk_mean_smth,'-',color=colorSet[i_tm],linewidth=1.5,label=label0)

axs[0].set_ylabel('death rate (fraction / hr)')
axs[0].legend(bbox_to_anchor=(1, 1.0))
axs[0].set_xlim(-.2,13.0)


axs[1].set_ylabel('ERK activity (c/n ratio)')
#axs[1].legend(bbox_to_anchor=(1, 1.0))
axs[1].set_xlabel('time (hrs)')
axs[1].set_xlim(-.2,13.0)
axs[1].set_ylim(.7,1.02)
plt.savefig(f'{date}/erk_means_death_rates.svg')

In [ ]:
def get_erk_death_number_info(i_tm,random_shuffle=False,tl=0,tf=20,norm_death=True):
    indmodels=np.where(inds_tmSet_models==i_tm)[0]
    indmodels=np.intersect1d(indmodels,indgood_models)
    inds_death=[]
    times_death=[]
    times_live=[]
    times_number=[]
    erk_activity=[]
    cell_rel_number=[]
    if random_shuffle:
        indmodels = np.random.choice(indmodels, size=indmodels.size, replace=True)
    for imodel in indmodels:
        sctm=modelSet[imodel]
        cellinds_death=sctm.inds_death.copy()
        if np.isscalar(cellinds_death):
            cellinds_death=np.array([cellinds_death])
        fl=initial_frames_model[imodel]
        fu=final_frames_model[imodel]
        timestep=timestep_model[imodel]
        if cellinds_death.size>0:
            times_death.append(timestep*(sctm.cells_frameSet[cellinds_death]-fl))
            inds_death.append(sctm.cells_frameSet[cellinds_death])
        times_live.append(timestep*(sctm.cells_frameSet-fl))
        erk_activity.append(sctm.x_signal)
        for iframe in range(fl,fu):
            times_number.append(timestep*(iframe-fl))
            cell_rel_number.append(np.sum(sctm.cells_frameSet==iframe)/np.sum(sctm.cells_frameSet==fl))
    if len(times_death)>0:
        times_death=np.concatenate(times_death)
        inds_death=np.concatenate(inds_death)
    times_live=np.concatenate(times_live)
    erk_activity=np.concatenate(erk_activity)
    #print(cell_rel_number)
    cell_rel_number=np.array(cell_rel_number)
    times_number=np.array(times_number)
    times_death=times_death[np.logical_and(times_death<tf,times_death>tl)]
    inds_live=np.where(np.logical_and(times_live<tf,times_live>=tl))[0]
    times_live=times_live[inds_live]
    erk_activity=erk_activity[inds_live]
    times_death=np.sort(times_death)
    time_set=np.arange(tl,tf,dt)
    times_death_fraction=np.zeros_like(time_set)
    erk_mean=np.zeros_like(time_set)
    fraction_on=np.zeros_like(time_set)
    number_mean=np.zeros_like(time_set)
    for itt in range(time_set.size):
        tu=time_set[itt]
        times_death_fraction[itt]=np.sum(times_death<=tu)
        indstw=np.where(np.logical_and(times_live>tu-dt,times_live<=tu))[0]
        indstw_number=np.where(np.logical_and(times_number>tu-dt,times_number<=tu))[0]
        #print(indstw_number)
        erk_mean[itt]=np.nanmean(erk_activity[indstw])
        fraction_on[itt]=np.sum(erk_activity[indstw]>midcut)/np.sum(np.isfinite(erk_activity[indstw]))
        number_mean[itt]=np.nanmean(cell_rel_number[indstw_number])
    times_death_fraction=times_death_fraction/times_death_fraction[-1]
    death_rate=np.gradient(times_death_fraction,time_set)
    if norm_death:
        death_rate=(death_rate-np.nanmin(death_rate))/(np.nanmax(death_rate)-np.nanmin(death_rate))
    return erk_mean,death_rate,number_mean

In [ ]:
midcut=1.0
tf=21.0
tl=0.
dt=1.0
time_set=np.arange(tl,tf,dt)
tms=[1]
fig,axs=plt.subplots(3,1,sharex=True,figsize=(8,7.5))

for i_tm in tms:
    label0=tmSet[i_tm]
    nsample=100
    erk_means=[]
    death_rates=[]
    number_means=[]
    for isample in range(nsample):
        erk_mean,death_rate,number_mean=get_erk_death_number_info(i_tm,tl=0.,tf=tf,random_shuffle=True,norm_death=False)
        #axs[0].scatter(time_set,death_rate,s=30,c=colorSet[i_tm],alpha=.3,marker='o')
        #axs[1].scatter(time_set,erk_mean,s=30,c=colorSet[i_tm],alpha=.3,marker='o')
        erk_means.append(erk_mean)
        death_rates.append(death_rate)
        number_means.append(number_mean)
    erk_means=np.array(erk_means)
    death_rates=np.array(death_rates)
    number_means=np.array(number_means)
    nrd=death_rates.shape[1]
    erk_means_CR=np.zeros((nrd,3))
    number_means_CR=np.zeros((nrd,3))
    death_rates_CR=np.zeros((nrd,3))
    erk_means_CR[:,0]=np.nanmean(erk_means,axis=0)
    death_rates_CR[:,0]=np.nanmean(death_rates,axis=0)
    number_means_CR[:,0]=np.nanmean(number_means,axis=0)
    for ird in range(nrd):
        erk_means_CR[ird,1:]=bootstrap.get_CI(erk_means[:,ird],1000)
        death_rates_CR[ird,1:]=bootstrap.get_CI(death_rates[:,ird],1000)
        number_means_CR[ird,1:]=bootstrap.get_CI(number_means[:,ird],1000)
    
    death_rate_m=death_rates_CR[:,0]
    erk_mean_m=erk_means_CR[:,0]
    number_means_CR[0,:]=1.
    time_set_smth=np.linspace(time_set[0],time_set[-1],201)
    spl_death=csaps.CubicSmoothingSpline(time_set[np.isfinite(death_rate_m)], death_rate_m[np.isfinite(death_rate_m)], smooth=.7)
    death_rate_smth=spl_death(time_set_smth)    
    spl_erk=csaps.CubicSmoothingSpline(time_set[np.isfinite(erk_mean_m)], erk_mean_m[np.isfinite(erk_mean_m)], smooth=.1)
    erk_mean_smth=spl_erk(time_set_smth)
    spl_number=csaps.CubicSmoothingSpline(time_set[np.isfinite(number_means_CR[:,0])], number_means_CR[np.isfinite(erk_mean_m),0], smooth=.1)
    number_mean_smth=spl_number(time_set_smth)
    axs[0].scatter(time_set,number_means_CR[:,0],s=30,c=colorSet[i_tm],alpha=.3,marker='o',label=label0)
    axs[0].fill_between(time_set,number_means_CR[:,1],number_means_CR[:,2],color=colorSet[i_tm],alpha=.25)
    axs[1].fill_between(time_set,death_rates_CR[:,1],death_rates_CR[:,2],color=colorSet[i_tm],alpha=.25)
    axs[1].scatter(time_set,death_rate_m,s=30,c=colorSet[i_tm],alpha=.3,marker='o',label=label0)
    axs[2].scatter(time_set,erk_mean_m,s=30,c=colorSet[i_tm],alpha=.3,marker='o',label=label0)
    axs[2].fill_between(time_set,erk_means_CR[:,1],erk_means_CR[:,2],color=colorSet[i_tm],alpha=.25)
    axs[1].plot(time_set_smth,death_rate_smth,'-',color=colorSet[i_tm],linewidth=1.5,label=label0)
    axs[2].plot(time_set_smth,erk_mean_smth,'-',color=colorSet[i_tm],linewidth=1.5,label=label0)
    axs[0].plot(time_set_smth,number_mean_smth,'-',color=colorSet[i_tm],linewidth=1.5,label=label0)

axs[0].set_ylabel('relative cell number')
#axs[0].legend(bbox_to_anchor=(1, 1.0))
axs[0].set_xlim(-.2,tf-dt)
axs[0].set_ylim(.2,1.02)

axs[1].set_ylabel('death rate (fraction / hr)')
#axs[1].legend(bbox_to_anchor=(1, 1.0))
axs[1].set_xlim(-.2,tf-dt)


axs[2].set_ylabel('ERK activity (c/n ratio)')
#axs[1].legend(bbox_to_anchor=(1, 1.0))
axs[2].set_xlabel('time (hrs)')
axs[2].set_xlim(-.2,tf-dt)
axs[2].set_ylim(.76,1.02)
plt.savefig(f'{date}/OS17_number_erkmeans_deathrates.svg')

### Drug response state analysis Markov modeling

In [ ]:
clusters_x=clusters_minima.clustercenters.copy()
n_feat=clusters_x.shape[1];n_clusters=clusters_x.shape[0]
clustercenters0=np.concatenate((clusters_x,np.zeros(n_clusters)[:,np.newaxis]),axis=1)
clustercenters1=np.concatenate((clusters_x,np.ones(n_clusters)[:,np.newaxis]),axis=1)
clustercenters_death=np.append(clustercenters0,clustercenters1,axis=0)
n_clusters_death=clustercenters_death.shape[0]
#clusters_death=coor.clustering.AssignCenters(clustercenters_death, metric='euclidean')
stateSet_death=np.append(stateSet,stateSet+n_states)

Slow=lowcut
Shigh=highcut
clustercenters0=np.concatenate((clustercenters_death,Slow*np.ones(n_clusters_death)[:,np.newaxis]),axis=1)
clustercenters1=np.concatenate((clustercenters_death,Shigh*np.ones(n_clusters_death)[:,np.newaxis]),axis=1)
clustercenters=np.append(clustercenters0,clustercenters1,axis=0)
n_clusters_death=clustercenters.shape[0]
clusters_death=coor.clustering.AssignCenters(clustercenters, metric='euclidean')
stateSet_death=np.append(stateSet_death,stateSet_death+2*n_states)

In [ ]:
#inds_trajsteps_models=np.append(inds_trajsteps_models,np.ones(x0.shape[0]).astype(int)*i)
#inds_trajsteps=np.append(inds_trajsteps,modelSet[i].inds_trajp1,axis=0)
i_tmr=1
indtm_models=np.where(inds_tmSet_models==i_tmr)[0]
x0set_death=[]
x1set_death=[]
tl=0
tu=20
for imodel in indtm_models:
    sctm=modelSet[imodel]
    indmodel=np.where(inds_trajsteps_models==imodel)[0]
    #inds_trajsteps_model1=inds_trajsteps[indmodel,1]
    #inds_death_trajsteps_model=np.where(np.isin(inds_trajsteps_model1,inds_death_traj_set[i_tmr]))[0]
    #inds_death_trajsteps_model=np.where(np.isin(indmodel,inds_death_traj_set[i_tmr]))[0]
    inds1=inds_trajsteps[indmodel,1]
    times1=sctm.cells_timeSet[inds1]
    indmodel=indmodel[np.logical_and(times1<tu,times1>tl)]
    inds1=inds1[np.logical_and(times1<tu,times1>tl)]
    x0=x0set[indmodel,:]
    x1=x1set[indmodel,:]
    x0=np.append(x0,np.zeros(x0.shape[0])[:,np.newaxis],axis=1)
    x1=np.append(x1,np.zeros(x1.shape[0])[:,np.newaxis],axis=1)
    inds_death_trajsteps_model=np.where(np.isin(inds1,sctm.inds_death))[0]
    x1[inds_death_trajsteps_model,-1]=1.
    erk_model0=sctm.x_signal[inds_trajsteps[indmodel,0]]
    erk_model1=sctm.x_signal[inds_trajsteps[indmodel,1]]
    x0=np.append(x0,erk_model0[:,np.newaxis],axis=1)
    x1=np.append(x1,erk_model1[:,np.newaxis],axis=1)
    indgood0=np.isfinite(np.sum(x0,axis=1))
    indgood1=np.isfinite(np.sum(x1,axis=1))
    indgood=np.logical_and(indgood0,indgood1)
    x0set_death.append(x0[indgood,:])
    x1set_death.append(x1[indgood,:])

x0set_death=np.concatenate(x0set_death)
x1set_death=np.concatenate(x1set_death)
    

In [ ]:
tmSet

In [ ]:
np.sum(x1set_death[:,-2]==1.)

In [ ]:
SR_clusters=np.array([0.01432603, 0.83945825, 3.31683432, 6.02338035])

In [ ]:
##do with S_r_death states
clusters_x=clusters_minima.clustercenters.copy()
n_feat=clusters_x.shape[1];n_clusters=clusters_x.shape[0]
clustercenters0=np.concatenate((clusters_x,np.zeros(n_clusters)[:,np.newaxis]),axis=1)
clustercenters1=np.concatenate((clusters_x,np.ones(n_clusters)[:,np.newaxis]),axis=1)
clustercenters_death=np.append(clustercenters0,clustercenters1,axis=0)
n_clusters_death=clustercenters_death.shape[0]
#clusters_death=coor.clustering.AssignCenters(clustercenters_death, metric='euclidean')
stateSet_death=np.append(stateSet,stateSet+n_states)
n_states_death=2*n_states

SDlow=SR_clusters[1]
#SDmid=1.
SDhigh=SR_clusters[3]
clustercenters0=np.concatenate((clustercenters_death,SDlow*np.ones(n_clusters_death)[:,np.newaxis]),axis=1)
#clustercenters1=np.concatenate((clustercenters_death,SDmid*np.ones(n_clusters_death)[:,np.newaxis]),axis=1)
clustercenters2=np.concatenate((clustercenters_death,SDhigh*np.ones(n_clusters_death)[:,np.newaxis]),axis=1)
#clustercenters_death=np.concatenate((clustercenters0,clustercenters1,clustercenters2),axis=0)
clustercenters_death=np.concatenate((clustercenters0,clustercenters2),axis=0)
n_clusters_death=clustercenters_death.shape[0]
clusters_death=coor.clustering.AssignCenters(clustercenters_death, metric='euclidean')
#stateSet_death1=np.append(stateSet_death,stateSet_death+2*n_states)
stateSet_death=np.append(stateSet_death,stateSet_death+n_states_death)
n_states_death=2*n_states_death

Slow=lowcut #.37 #lowcut
Shigh=highcut #1.45 #highcut
clustercenters0=np.concatenate((clustercenters_death,Slow*np.ones(n_clusters_death)[:,np.newaxis]),axis=1)
clustercenters1=np.concatenate((clustercenters_death,Shigh*np.ones(n_clusters_death)[:,np.newaxis]),axis=1)
clustercenters_death=np.append(clustercenters0,clustercenters1,axis=0)
n_clusters_death=clustercenters_death.shape[0]
clusters_death=coor.clustering.AssignCenters(clustercenters_death, metric='euclidean')
stateSet_death=np.append(stateSet_death,stateSet_death+n_states_death)
n_states_death=2*n_states_death

inds_death_feats=np.arange(clusters_minima.clustercenters.shape[0]).astype(int)
inds_death_life=clusters_minima.clustercenters.shape[1]
inds_death_SD=clusters_minima.clustercenters.shape[1]+1
inds_death_ERK=clusters_minima.clustercenters.shape[1]+2

states_death_ERKlow=np.unique(stateSet_death[np.where(clusters_death.clustercenters[:,inds_death_ERK]==Slow)[0]])
states_death_ERKhigh=np.unique(stateSet_death[np.where(clusters_death.clustercenters[:,inds_death_ERK]==Shigh)[0]])
states_death_SDlow=np.unique(stateSet_death[np.where(clusters_death.clustercenters[:,inds_death_SD]==SDlow)[0]])
states_death_SDhigh=np.unique(stateSet_death[np.where(clusters_death.clustercenters[:,inds_death_SD]==SDhigh)[0]])
states_death_alive=np.unique(stateSet_death[np.where(clusters_death.clustercenters[:,inds_death_life]==0.)[0]])
states_death_dead=np.unique(stateSet_death[np.where(clusters_death.clustercenters[:,inds_death_life]==1.)[0]])

In [ ]:
indmodels=np.where(inds_tmSet_models==i_tmr)[0]
indmodels=np.intersect1d(indmodels,indgood_models)
x0set_death=[]
x1set_death=[]
tl=0
tu=25.5
fw_erk=1
min_nerk=0
for imodel in indmodels:
    sctm=modelSet[imodel]
    indmodel=np.where(inds_trajsteps_models==imodel)[0]
    #inds_trajsteps_model1=inds_trajsteps[indmodel,1]
    #inds_death_trajsteps_model=np.where(np.isin(inds_trajsteps_model1,inds_death_traj_set[i_tmr]))[0]
    #inds_death_trajsteps_model=np.where(np.isin(indmodel,inds_death_traj_set[i_tmr]))[0]
    inds1=inds_trajsteps[indmodel,1]
    times1=sctm.cells_timeSet[inds1]
    indmodel=indmodel[np.logical_and(times1<tu,times1>tl)]
    inds1=inds1[np.logical_and(times1<tu,times1>tl)]
    x0=x0set[indmodel,:]
    x1=x1set[indmodel,:]
    x0=np.append(x0,np.zeros(x0.shape[0])[:,np.newaxis],axis=1)
    x1=np.append(x1,np.zeros(x1.shape[0])[:,np.newaxis],axis=1)
    inds_death_trajsteps_model=np.where(np.isin(inds1,sctm.inds_death))[0]
    x1[inds_death_trajsteps_model,-1]=1.
    SD0=sctm.S_r_death[inds_trajsteps[indmodel,0]]
    #SD0[SD0>5.5]=np.nan
    SD1=sctm.S_r_death[inds_trajsteps[indmodel,1]]
    #SD1[SD1>5.5]=np.nan
    x0=np.append(x0,SD0[:,np.newaxis],axis=1)
    x1=np.append(x1,SD1[:,np.newaxis],axis=1)
    erk_model0=np.ones(inds1.size)*np.nan
    erk_model1=np.ones(inds1.size)*np.nan
    for i in range(inds1.size):
        ic0=inds_trajsteps[indmodel[i],0]
        ic1=inds_trajsteps[indmodel[i],1]
        #print(sctm.get_cell_trajectory(ic0).size)
        erktraj0=sctm.x_signal[sctm.get_cell_trajectory(ic0)][-fw_erk:]
        good0=np.isfinite(erktraj0)
        if np.sum(good0)>min_nerk:
            #erk_model0[i]=np.nanmean(erktraj0)
            erk_model0[i]=erktraj0[np.where(np.isfinite(erktraj0))[0][-1]]
        erktraj1=sctm.x_signal[sctm.get_cell_trajectory(ic1)][-fw_erk:]
        good1=np.isfinite(erktraj1)
        if np.sum(good1)>min_nerk:
            erk_model1[i]=erktraj1[np.where(np.isfinite(erktraj1))[0][-1]]
            #erk_model1[i]=np.nanmean(erktraj1)
        if x1[i,inds_death_life]==1.:
            print(f'm{imodel} {erk_model0[i]} to {erk_model1[i]}')
            #print(f'fra-1: {sctm.x_signal_fra1[ic1]}')
    #erk_model0=sctm.x_signal[inds_trajsteps[indmodel,0]]
    #erk_model1=sctm.x_signal[inds_trajsteps[indmodel,1]]
    x0=np.append(x0,erk_model0[:,np.newaxis],axis=1)
    x1=np.append(x1,erk_model1[:,np.newaxis],axis=1)
    indgood0=np.isfinite(np.sum(x0,axis=1))
    indgood1=np.isfinite(np.sum(x1,axis=1))
    indgood=np.logical_and(indgood0,indgood1)
    x0set_death.append(x0[indgood,:])
    x1set_death.append(x1[indgood,:])

x0set_death=np.concatenate(x0set_death)
x1set_death=np.concatenate(x1set_death)

In [ ]:
pdeath_ERKhigh=np.sum(np.logical_and(x1set_death[:,inds_death_life]==1,x1set_death[:,inds_death_ERK]>midcut))/np.sum(x1set_death[:,inds_death_ERK]>midcut)
pdeath_ERKlow=np.sum(np.logical_and(x1set_death[:,inds_death_life]==1,x1set_death[:,inds_death_ERK]<midcut))/np.sum(x1set_death[:,inds_death_ERK]<midcut)
print(f'death rate ERK-low: {pdeath_ERKlow:.2e}')
print(f'death rate ERK-high: {pdeath_ERKhigh:.2e}')

##### state markov model construction

In [ ]:
P_states=model.get_transition_matrix_CG(x0set_death,x1set_death,clusters=clusters_death,states=stateSet_death,return_counts=True)
plt.imshow(np.log(P_states),origin='lower')

In [ ]:
def clean_P(P_states,pdeath_ERKhigh=0.,pdeath_ERKlow=0.,death_boost=1.,states_death_dead=None,states_death_alive=None,states_death_ERKlow=None,states_death_ERKhigh=None,states_death_SDhigh=None,states_death_SDlow=None):
    #zero out all death counts
    inds_from=states_death_alive
    inds_to=states_death_dead
    for i in inds_from:
        for j in inds_to:
            P_states[i,j]=0.
    
    for i in inds_to:
        P_states[i,:]=0.
        P_states[i,i]=1.
    
    #move SD-high to SD-low transitions to SD-high to SD-high
    inds_from=np.intersect1d(states_death_SDhigh,states_death_alive)
    inds_to_old=np.intersect1d(states_death_SDlow,states_death_alive)
    inds_to_new=np.intersect1d(states_death_SDhigh,states_death_alive)
    for i in inds_from:
        for j in range(inds_to_old.size):
            j_old=inds_to_old[j]
            j_new=inds_to_new[j]
            P_states[i,j_new]=P_states[i,j_new]+P_states[i,j_old]
            P_states[i,j_old]=0.
            
    #re-normalize
    for irow in range(P_states.shape[0]):
        if np.sum(P_states[irow,:])==0.:
            P_states[irow,irow]=1.
        else:
            P_states[irow,:]=P_states[irow,:]/np.sum(P_states[irow,:])
            
    #add in death rates
    inds_from=np.intersect1d(states_death_ERKhigh,states_death_alive)
    inds_to=np.intersect1d(states_death_ERKhigh,states_death_dead)
    for i_state in range(inds_from.size):
        P_states[inds_from[i_state],inds_to[i_state]]=pdeath_ERKhigh*death_boost
    
    inds_from=np.intersect1d(states_death_ERKlow,states_death_alive)
    inds_to=np.intersect1d(states_death_ERKlow,states_death_dead)
    for i_state in range(inds_from.size):
        P_states[inds_from[i_state],inds_to[i_state]]=pdeath_ERKlow*death_boost
    
    #re-normalize
    for irow in range(P_states.shape[0]):
        if np.sum(P_states[irow,:])==0.:
            P_states[irow,irow]=1.
        else:
            P_states[irow,:]=P_states[irow,:]/np.sum(P_states[irow,:])
    
    return P_states

In [ ]:
death_boost=3.15 #for reg mcl1i
#death_boost=2. #for FGFRi combo
#death_boost=2.25 #for pazo combo
#P_states=model.get_transition_matrix_CG(x0set_death,x1set_death,clusters=clusters_death,states=stateSet_death,return_counts=True)
P_states=clean_P(P_states,pdeath_ERKhigh=pdeath_ERKhigh,pdeath_ERKlow=pdeath_ERKlow,death_boost=death_boost,states_death_dead=states_death_dead,states_death_alive=states_death_alive,states_death_ERKlow=states_death_ERKlow,states_death_ERKhigh=states_death_ERKhigh,states_death_SDhigh=states_death_SDhigh,states_death_SDlow=states_death_SDlow)

x_initial=x[inds_initial_traj,:]
x_initial=np.append(x_initial,np.zeros_like(cratio_erk_traj[inds_initial_traj,np.newaxis]),axis=1)
x_initial=np.append(x_initial,Srd_traj[inds_initial_traj,np.newaxis],axis=1)
x_initial=np.append(x_initial,cratio_erk_traj[inds_initial_traj,np.newaxis],axis=1)
indgood=np.where(np.isnan(np.sum(x_initial,axis=1))==0)[0]
x_initial=x_initial[indgood,:]


ind_states_death=stateSet_death[clusters_death.assign(x_initial)]
states, counts = np.unique(ind_states_death,return_counts=True)
prob_initial=np.zeros(P_states.shape[0])
for istate in states:
    indstate=np.where(states==istate)[0]
    prob_initial[istate]=np.sum(counts[indstate])

prob_initial=prob_initial/np.sum(prob_initial)

Nt=48
times=np.arange(Nt)*.5
prob0=prob_initial.copy()
prob=prob0.copy()
mfpt=np.zeros(n_states)
prob_states=np.zeros((n_states,Nt))
prob_states[:,0]=prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDlow),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDlow),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDhigh),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDhigh),states_death_alive)]  
for itt in range(1,Nt):
    prob=np.matmul(prob,P_states)
    prob_states[:,itt]=prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDlow),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDlow),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDhigh),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDhigh),states_death_alive)]  

nplots=int(np.ceil(np.sqrt(n_states+1)))
fig,axs=plt.subplots(nplots,nplots,sharex=True,sharey=True,figsize=(12,12))
axs=axs.flatten()

axs[0].plot(times,np.sum(prob_states,axis=0),label='model')
#plt.plot(times,np.sum(prob_states_erki,axis=0))
axs[0].plot(time_inds-time_inds[0],np.sum(state_probs_times[2,...],axis=0),label='observed')
axs[0].set_ylabel('population')
axs[0].set_xlabel('time (hrs)')
axs[0].legend(loc='upper right')

for istate in range(n_states):
    ax=axs[istate+1]
    ax.plot(times,prob_states[istate,:])
    ax.plot(time_inds-time_inds[0],state_probs_times[2,istate,:]) 
    #ax.plot(times,prob_states_erki[istate,:])
    ax.set_ylim(0,.8)
    ax.set_ylabel(f'state {state_labels[istate]} population')
    ax.set_xlabel('time (hrs)')

for iplot in range(n_states+1,nplots):
    ax=axs[iplot]
    ax.axis('off')

In [ ]:
P_states=model.get_transition_matrix_CG(x0set_death,x1set_death,clusters=clusters_death,states=stateSet_death,return_counts=True)
P_states=clean_P(P_states,pdeath_ERKhigh=pdeath_ERKhigh,pdeath_ERKlow=pdeath_ERKlow,death_boost=death_boost,states_death_dead=states_death_dead,states_death_alive=states_death_alive,states_death_ERKlow=states_death_ERKlow,states_death_ERKhigh=states_death_ERKhigh,states_death_SDhigh=states_death_SDhigh,states_death_SDlow=states_death_SDlow)
plt.imshow(np.log(P_states),origin='lower')

plt.savefig(f'{date}/transition_matrix_states.svg')

In [ ]:
fig,axs=plt.subplots(2,2,sharey=True,figsize=(12,12))
axs=axs.flatten()
indsfrom=np.intersect1d(states_death_ERKlow,states_death_SDlow)
indsfrom=np.intersect1d(indsfrom,states_death_alive)
indsto=np.intersect1d(states_death_dead,states_death_dead)
indsto=np.intersect1d(indsto,states_death_dead)
ptest=np.zeros(indsfrom.size)
itest=0
for i in indsfrom:
    for j in indsto:
        ptest[itest]=ptest[itest]+P_states[i,j]
    itest=itest+1

axs[0].bar(np.arange(n_states),ptest,color='black')
axs[0].set_xticks(np.arange(n_states))
axs[0].set_xticklabels(state_labels)
axs[0].set_title('ERK-low and SD-low')

indsfrom=np.intersect1d(states_death_ERKhigh,states_death_SDlow)
indsfrom=np.intersect1d(indsfrom,states_death_alive)
indsto=np.intersect1d(states_death_dead,states_death_dead)
indsto=np.intersect1d(indsto,states_death_dead)
ptest=np.zeros(indsfrom.size)
itest=0
for i in indsfrom:
    for j in indsto:
        ptest[itest]=ptest[itest]+P_states[i,j]
    itest=itest+1

axs[1].bar(np.arange(n_states),ptest,color='black')
axs[1].set_xticks(np.arange(n_states))
axs[1].set_xticklabels(state_labels)
axs[1].set_title('ERK-high and SD-low')

indsfrom=np.intersect1d(states_death_ERKlow,states_death_SDhigh)
indsfrom=np.intersect1d(indsfrom,states_death_alive)
indsto=np.intersect1d(states_death_dead,states_death_dead)
indsto=np.intersect1d(indsto,states_death_dead)
ptest=np.zeros(indsfrom.size)
itest=0
for i in indsfrom:
    for j in indsto:
        ptest[itest]=ptest[itest]+P_states[i,j]
    itest=itest+1

axs[2].bar(np.arange(n_states),ptest,color='black')
axs[2].set_xticks(np.arange(n_states))
axs[2].set_xticklabels(state_labels)
axs[2].set_title('ERK-low and SD-high')

indsfrom=np.intersect1d(states_death_ERKhigh,states_death_SDhigh)
indsfrom=np.intersect1d(indsfrom,states_death_alive)
indsto=np.intersect1d(states_death_dead,states_death_dead)
indsto=np.intersect1d(indsto,states_death_dead)
ptest=np.zeros(indsfrom.size)
itest=0
for i in indsfrom:
    for j in indsto:
        ptest[itest]=ptest[itest]+P_states[i,j]
    itest=itest+1

axs[3].bar(np.arange(n_states),ptest,color='black')
axs[3].set_xticks(np.arange(n_states))
axs[3].set_xticklabels(state_labels)
axs[3].set_title('ERK-high and SD-high')

In [ ]:
fig,axs=plt.subplots(2,2,sharey=True,figsize=(12,12))
axs=axs.flatten()
indsfrom=np.intersect1d(states_death_ERKlow,states_death_SDlow)
indsfrom=np.intersect1d(indsfrom,states_death_alive)
indsto=np.intersect1d(states_death_ERKhigh,states_death_alive)
indsto=np.intersect1d(indsto,states_death_alive)
ptest=np.zeros(indsfrom.size)
itest=0
for i in indsfrom:
    for j in indsto:
        ptest[itest]=ptest[itest]+P_states[i,j]
    itest=itest+1

axs[0].bar(np.arange(n_states),ptest,color='black')
axs[0].set_xticks(np.arange(n_states))
axs[0].set_xticklabels(state_labels)
axs[0].set_title('ERK-on rates and SD-low')

indsfrom=np.intersect1d(states_death_ERKhigh,states_death_SDlow)
indsfrom=np.intersect1d(indsfrom,states_death_alive)
indsto=np.intersect1d(states_death_ERKlow,states_death_alive)
indsto=np.intersect1d(indsto,states_death_alive)
ptest=np.zeros(indsfrom.size)
itest=0
for i in indsfrom:
    for j in indsto:
        ptest[itest]=ptest[itest]+P_states[i,j]
    itest=itest+1

axs[1].bar(np.arange(n_states),ptest,color='black')
axs[1].set_xticks(np.arange(n_states))
axs[1].set_xticklabels(state_labels)
axs[1].set_title('ERK-off rates and SD-low')

indsfrom=np.intersect1d(states_death_ERKlow,states_death_SDhigh)
indsfrom=np.intersect1d(indsfrom,states_death_alive)
indsto=np.intersect1d(states_death_ERKhigh,states_death_alive)
indsto=np.intersect1d(indsto,states_death_alive)
ptest=np.zeros(indsfrom.size)
itest=0
for i in indsfrom:
    for j in indsto:
        ptest[itest]=ptest[itest]+P_states[i,j]
    itest=itest+1

axs[2].bar(np.arange(n_states),ptest,color='black')
axs[2].set_xticks(np.arange(n_states))
axs[2].set_xticklabels(state_labels)
axs[2].set_title('ERK-on rates and SD-high')

indsfrom=np.intersect1d(states_death_ERKhigh,states_death_SDhigh)
indsfrom=np.intersect1d(indsfrom,states_death_alive)
indsto=np.intersect1d(states_death_ERKlow,states_death_alive)
indsto=np.intersect1d(indsto,states_death_alive)
ptest=np.zeros(indsfrom.size)
itest=0
for i in indsfrom:
    for j in indsto:
        ptest[itest]=ptest[itest]+P_states[i,j]
    itest=itest+1

axs[3].bar(np.arange(n_states),ptest,color='black')
axs[3].set_xticks(np.arange(n_states))
axs[3].set_xticklabels(state_labels)
axs[3].set_title('ERK-off rates and SD-high')

In [ ]:
def get_state_fluxplot(P_states,state_probs,state_centers,state_labels=None,ax=None,state_colors=None,fcut=.001,pcut=.02,lws=40.,rad=.3,sh=2,mscale=40.,point_mscale=1e7,arrowstyle='->,head_width=1.,head_length=1.'):
    n_states=state_probs.size
    if ax is None:
        fig,ax=plt.subplots(1,1)
    if state_labels is None:
        state_labels=[""]*n_states #np.array(list(string.ascii_uppercase))[0:nstates]
    if state_colors is None:
        state_colors=[]
        for istate in range(state_probs.size):
            state_colors.append(plt.cm.jet(istate/state_probs.size))
    for j in range(n_states):
        for k in range(n_states):
            fluxjk=state_probs[j]*P_states[j,k]
            fluxkj=state_probs[k]*P_states[k,j]
            flux=fluxjk-fluxkj
            #flux=fluxjk
            if j<k and flux>fcut and state_probs[j]>pcut: #Mt[j,k]>pcut:
                arrow = matplotlib.patches.FancyArrowPatch((state_centers[j,0], state_centers[j,1]), (state_centers[k,0], state_centers[k,1]),shrinkA=sh,shrinkB=sh,mutation_scale=mscale*flux,color='black',arrowstyle=arrowstyle,connectionstyle='arc3,rad='+str(rad),lw=lws*flux,joinstyle='miter',capstyle='butt',alpha=.5) #note j and k flipped because drawing arrow at tail
                ax.add_patch(arrow)
            if j>k and flux>fcut and state_probs[j]>pcut: #Mt[j,k]>pcut:
                arrow = matplotlib.patches.FancyArrowPatch((state_centers[j,0], state_centers[j,1]), (state_centers[k,0], state_centers[k,1]),shrinkA=sh,shrinkB=sh,mutation_scale=mscale*flux,color='black',arrowstyle=arrowstyle,connectionstyle='arc3,rad='+str(rad),lw=lws*flux,joinstyle='miter',capstyle='butt',alpha=.5)
                ax.add_patch(arrow)
        ax.text(state_centers[j,0],state_centers[j,1],state_labels[j],color='black',ma='center')
        ax.scatter(state_centers[j,0],state_centers[j,1],s=point_mscale*state_probs[j],color=state_colors[j],alpha=.6)
    ax.set_xlim(np.min(state_centers[:,0])-.3,np.max(state_centers[:,0])+.3)
    ax.set_ylim(np.min(state_centers[:,1])-.3,np.max(state_centers[:,1])+.3)
    ax.axis('off')

In [ ]:
state_labels_all=[state_label + ' live' for state_label in state_labels.tolist()]+[state_label + ' dead' for state_label in state_labels.tolist()]
state_labels_all=[state_label + ' SD-' for state_label in state_labels_all]+[state_label + ' SD+' for state_label in state_labels_all]
state_labels_all=[state_label + ' ERK-' for state_label in state_labels_all]+[state_label + ' ERK+' for state_label in state_labels_all]
print(state_labels_all)

In [ ]:
state_colors_all=[]
for istate in range(n_states):
    state_colors_all.append(plt.cm.jet(istate/n_states))
state_colors_all=state_colors_all*8
print(state_colors_all)

In [ ]:
import numpy as np

# Base states (can be a NumPy array; .tolist() matches your original code)
state_labels = np.array(["A","B","C"])  # <-- replace with your array if needed

# --- Build labels in the SAME ORDER as your code ---
L0 = [s + " L+" for s in state_labels.tolist()] + [s + " L-" for s in state_labels.tolist()]
L1 = [lab + " SD-" for lab in L0] + [lab + " SD+" for lab in L0]
state_labels_all = [lab + " ERK-" for lab in L1] + [lab + " ERK+" for lab in L1]

# --- Geometry parameters ---
x_base = {s:i for i, s in enumerate(state_labels.tolist())}  # A->0, B->1, C->2
# Offsets
dy_sd   = .2   # vertical separation for SD +/- 
dx_erk  = 0.25  # small x shift for ERK +/- 
dy_dead = 0.25   # vertical shift for dead (above/below live)

# --- Compute coordinates in the SAME ORDER as state_labels_all ---
coords = []
for lab in state_labels_all:
    s, live_dead, sd_tag, erk_tag = lab.split()  # e.g., "A L+ SD- ERK-"
    xtemp = x_base[s] + (-dx_erk if erk_tag == "ERK-" else +dx_erk)
    ytemp = (+dy_sd if sd_tag == "SD+" else -dy_sd)
    if live_dead == "L-":
        ytemp += (+dy_dead if sd_tag == "SD+" else -dy_dead)
    coords.append([xtemp, ytemp])

state_centers_all = np.array(coords)

# Quick sanity checks
print(len(state_labels_all), state_centers_all.shape)  # 24, (24,2)
# print(state_labels_all)  # uncomment to inspect order


In [ ]:
x_base = {s:i for i, s in enumerate(state_labels.tolist())}
dy_sd, dx_erk, dy_dead
x_min = min(x_base.values()) - dx_erk
x_max = max(x_base.values()) + dx_erk
y_top =  dy_sd + dy_dead
y_bot = -dy_sd - dy_dead

# Left margins for row labels
x_margin_left_ld = 0.15   # live/dead row labels (closer to nodes)
x_margin_left_sd = 0.5   # SD band labels (a bit further left)
y_margin=.4

x_left_ld = x_min - x_margin_left_ld
x_left_sd = x_min - x_margin_left_sd

# ---------------------------
# Row labels: live/dead moved to the LEFT
# ---------------------------
row_ld_labels = [
    "live SD-", "dead SD-",
    "live SD+", "dead SD+",
]
row_ld_xy = [
    [x_left_ld, -dy_sd],                 # live on SD-
    [x_left_ld, -dy_sd - dy_dead],       # dead below live on SD-
    [x_left_ld, +dy_sd],                 # live on SD+
    [x_left_ld, +dy_sd + dy_dead],       # dead above live on SD+
]
row_ld_pos = dict(zip(row_ld_labels, row_ld_xy))

# ---------------------------
# SD band labels: centered in y between live/dead for each band, further left
# ---------------------------
row_sd_labels = ["SD-", "SD+"]
row_sd_xy = [
    [x_left_sd, -dy_sd - dy_dead/2.0],   # centered between live/dead on SD-
    [x_left_sd, +dy_sd + dy_dead/2.0],   # centered between live/dead on SD+
]
row_sd_pos = dict(zip(row_sd_labels, row_sd_xy))

# ---------------------------
# Column labels (same as before)
# ---------------------------
col_labels_major = state_labels.tolist()
col_label_major_xy = [[x_base[s], y_top + y_margin] for s in col_labels_major]
col_labels_major_pos = dict(zip(col_labels_major, col_label_major_xy))

col_labels_minor = []
col_label_minor_xy = []
for s in state_labels.tolist():
    col_labels_minor += [f"{s} ERK-", f"{s} ERK+"]
    col_label_minor_xy += [
        [x_base[s] - dx_erk, y_top + y_margin/2],   # ERK-
        [x_base[s] + dx_erk, y_top + y_margin/2],   # ERK+
    ]
col_labels_minor_pos = dict(zip(col_labels_minor, col_label_minor_xy))

# ---------------------------
# Helper to draw everything (matplotlib)
# ---------------------------
def draw_headers(ax):
    # SD band labels (further left, centered vertically between live/dead)
    #for lab, (x,y) in row_sd_pos.items():
    #    ax.text(x, y, lab, ha='right', va='center', fontsize=11, fontweight='bold')

    # Live/dead row labels (left)
    for lab, (xt,yt) in row_ld_pos.items():
        ax.text(xt, yt, lab.replace("(SD-", "").replace("(SD+)", "").strip(),
                ha='right', va='center', fontsize=9)

    # Major column headers (A/B/C)
    for lab, (xt,yt) in col_labels_major_pos.items():
        ax.text(xt, yt, lab, ha='center', va='bottom', fontsize=12, fontweight='bold')

    # Minor column headers (ERK ± under each major)
    for lab, (xt,yt) in col_labels_minor_pos.items():
        ax.text(xt, yt, "ERK−" if lab.endswith("ERK-") else "ERK+",
                ha='center', va='bottom', fontsize=10)

# Example usage after plotting your nodes:
fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(state_centers_all[:,0], state_centers_all[:,1])
for (xt,yt), lab in zip(state_centers_all, state_labels_all):
 ax.text(xt, yt, lab, fontsize=7, ha='center', va='bottom')
draw_headers(ax)
ax.set_xlim(x_min - 1.2, x_max + 0.5)
ax.set_ylim(y_bot - 0.5, y_top + 0.9)
ax.axhline(0, ls=':', lw=0.6)
plt.tight_layout(); plt.show()


In [ ]:
fig=plt.figure(figsize=(12,4))
ax=plt.gca()
get_state_fluxplot(P_states,prob,state_centers_all,ax=ax,state_labels=None,state_colors=state_colors_all,lws=2000,mscale=2000,sh=0.5,fcut=.0001,point_mscale=3000)

In [ ]:
def get_state_barplot(prob,ax=None,state_labels=None,color='black'):
    nstates=prob.size
    if ax is None:
        fig,ax=plt.subplots(1,1)
    if state_labels is None:
        state_labels=np.array(list(string.ascii_uppercase))[0:nstates]
    for istate in range(prob.size):
        ax.bar(istate,prob[istate],color=color)
    ax.set_xticks(np.arange(nstates))
    ax.set_xticklabels(state_labels)
    #ax.set_ylabel('population fraction')
    #ax.set_xlabel('state')

In [ ]:
state_labels_SD=[state_label + ' SD-' for state_label in state_labels.tolist()]+[state_label + ' SD+' for state_label in state_labels.tolist()]

In [ ]:
state_labels_SD

In [ ]:
x_initial=x[inds_initial_traj,:]
x_initial=np.append(x_initial,np.zeros_like(cratio_erk_traj[inds_initial_traj,np.newaxis]),axis=1)
x_initial=np.append(x_initial,Srd_traj[inds_initial_traj,np.newaxis],axis=1)
x_initial=np.append(x_initial,cratio_erk_traj[inds_initial_traj,np.newaxis],axis=1)
indgood=np.where(np.isnan(np.sum(x_initial,axis=1))==0)[0]
x_initial=x_initial[indgood,:]


ind_states_death=stateSet_death[clusters_death.assign(x_initial)]
states, counts = np.unique(ind_states_death,return_counts=True)
prob_initial=np.zeros(P_states.shape[0])
for istate in states:
    indstate=np.where(states==istate)[0]
    prob_initial[istate]=np.sum(counts[indstate])

prob_initial=prob_initial/np.sum(prob_initial)

phigh=.8
prob=prob_initial.copy()
itt=0
fig,axs=plt.subplots(2,2,sharex=True,sharey=True,figsize=(10,5))
axs=axs.flatten()
#axs[0].scatter(state_centers_minima[:,0],state_centers_minima[:,1],s=30,c=prob[0:n_states],cmap=plt.cm.binary)#,clim=(0,1))
inds=np.intersect1d(states_death_ERKlow,states_death_alive)
get_state_barplot(prob[inds],state_labels=state_labels_SD,ax=axs[0])
axs[0].set_ylim(0,phigh)
axs[0].set_title(f'{.5*itt:.0f} hrs: ERK low fraction alive {np.sum(prob[inds]):.2f}')
#axs[0].axis('off')
#axs[1].scatter(state_centers_minima[:,0],state_centers_minima[:,1],s=30,c=prob[n_states:2*n_states],cmap=plt.cm.binary)#,clim=(0,1))
inds=np.intersect1d(states_death_ERKlow,states_death_dead)
get_state_barplot(prob[inds],state_labels=state_labels_SD,ax=axs[1])
axs[1].set_title(f'{.5*itt:.0f} hrs: ERK low fraction died {np.sum(prob[inds]):.2f}')
axs[1].set_ylim(0,phigh)
#axs[1].axis('off')
#axs[2].scatter(state_centers_minima[:,0],state_centers_minima[:,1],s=30,c=prob[2*n_states:3*n_states],cmap=plt.cm.binary)#,clim=(0,1))
inds=np.intersect1d(states_death_ERKhigh,states_death_alive)
get_state_barplot(prob[inds],state_labels=state_labels_SD,ax=axs[2])
axs[2].set_title(f'{.5*itt:.0f} hrs: ERK high fraction alive {np.sum(prob[inds]):.2f}')
axs[2].set_ylim(0,phigh)
#axs[2].axis('off')
#axs[3].scatter(state_centers_minima[:,0],state_centers_minima[:,1],s=30,c=prob[3*n_states:],cmap=plt.cm.binary)#,clim=(0,1))
inds=np.intersect1d(states_death_ERKhigh,states_death_dead)
get_state_barplot(prob[inds],state_labels=state_labels_SD,ax=axs[3])
axs[3].set_title(f'{.5*itt:.0f} hrs: ERK high fraction died {np.sum(prob[inds]):.2f}')
axs[3].set_ylim(0,phigh)
#axs[3].axis('off')
axs[0].set_ylabel('population fraction')
axs[2].set_xlabel('states')
axs[2].set_ylabel('population fraction')
axs[3].set_xlabel('states')
plt.savefig(f'{date}/prob_initial_states.png')
plt.show()    

fig=plt.figure(figsize=(12,4))
ax=plt.gca()
get_state_fluxplot(P_states,prob_initial,state_centers_all,ax=ax,state_labels=state_labels_all,state_colors=state_colors_all,lws=50,mscale=50,sh=2,point_mscale=5000,fcut=.005)
draw_headers(ax)
plt.savefig(f'{date}/prob_initial_flux_states.svg')

In [ ]:
fig=plt.figure(figsize=(12,4))
ax=plt.gca()
get_state_fluxplot(P_states,prob_initial,state_centers_all,ax=ax,state_labels=state_labels_all,state_colors=state_colors_all,lws=50,mscale=50,sh=2,point_mscale=5000,fcut=.005)
draw_headers(ax)
plt.savefig(f'{date}/prob_initial_flux_states.svg')

In [ ]:
Nt=48
x_states=state_centers_minima.copy()
#prob0=np.append(np.ones(n_clusters),np.zeros(n_clusters))
#prob0=np.append(prob_initial,np.zeros(3*n_clusters))
prob0=prob_initial.copy()
prob=prob0.copy()
mfpt=np.zeros(n_states)
prob_states=np.zeros((n_states,Nt))
prob_states_ERKlow=np.zeros((n_states,Nt))
prob_states_ERKhigh=np.zeros((n_states,Nt))
prob_states[:,0]=prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDlow),states_death_alive)]+\
                   prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDlow),states_death_alive)]+\
                   prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDhigh),states_death_alive)]+\
                   prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDhigh),states_death_alive)] 
prob_states_ERKlow[:,0]=prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDlow),states_death_alive)]+\
                   prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDhigh),states_death_alive)]
prob_states_ERKhigh[:,0]=prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDlow),states_death_alive)]+\
                   prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDhigh),states_death_alive)] 
visual=True;vizpath=f'{date}/movie_prob_death_states'
os.system(f'mkdir {vizpath}')
for itt in range(1,Nt):
    prob=np.matmul(prob,P_states)
    prob_states[:,itt]=prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDlow),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDlow),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDhigh),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDhigh),states_death_alive)]
    prob_states_ERKlow[:,itt]=prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDlow),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDhigh),states_death_alive)]
    prob_states_ERKhigh[:,itt]=prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDlow),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDhigh),states_death_alive)] 
    mfpt=mfpt+.5*itt*prob[0:n_states]
    if visual and itt%1==0:
        clear_output(wait=True)
        fig=plt.figure(figsize=(12,4))
        ax=plt.gca()
        get_state_fluxplot(P_states,prob,state_centers_all,ax=ax,state_labels=None,state_colors=state_colors_all,lws=800,mscale=800,sh=2,point_mscale=5000,fcut=.001)
        draw_headers(ax)
        ax.text(row_sd_pos['SD-'][0],col_labels_major_pos['A'][1],f'{.5*itt} hrs')
        plt.savefig(f'{vizpath}/probdeath_f{itt:04d}.png')
        plt.savefig(f'{vizpath}/probdeath_f{itt:04d}.svg')
        plt.show()
        plt.pause(.1)

prob_states_erk=prob_states.copy()
prob_states_erk_ERKhigh=prob_states_ERKhigh.copy()
prob_states_erk_ERKlow=prob_states_ERKlow.copy()
prob_all=prob.copy()

In [ ]:
#ERKi
#erk inhibition
#P_states=model.get_transition_matrix_CG(x0set_death,x1set_death,clusters=clusters_death,states=stateSet_death)
indsfrom=np.intersect1d(states_death_ERKlow,states_death_alive)
indsto=np.intersect1d(states_death_ERKhigh,states_death_alive)
for i in indsfrom:
    for j in indsto:
        P_states[i,j]=0.

for istate in range(P_states.shape[0]):
    P_states[istate,:]=P_states[istate,:]/np.sum(P_states[istate,:])
    
plt.imshow(np.log(P_states),origin='lower')
plt.savefig(f'{date}/transition_matrix_states_erki.svg')

In [ ]:
x_initial=x[inds_initial_traj,:]
x_initial=np.append(x_initial,np.zeros_like(cratio_erk_traj[inds_initial_traj,np.newaxis]),axis=1)
x_initial=np.append(x_initial,Srd_traj[inds_initial_traj,np.newaxis],axis=1)
x_initial=np.append(x_initial,cratio_erk_traj[inds_initial_traj,np.newaxis],axis=1)
indgood=np.where(np.isnan(np.sum(x_initial,axis=1))==0)[0]
x_initial=x_initial[indgood,:]


ind_states_death=stateSet_death[clusters_death.assign(x_initial)]
states, counts = np.unique(ind_states_death,return_counts=True)
prob_initial=np.zeros(P_states.shape[0])
for istate in states:
    indstate=np.where(states==istate)[0]
    prob_initial[istate]=np.sum(counts[indstate])

prob_initial[states_death_ERKlow]=prob_initial[states_death_ERKlow]+prob_initial[states_death_ERKhigh]
prob_initial[states_death_ERKhigh]=0.
prob_initial=prob_initial/np.sum(prob_initial)

phigh=.8
prob=prob_initial.copy()
itt=0
fig,axs=plt.subplots(2,2,sharex=True,sharey=True,figsize=(10,5))
axs=axs.flatten()
#axs[0].scatter(state_centers_minima[:,0],state_centers_minima[:,1],s=30,c=prob[0:n_states],cmap=plt.cm.binary)#,clim=(0,1))
inds=np.intersect1d(states_death_ERKlow,states_death_alive)
get_state_barplot(prob[inds],state_labels=state_labels_SD,ax=axs[0])
axs[0].set_ylim(0,phigh)
axs[0].set_title(f'{.5*itt:.0f} hrs: ERK low fraction alive {np.sum(prob[inds]):.2f}')
#axs[0].axis('off')
#axs[1].scatter(state_centers_minima[:,0],state_centers_minima[:,1],s=30,c=prob[n_states:2*n_states],cmap=plt.cm.binary)#,clim=(0,1))
inds=np.intersect1d(states_death_ERKlow,states_death_dead)
get_state_barplot(prob[inds],state_labels=state_labels_SD,ax=axs[1])
axs[1].set_title(f'{.5*itt:.0f} hrs: ERK low fraction died {np.sum(prob[inds]):.2f}')
axs[1].set_ylim(0,phigh)
#axs[1].axis('off')
#axs[2].scatter(state_centers_minima[:,0],state_centers_minima[:,1],s=30,c=prob[2*n_states:3*n_states],cmap=plt.cm.binary)#,clim=(0,1))
inds=np.intersect1d(states_death_ERKhigh,states_death_alive)
get_state_barplot(prob[inds],state_labels=state_labels_SD,ax=axs[2])
axs[2].set_title(f'{.5*itt:.0f} hrs: ERK high fraction alive {np.sum(prob[inds]):.2f}')
axs[2].set_ylim(0,phigh)
#axs[2].axis('off')
#axs[3].scatter(state_centers_minima[:,0],state_centers_minima[:,1],s=30,c=prob[3*n_states:],cmap=plt.cm.binary)#,clim=(0,1))
inds=np.intersect1d(states_death_ERKhigh,states_death_dead)
get_state_barplot(prob[inds],state_labels=state_labels_SD,ax=axs[3])
axs[3].set_title(f'{.5*itt:.0f} hrs: ERK high fraction died {np.sum(prob[inds]):.2f}')
axs[3].set_ylim(0,phigh)
#axs[3].axis('off')
axs[0].set_ylabel('population fraction')
axs[2].set_xlabel('states')
axs[2].set_ylabel('population fraction')
axs[3].set_xlabel('states')
plt.savefig(f'{date}/prob_initial_states_erki.png')
plt.show()    

In [ ]:
Nt=48
x_states=state_centers_minima.copy()
#prob0=np.append(np.ones(n_clusters),np.zeros(n_clusters))
#prob0=np.append(prob_initial,np.zeros(3*n_clusters))
prob0=prob_initial.copy()
prob=prob0.copy()
mfpt=np.zeros(n_states)
prob_states=np.zeros((n_states,Nt))
prob_states_ERKlow=np.zeros((n_states,Nt))
prob_states_ERKhigh=np.zeros((n_states,Nt))
prob_states[:,0]=prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDlow),states_death_alive)]+\
                   prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDlow),states_death_alive)]+\
                   prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDhigh),states_death_alive)]+\
                   prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDhigh),states_death_alive)] 
prob_states_ERKlow[:,0]=prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDlow),states_death_alive)]+\
                   prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDhigh),states_death_alive)]
prob_states_ERKhigh[:,0]=prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDlow),states_death_alive)]+\
                   prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDhigh),states_death_alive)] 
visual=True;vizpath=f'{date}/movie_prob_death_states_erki'
os.system(f'mkdir {vizpath}')
for itt in range(1,Nt):
    prob=np.matmul(prob,P_states)
    prob_states[:,itt]=prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDlow),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDlow),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDhigh),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDhigh),states_death_alive)]  
    prob_states_ERKlow[:,itt]=prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDlow),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKlow,states_death_SDhigh),states_death_alive)]
    prob_states_ERKhigh[:,itt]=prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDlow),states_death_alive)]+\
                       prob[np.intersect1d(np.intersect1d(states_death_ERKhigh,states_death_SDhigh),states_death_alive)] 
    mfpt=mfpt+.5*itt*prob[0:n_states]
    if visual and itt%1==0:
        clear_output(wait=True)
        fig=plt.figure(figsize=(12,4))
        ax=plt.gca()
        get_state_fluxplot(P_states,prob,state_centers_all,ax=ax,state_labels=None,state_colors=state_colors_all,lws=800,mscale=800,sh=2,point_mscale=5000,fcut=.001)
        draw_headers(ax)
        ax.text(row_sd_pos['SD-'][0],col_labels_major_pos['A'][1],f'{.5*itt} hrs')
        plt.savefig(f'{vizpath}/probdeath_f{itt:04d}.png')
        plt.savefig(f'{vizpath}/probdeath_f{itt:04d}.svg')
        plt.show()
        
prob_states_erki=prob_states.copy()
prob_states_erki_ERKlow=prob_states_ERKlow.copy()
prob_states_erki_ERKhigh=prob_states_ERKhigh.copy()

In [ ]:
fig,axs=plt.subplots(2,2,figsize=(10,10))
axs=axs.flatten()
Nt=48
times=np.arange(Nt)*.5
xlim=Nt-1
ylim=.8
for istate in range(n_states):
    #plt.plot(times,np.divide(prob_states[istate,:],np.sum(prob_states,axis=0)),color=plt.cm.Reds(istate/n_states))
    axs[0].plot(times,prob_states_erk[istate,:],color=plt.cm.jet(istate/n_states),label=f'state {state_labels[istate]}')
    #axs[0].plot(time_inds-time_inds[0],state_probs_times[1,istate,:],color=plt.cm.jet(istate/n_states),alpha=.5) 
#for istate in range(n_states):
    #axs[0].text(times[xlim],prob_states_erk[istate,xlim],state_labels[istate])
axs[0].set_xlim(0,times[xlim]-.05)
axs[0].set_ylim(0,ylim)
axs[0].set_ylabel('combined population fraction')
axs[0].set_xlabel('time (hrs)')
#axs[0].set_ylim(.2,1.e-3)

for istate in range(n_states):
    #plt.plot(times,np.divide(prob_states[istate,:],np.sum(prob_states,axis=0)),color=plt.cm.Reds(istate/n_states))
    axs[3].plot(times,prob_states_erki[istate,:],':',color=plt.cm.jet(istate/n_states))#,label=f'state {state_labels[istate]}')
    axs[3].plot(times,prob_states_erk[istate,:],'-',color=plt.cm.jet(istate/n_states),alpha=.5)#,label=f'state {state_labels[istate]}')
    #axs[1].plot(time_inds-time_inds[0],state_probs_times[2,istate,:],color=plt.cm.jet(istate/n_states),alpha=.5) 
#for istate in range(n_states):
    #axs[1].text(times[xlim],prob_states_erki[istate,xlim],state_labels[istate])
axs[3].set_xlim(0,times[xlim]-.3)
#axs[3].set_ylim(0,ylim)
axs[3].set_xlabel('time (hrs)')

axs[0].plot(np.insert(times,0,-1),np.insert(np.sum(prob_states_erk,axis=0),0,1.),color='black',label='total')
#axs[0].text(times[xlim],np.sum(prob_states_erk[:,xlim],axis=0),'total')
axs[0].set_xlim(0,times[xlim]-.05)
axs[0].set_ylim(0.,1)
#axs[0].set_ylabel('relative total population')
axs[0].set_xlabel('time (hrs)')
axs[0].legend(loc='upper right')
#axs[0].set_ylim(.2,1.e-3)

axs[3].plot(np.insert(times,0,-1),np.insert(np.sum(prob_states_erk,axis=0),0,1.),'-',color='black',label='total MCL1i')
axs[3].plot(np.insert(times,0,-1),np.insert(np.sum(prob_states_erki,axis=0),0,1.),':',color='black',label='total MCL1i+ERKi')
#axs[1].text(times[xlim],np.sum(prob_states_erki[:,xlim],axis=0),'total')
axs[3].set_xlim(0,times[xlim]-.3)
#axs[3].set_ylim(0.,1)
axs[3].set_xlabel('time (hrs)')
axs[3].set_ylabel('combined population fraction')
axs[3].set_title('MCL1i + ERKi prediction')
axs[3].legend(loc='upper right')
#axs[1].set_ylim(.2,1.e-3)


for istate in range(n_states):
    #plt.plot(times,np.divide(prob_states[istate,:],np.sum(prob_states,axis=0)),color=plt.cm.Reds(istate/n_states))
    axs[2].plot(times,np.divide(prob_states_erk_ERKhigh[istate,:],prob_states_erk_ERKlow[istate,:]),color=plt.cm.jet(istate/n_states),label=f'state {state_labels[istate]}')
    #axs[0].plot(time_inds-time_inds[0],state_probs_times[1,istate,:],color=plt.cm.jet(istate/n_states),alpha=.5) 
#for istate in range(n_states):
    #axs[0].text(times[xlim],prob_states_erk[istate,xlim],state_labels[istate])
axs[2].plot(times,np.divide(prob_states_erk_ERKhigh[istate,:],prob_states_erk_ERKlow[istate,:]),color=plt.cm.jet(istate/n_states),label=f'state {state_labels[istate]}')
axs[2].set_xlim(0,times[xlim]-.05)
#axs[1].set_ylim(0,ylim)
axs[2].set_ylabel('ERK high/low relative population')
axs[2].set_xlabel('time (hrs)')
#axs[0].set_ylim(.2,1.e-3)

#for istate in range(n_states):
#    #plt.plot(times,np.divide(prob_states[istate,:],np.sum(prob_states,axis=0)),color=plt.cm.Reds(istate/n_states))
#    axs[1].plot(times,prob_states_erk_ERKlow[istate,:],'-',color=plt.cm.jet(istate/n_states),label=f'state {state_labels[istate]} ERK-low')
#    axs[1].plot(times,prob_states_erk_ERKhigh[istate,:],'--',color=plt.cm.jet(istate/n_states),label=f'state {state_labels[istate]} ERK-high')

    #axs[1].plot(time_inds-time_inds[0],state_probs_times[2,istate,:],color=plt.cm.jet(istate/n_states),alpha=.5) 
#for istate in range(n_states):
    #axs[1].text(times[xlim],prob_states_erki[istate,xlim],state_labels[istate])
axs[1].plot(np.insert(times,0,-1),np.insert(np.sum(prob_states_erk_ERKlow,axis=0),0,1.),'-',color='black',label='total ERK-low')
axs[1].plot(np.insert(times,0,-1),np.insert(np.sum(prob_states_erk_ERKhigh,axis=0),0,1.),'--',color='black',label='total ERK-high')
axs[1].set_xlim(0,times[xlim]-.3)
axs[1].set_ylim(0,1)
#axs[2].set_ylim(0,ylim)
axs[1].legend(loc='upper right')
axs[1].set_xlabel('time (hrs)')
axs[1].set_ylabel('population fraction')

#axs[1].plot(np.insert(times,0,-1),np.insert(np.sum(prob_states_erk_ERKlow,axis=0),0,1.),color='black',label='total')
#axs[0].text(times[xlim],np.sum(prob_states_erk[:,xlim],axis=0),'total')
#axs[1].set_ylim(0.,1)
#axs[1].legend(loc='upper right')
#axs[0].set_ylim(.2,1.e-3)

#axs[2].plot(np.insert(times,0,-1),np.insert(np.sum(prob_states_erk_ERKhigh,axis=0),0,1.),color='black',label='total')
#axs[1].text(times[xlim],np.sum(prob_states_erki[:,xlim],axis=0),'total')
#axs[2].legend(loc='upper right')
#axs[1].set_ylim(.2,1.e-3)
plt.savefig(f'{date}/mcl1i_ERKi_sim.svg')

In [ ]:
prob=prob_all.copy()
phigh=.23
fig,axs=plt.subplots(2,2,sharex=True,sharey=True,figsize=(10,5))
axs=axs.flatten()
inds=np.intersect1d(states_death_ERKlow,states_death_alive)
get_state_barplot(prob[inds],state_labels=state_labels_SD,ax=axs[0])
axs[0].set_ylim(0,phigh)
axs[0].set_title(f'{.5*itt:.0f} hrs: ERK low fraction alive {np.sum(prob[inds]):.2f}')
#axs[0].axis('off')
#axs[1].scatter(state_centers_minima[:,0],state_centers_minima[:,1],s=30,c=prob[n_states:2*n_states],cmap=plt.cm.binary)#,clim=(0,1))
inds=np.intersect1d(states_death_ERKlow,states_death_dead)
get_state_barplot(prob[inds],state_labels=state_labels_SD,ax=axs[1])
axs[1].set_title(f'{.5*itt:.0f} hrs: ERK low fraction died {np.sum(prob[inds]):.2f}')
axs[1].set_ylim(0,phigh)
#axs[1].axis('off')
#axs[2].scatter(state_centers_minima[:,0],state_centers_minima[:,1],s=30,c=prob[2*n_states:3*n_states],cmap=plt.cm.binary)#,clim=(0,1))
inds=np.intersect1d(states_death_ERKhigh,states_death_alive)
get_state_barplot(prob[inds],state_labels=state_labels_SD,ax=axs[2])
axs[2].set_title(f'{.5*itt:.0f} hrs: ERK high fraction alive {np.sum(prob[inds]):.2f}')
axs[2].set_ylim(0,phigh)
#axs[2].axis('off')
#axs[3].scatter(state_centers_minima[:,0],state_centers_minima[:,1],s=30,c=prob[3*n_states:],cmap=plt.cm.binary)#,clim=(0,1))
inds=np.intersect1d(states_death_ERKhigh,states_death_dead)
get_state_barplot(prob[inds],state_labels=state_labels_SD,ax=axs[3])
axs[3].set_title(f'{.5*itt:.0f} hrs: ERK high fraction died {np.sum(prob[inds]):.2f}')
axs[3].set_ylim(0,phigh)
#axs[3].axis('off')
axs[0].set_ylabel('population fraction')
axs[2].set_xlabel('states')
axs[2].set_ylabel('population fraction')
axs[3].set_xlabel('states')
plt.savefig(f'{date}/probfinal_mcl1i_sim.svg')

In [ ]:
plt.plot(np.insert(times,0,-1),np.insert(np.sum(prob_states_erk,axis=0),0,1.),color='black')
plt.plot(np.insert(times,0,-1),np.insert(np.sum(prob_states_erki,axis=0),0,1.),color='purple')
#axs[3].set_xlim(0,times[xlim]-.3)
#axs[3].set_ylim(0,1)
plt.xlim(times[0],times[xlim]-.3)
plt.xlabel('time (hrs)')
plt.title('MCL1i + complete ERK inhibition in silico')

In [ ]:
x0set_death=[]
x1set_death=[]
fw_erk=1
min_nerk=0
for imodel in indgood_models:
    sctm=modelSet[imodel]
    indmodel=np.where(inds_trajsteps_models==imodel)[0]
    #inds_trajsteps_model1=inds_trajsteps[indmodel,1]
    #inds_death_trajsteps_model=np.where(np.isin(inds_trajsteps_model1,inds_death_traj_set[i_tmr]))[0]
    #inds_death_trajsteps_model=np.where(np.isin(indmodel,inds_death_traj_set[i_tmr]))[0]
    inds1=inds_trajsteps[indmodel,1]
    #times1=sctm.cells_timeSet[inds1]
    #indmodel=indmodel[np.logical_and(times1<tu,times1>tl)]
    #inds1=inds1[np.logical_and(times1<tu,times1>tl)]
    x0=x0set[indmodel,:]
    x1=x1set[indmodel,:]
    x0=np.append(x0,np.zeros(x0.shape[0])[:,np.newaxis],axis=1)
    x1=np.append(x1,np.zeros(x1.shape[0])[:,np.newaxis],axis=1)
    inds_death_trajsteps_model=np.where(np.isin(inds1,sctm.inds_death))[0]
    x1[inds_death_trajsteps_model,-1]=1.
    SD0=sctm.S_r_death[inds_trajsteps[indmodel,0]]
    #SD0[SD0>5.5]=np.nan
    SD1=sctm.S_r_death[inds_trajsteps[indmodel,1]]
    #SD1[SD1>5.5]=np.nan
    x0=np.append(x0,SD0[:,np.newaxis],axis=1)
    x1=np.append(x1,SD1[:,np.newaxis],axis=1)
    erk_model0=np.ones(inds1.size)*np.nan
    erk_model1=np.ones(inds1.size)*np.nan
    for i in range(inds1.size):
        ic0=inds_trajsteps[indmodel[i],0]
        ic1=inds_trajsteps[indmodel[i],1]
        erktraj0=sctm.x_signal[sctm.get_cell_trajectory(ic0)][-fw_erk:]
        good0=np.isfinite(erktraj0)
        if np.sum(good0)>min_nerk:
            #erk_model0[i]=np.nanmean(erktraj0)
            erk_model0[i]=erktraj0[np.where(np.isfinite(erktraj0))[0][-1]]
        erktraj1=sctm.x_signal[sctm.get_cell_trajectory(ic1)][-fw_erk:]
        good1=np.isfinite(erktraj1)
        if np.sum(good1)>min_nerk:
            erk_model1[i]=erktraj1[np.where(np.isfinite(erktraj1))[0][-1]]
            #erk_model1[i]=np.nanmean(erktraj1)
        if x1[i,inds_death_life]==1.:
            print(f'm{imodel} {erk_model0[i]} to {erk_model1[i]}')
    #erk_model0=sctm.x_signal[inds_trajsteps[indmodel,0]]
    #erk_model1=sctm.x_signal[inds_trajsteps[indmodel,1]]
    x0=np.append(x0,erk_model0[:,np.newaxis],axis=1)
    x1=np.append(x1,erk_model1[:,np.newaxis],axis=1)
    #indgood0=np.isfinite(np.sum(x0,axis=1))
    #indgood1=np.isfinite(np.sum(x1,axis=1))
    #indgood=np.logical_and(indgood0,indgood1)
    x0set_death.append(x0) #[indgood,:])
    x1set_death.append(x1) #[indgood,:])

x0set_death=np.concatenate(x0set_death)
x1set_death=np.concatenate(x1set_death)

In [ ]:
tms=[1] #for mcl1i
tl=0
tu=25.5
nboots=100
fig,axs=plt.subplots(1,1,figsize=(6,6))
for iplot in range(len(tms)):
    i_tm=tms[iplot]
    #centers_minima=clusters.clustercenters.copy()
    #clusters_minima=coor.clustering.AssignCenters(centers_minima, metric='euclidean')
    indmodels=np.where(inds_tmSet_models==i_tm)[0]
    P_ons_SDlow_set=[]
    P_ons_SDhigh_set=[]
    P_offs_SDlow_set=[]
    P_offs_SDhigh_set=[]
    P_deaths_ERKlow_set=[]
    P_deaths_ERKhigh_set=[]
    P_ons_SDlow_CR=np.zeros((n_states,3))
    P_ons_SDhigh_CR=np.zeros((n_states,3))
    P_offs_SDlow_CR=np.zeros((n_states,3))
    P_offs_SDhigh_CR=np.zeros((n_states,3))
    P_deaths_ERKlow_CR=np.zeros((n_states,3))
    P_deaths_ERKhigh_CR=np.zeros((n_states,3))
    imagingSets=np.unique(inds_imagingSet_models[indmodels])
    for iSet in imagingSets:
        x0=[]
        x1=[]
        indmodels_imagingSets=np.where(inds_imagingSet_models==iSet)[0]
        #indmodels_Set=np.intersect1d(indmodels,indmodels_imagingSets)
        indmodels_Set=np.setdiff1d(indmodels,indmodels_imagingSets)
        indmodels_Set=np.intersect1d(indmodels_Set,indgood_models)
        for iboot in range(nboots):
            x0=[]
            x1=[]
            indmodels_boots= np.random.choice(indmodels_Set, size=indmodels_Set.size, replace=True)
            for imodel in indmodels_boots:
                sctm=modelSet[imodel]
                inds_model=np.where(inds_trajsteps_models==imodel)[0]
                inds1=inds_trajsteps[inds_model,1]
                times1=sctm.cells_timeSet[inds1]
                inds_model=inds_model[np.logical_and(times1<tu,times1>tl)]
                inds1=inds1[np.logical_and(times1<tu,times1>tl)]
                #x0=x0set[inds_model,:]
                #x1=x1set[inds_model,:]
                x0m=x0set_death[inds_model,:]
                x1m=x1set_death[inds_model,:]
                indgood0=np.isfinite(np.sum(x0m,axis=1))
                indgood1=np.isfinite(np.sum(x1m,axis=1))
                indgood=np.logical_and(indgood0,indgood1)
                x0.append(x0set_death[inds_model[indgood],:])
                x1.append(x1set_death[inds_model[indgood],:])    
            x0=np.concatenate(x0)
            x1=np.concatenate(x1)
            pdeath_ERKhigh=np.sum(np.logical_and(x1[:,inds_death_life]==1,x1[:,inds_death_ERK]>midcut))/np.sum(x1[:,inds_death_ERK]>midcut)
            pdeath_ERKlow=np.sum(np.logical_and(x1[:,inds_death_life]==1,x1[:,inds_death_ERK]<midcut))/np.sum(x1[:,inds_death_ERK]<midcut)
            P_states=model.get_transition_matrix_CG(x0,x1,clusters=clusters_death,states=stateSet_death,return_counts=True)
            P_states=clean_P(P_states,pdeath_ERKhigh=pdeath_ERKhigh,pdeath_ERKlow=pdeath_ERKlow,death_boost=death_boost,states_death_dead=states_death_dead,states_death_alive=states_death_alive,states_death_ERKlow=states_death_ERKlow,states_death_ERKhigh=states_death_ERKhigh,states_death_SDhigh=states_death_SDhigh,states_death_SDlow=states_death_SDlow)
            P_ons_SDlow=np.zeros(n_states)
            P_ons_SDhigh=np.zeros(n_states)
            P_offs_SDlow=np.zeros(n_states)
            P_offs_SDhigh=np.zeros(n_states)
            P_deaths_ERKlow=np.zeros(n_states)
            P_deaths_ERKhigh=np.zeros(n_states)
            indsfrom=np.intersect1d(states_death_ERKlow,states_death_SDlow)
            indsfrom=np.intersect1d(indsfrom,states_death_alive)
            indsto=np.intersect1d(states_death_ERKhigh,states_death_alive)
            indsto=np.intersect1d(indsto,states_death_alive)
            ptest=np.zeros(indsfrom.size)
            itest=0
            for i in indsfrom:
                for j in indsto:
                    ptest[itest]=ptest[itest]+P_states[i,j]
                itest=itest+1
            P_ons_SDlow=ptest.copy()
            indsfrom=np.intersect1d(states_death_ERKhigh,states_death_SDlow)
            indsfrom=np.intersect1d(indsfrom,states_death_alive)
            indsto=np.intersect1d(states_death_ERKlow,states_death_alive)
            indsto=np.intersect1d(indsto,states_death_alive)
            ptest=np.zeros(indsfrom.size)
            itest=0
            for i in indsfrom:
                for j in indsto:
                    ptest[itest]=ptest[itest]+P_states[i,j]
                itest=itest+1
            P_offs_SDlow=ptest.copy()
            indsfrom=np.intersect1d(states_death_ERKlow,states_death_SDhigh)
            indsfrom=np.intersect1d(indsfrom,states_death_alive)
            indsto=np.intersect1d(states_death_ERKhigh,states_death_alive)
            indsto=np.intersect1d(indsto,states_death_alive)
            ptest=np.zeros(indsfrom.size)
            itest=0
            for i in indsfrom:
                for j in indsto:
                    ptest[itest]=ptest[itest]+P_states[i,j]
                itest=itest+1
            P_ons_SDhigh=ptest.copy()
            indsfrom=np.intersect1d(states_death_ERKhigh,states_death_SDhigh)
            indsfrom=np.intersect1d(indsfrom,states_death_alive)
            indsto=np.intersect1d(states_death_ERKlow,states_death_alive)
            indsto=np.intersect1d(indsto,states_death_alive)
            ptest=np.zeros(indsfrom.size)
            itest=0
            for i in indsfrom:
                for j in indsto:
                    ptest[itest]=ptest[itest]+P_states[i,j]
                itest=itest+1
            P_offs_SDhigh=ptest.copy()
            indsfrom=np.intersect1d(states_death_ERKlow,states_death_SDlow)
            indsfrom=np.intersect1d(indsfrom,states_death_alive)
            indsto=np.intersect1d(states_death_dead,states_death_dead)
            indsto=np.intersect1d(indsto,states_death_dead)
            ptest=np.zeros(indsfrom.size)
            itest=0
            for i in indsfrom:
                for j in indsto:
                    ptest[itest]=ptest[itest]+P_states[i,j]
                itest=itest+1
            P_deaths_ERKlow=ptest.copy()
            indsfrom=np.intersect1d(states_death_ERKhigh,states_death_SDlow)
            indsfrom=np.intersect1d(indsfrom,states_death_alive)
            indsto=np.intersect1d(states_death_dead,states_death_dead)
            indsto=np.intersect1d(indsto,states_death_dead)
            ptest=np.zeros(indsfrom.size)
            itest=0
            for i in indsfrom:
                for j in indsto:
                    ptest[itest]=ptest[itest]+P_states[i,j]
                itest=itest+1
            P_deaths_ERKhigh=ptest.copy()
            #print(f'rep {iSet} death rate ERK-low: {np.sum(P_deaths_ERKlow)} ERK-high: {np.sum(P_deaths_ERKhigh)}')
            if np.sum(P_ons_SDlow>0.):
                P_ons_SDlow_set.append(P_ons_SDlow)
            if np.sum(P_ons_SDhigh>0.):
                P_ons_SDhigh_set.append(P_ons_SDhigh)
            if np.sum(P_offs_SDlow>0.):
                P_offs_SDlow_set.append(P_offs_SDlow)
            if np.sum(P_offs_SDhigh>0.):
                P_offs_SDhigh_set.append(P_offs_SDhigh)
            if np.sum(P_deaths_ERKlow>0.):
                P_deaths_ERKlow_set.append(P_deaths_ERKlow)
            if np.sum(P_deaths_ERKhigh>-1): #include even if no counts for control# 0.):
                P_deaths_ERKhigh_set.append(P_deaths_ERKhigh)
    P_ons_SDlow_set=np.array(P_ons_SDlow_set)
    P_ons_SDhigh_set=np.array(P_ons_SDhigh_set)
    P_offs_SDlow_set=np.array(P_offs_SDlow_set)
    P_offs_SDhigh_set=np.array(P_offs_SDhigh_set)
    P_deaths_ERKhigh_set=np.array(P_deaths_ERKhigh_set)
    P_deaths_ERKlow_set=np.array(P_deaths_ERKlow_set)
    for i in range(n_states):
        P_ons_SDlow_CR[i,0]=np.nanmean(P_ons_SDlow_set[:,i])
        P_ons_SDhigh_CR[i,0]=np.nanmean(P_ons_SDhigh_set[:,i])
        P_offs_SDlow_CR[i,0]=np.nanmean(P_offs_SDlow_set[:,i])
        P_offs_SDhigh_CR[i,0]=np.nanmean(P_offs_SDhigh_set[:,i])
        P_deaths_ERKlow_CR[i,0]=np.nanmean(P_deaths_ERKlow_set[:,i])
        P_deaths_ERKhigh_CR[i,0]=np.nanmean(P_deaths_ERKhigh_set[:,i])
        P_ons_SDlow_CR[i,1:]=bayesian_bootstrap.central_credible_interval(bayesian_bootstrap.mean(P_ons_SDlow_set[:,i],n_replications=1000),alpha=.05)#bootstrap.get_CI(P_ons_SDlow_set[:,i],1000)
        P_ons_SDhigh_CR[i,1:]=bayesian_bootstrap.central_credible_interval(bayesian_bootstrap.mean(P_ons_SDhigh_set[:,i],n_replications=1000),alpha=.05)#bootstrap.get_CI(P_ons_SDhigh_set[:,i],1000)
        P_offs_SDlow_CR[i,1:]=bayesian_bootstrap.central_credible_interval(bayesian_bootstrap.mean(P_offs_SDlow_set[:,i],n_replications=1000),alpha=.05)#bootstrap.get_CI(P_offs_SDlow_set[:,i],1000)
        P_offs_SDhigh_CR[i,1:]=bayesian_bootstrap.central_credible_interval(bayesian_bootstrap.mean(P_offs_SDhigh_set[:,i],n_replications=1000),alpha=.05)#bootstrap.get_CI(P_offs_SDhigh_set[:,i],1000)
        P_deaths_ERKlow_CR[i,1:]=bayesian_bootstrap.central_credible_interval(bayesian_bootstrap.mean(P_deaths_ERKlow_set[:,i],n_replications=1000),alpha=.05)#bootstrap.get_CI(P_deaths_ERKlow_set[:,i],1000)
        P_deaths_ERKhigh_CR[i,1:]=bayesian_bootstrap.central_credible_interval(bayesian_bootstrap.mean(P_deaths_ERKhigh_set[:,i],n_replications=1000),alpha=.05)#bootstrap.get_CI(P_deaths_ERKhigh_set[:,i],1000)

In [ ]:
states_death_bystate=[None]*n_states
for istate in range(n_states):
    states_death_bystate[istate]=np.where(np.char.find(state_labels_all,state_labels[istate])!=-1)[0]
    print(states_death_bystate[istate])

nstates_death=len(state_labels_all)

In [ ]:
#get state probabilities
tms=[1]
tl=0
tu=25
tw=1 #frame window
timestep=.5
times_states=np.arange(tl,tu-tw,timestep)
nt=times_states.size
nboots=10
fig,axs=plt.subplots(1,1,figsize=(6,6))
for iplot in range(len(tms)):
    i_tm=tms[iplot]
    prob_states_all=np.zeros((nt,n_states,3))
    prob_states_erkon=np.zeros((nt,n_states,3))
    prob_states_erkoff=np.zeros((nt,n_states,3))
    for itt in range(nt):
        tl=times_states[itt]
        tu=tl+tw
        #centers_minima=clusters.clustercenters.copy()
        #clusters_minima=coor.clustering.AssignCenters(centers_minima, metric='euclidean')
        indmodels=np.where(inds_tmSet_models==i_tm)[0]
        P_all_set=[]
        P_erkon_set=[]
        P_erkoff_set=[]
        P_all_CR=np.zeros((n_states,3))
        P_erkon_CR=np.zeros((n_states,3))
        P_erkoff_CR=np.zeros((n_states,3))
        imagingSets=np.unique(inds_imagingSet_models[indmodels])
        for iSet in imagingSets:
            x0=[]
            x1=[]
            indmodels_imagingSets=np.where(inds_imagingSet_models==iSet)[0]
            indmodels_Set=np.intersect1d(indmodels,indmodels_imagingSets)
            #indmodels_Set=np.setdiff1d(indmodels,indmodels_imagingSets)
            for iboot in range(nboots):
                #x0=[]
                #x1=[]
                x_traj=[]
                indmodels_boots= np.random.choice(indmodels_Set, size=indmodels_Set.size, replace=True)
                prob_all=np.zeros(nt)
                for itt_boot in range(nt):
                    tl_boot=times_states[itt_boot]
                    tu_boot=tl_boot+tw
                    #indtm=np.where(indtreatment==i_tm)[0]
                    indtm=np.where(np.isin(indcells_models,indmodels_boots))[0]
                    indstw=np.where(np.logical_and(times_models<tu_boot,times_models>=tl_boot))[0]
                    ind_erkgood=np.where(np.isfinite(cratio_erk))[0]
                    indstw=np.intersect1d(indstw,ind_erkgood)
                    indstw=np.intersect1d(indstw,indtm)
                    if itt_boot==0:
                        c0=indstw.size
                    prob_all[itt_boot]=indstw.size/c0
                for imodel in indmodels_boots:
                    sctm=modelSet[imodel]
                    inds_model=np.where(inds_trajsteps_models==imodel)[0]
                    #inds1=inds_trajsteps[inds_model,1]
                    #times1=sctm.cells_timeSet[inds1]
                    #inds_model=inds_model[np.logical_and(times1<tu,times1>=tl)]
                    #inds1=inds1[np.logical_and(times1<tu,times1>tl)]
                    ##x0=x0set[inds_model,:]
                    ##x1=x1set[inds_model,:]
                    #x0m=x0set_death[inds_model,:]
                    #x1m=x1set_death[inds_model,:]
                    #indgood0=np.isfinite(np.sum(x0m,axis=1))
                    #indgood1=np.isfinite(np.sum(x1m,axis=1))
                    #indgood=np.logical_and(indgood0,indgood1)
                    #x0.append(x0set_death[inds_model[indgood],:])
                    #x1.append(x1set_death[inds_model[indgood],:])
                    indtraj=np.where(indtreatment_traj==imodel)[0]
                    xt=np.append(x[indtraj,:],np.zeros((indtraj.size,1)),axis=1)
                    xt=np.append(xt,Srd_traj[indtraj,np.newaxis],axis=1)
                    erkt=cratio_erk[indtraj,np.newaxis]
                    #erkt[np.isnan(erkt)]=0.
                    xt=np.append(xt,erkt,axis=1)
                    indgood=np.where(np.isfinite(np.sum(xt,axis=1)))[0]
                    x_traj.append(xt[indgood,:])
                #x0=np.concatenate(x0)
                #x1=np.concatenate(x1)
                x_traj=np.concatenate(x_traj)
                indc_traj=stateSet_death[clusters_death.assign(x_traj)]
                statesc,counts_c=np.unique(indc_traj,return_counts=True)
                counts_states=np.zeros(nstates_death)
                counts_states[statesc]=counts_c
                #counts_states_matrix=model.get_transition_matrix_CG(x0,x1,clusters=clusters_death,states=stateSet_death,return_counts=True)
                #counts_states=np.sum(counts_states_matrix,axis=1)
                prob_states=counts_states/np.sum(counts_states)
                prob_states=prob_states*prob_all[itt]
                P_all=np.zeros(n_states)
                P_erkon=np.zeros(n_states)
                P_erkoff=np.zeros(n_states)
                for istate in range(n_states):                
                    indsall=np.intersect1d(states_death_bystate[istate],states_death_alive)
                    P_all[istate]=np.sum(prob_states[indsall])
                    indson=np.intersect1d(indsall,states_death_ERKhigh)
                    P_erkon[istate]=np.sum(prob_states[indson])
                    indsoff=np.intersect1d(indsall,states_death_ERKlow)
                    P_erkoff[istate]=np.sum(prob_states[indsoff])
                #if np.sum(P_all)>0.:
                P_all_set.append(P_all)
                #if np.sum(P_erkon)>0.:
                P_erkon_set.append(P_erkon)
                #if np.sum(P_erkoff)>0.:
                P_erkoff_set.append(P_erkoff)
        P_all_set=np.array(P_all_set)
        P_erkon_set=np.array(P_erkon_set)
        P_erkoff_set=np.array(P_erkoff_set)
        for i in range(n_states):
            P_all_CR[i,0]=np.nanmean(P_all_set[:,i])
            P_erkon_CR[i,0]=np.nanmean(P_erkon_set[:,i])
            P_erkoff_CR[i,0]=np.nanmean(P_erkoff_set[:,i])
            P_all_CR[i,1:]=bayesian_bootstrap.central_credible_interval(bayesian_bootstrap.mean(P_all_set[:,i],n_replications=1000),alpha=.05)#bootstrap.get_CI(P_all_set[:,i],1000)
            P_erkon_CR[i,1:]=bayesian_bootstrap.central_credible_interval(bayesian_bootstrap.mean(P_erkon_set[:,i],n_replications=1000),alpha=.05)#bootstrap.get_CI(P_erkon_set[:,i],1000)     
            P_erkoff_CR[i,1:]=bayesian_bootstrap.central_credible_interval(bayesian_bootstrap.mean(P_erkoff_set[:,i],n_replications=1000),alpha=.05)#bootstrap.get_CI(P_erkoff_set[:,i],1000)
        prob_states_all[itt,:,:]=P_all_CR
        prob_states_erkon[itt,:,:]=P_erkon_CR
        prob_states_erkoff[itt,:,:]=P_erkoff_CR
        print(f't{times_states[itt]} prob {prob_states_all[itt,:,0]}')

In [ ]:
fw=2
prob_states_all_tsmth=np.zeros_like(prob_states_all)
prob_states_erkon_tsmth=np.zeros_like(prob_states_erkon)
prob_states_erkoff_tsmth=np.zeros_like(prob_states_erkoff)
prob_states_all_tsmth[0,:,:]=prob_states_all[0,:,:]
prob_states_erkon_tsmth[0,:,:]=prob_states_erkon[0,:,:]
prob_states_erkoff_tsmth[0,:,:]=prob_states_erkoff[0,:,:]
for itt in range(0,nt):
    fu=itt+fw
    if fu>nt:
        fu=nt
    indt=np.arange(itt,fu).astype(int)
    prob_states_all_tsmth[itt,:,:]=np.nanmean(prob_states_all[indt,:,:],axis=0)
    prob_states_erkon_tsmth[itt,:,:]=np.nanmean(prob_states_erkon[indt,:,:],axis=0)
    prob_states_erkoff_tsmth[itt,:,:]=np.nanmean(prob_states_erkoff[indt,:,:],axis=0)

In [ ]:
fig,axs=plt.subplots(2,3,figsize=(10,5),sharex=True,sharey=True)
#ax.fill_between(times_states,prob_states_erkoff_tsmth[:,istate,1],'--',color=plt.cm.jet(istate/n_states))
#axs=axs.flatten()
for istate in range(n_states):    
    ax=axs[0,istate]
    ax.fill_between(times_states+tw,prob_states_erkoff_tsmth[:,istate,1],prob_states_erkoff_tsmth[:,istate,2],color=plt.cm.jet(istate/n_states),alpha=.25)
    ax.plot(times_states+tw,prob_states_erkoff_tsmth[:,istate,0],'-',color=plt.cm.jet(istate/n_states))
    ax.plot(times,prob_states_erk_ERKlow[istate,:].T,'--',color=plt.cm.jet(istate/n_states))
    ax.set_xlim(-.2,22.5)
    ax.set_ylim(0,.55)
    if istate==0:
        ax.set_ylabel('population fraction')
    ax=axs[1,istate]
    ax.fill_between(times_states+tw,prob_states_erkon_tsmth[:,istate,1],prob_states_erkon_tsmth[:,istate,2],color=plt.cm.jet(istate/n_states),alpha=.25)
    ax.plot(times_states+tw,prob_states_erkon_tsmth[:,istate,0],'-',color=plt.cm.jet(istate/n_states))
    ax.plot(times,prob_states_erk_ERKhigh[istate,:].T,'--',color=plt.cm.jet(istate/n_states))
    ax.set_xlim(-.2,22.5)
    ax.set_ylim(0,.55)
    if istate==0:
        ax.set_ylabel('population fraction')
    ax.set_xlabel('time (hrs)')

plt.tight_layout()
plt.savefig(f'{date}/state_probabilities_modelcomp_mcl1i.svg')    


In [ ]:
#for regular
prob_initial=np.zeros(n_states_death)
inds=np.intersect1d(states_death_SDlow,states_death_alive)
inds=np.intersect1d(inds,states_death_ERKlow)
prob_initial[inds]=prob_states_erkoff_tsmth[0,:,0]
inds=np.intersect1d(states_death_SDlow,states_death_alive)
inds=np.intersect1d(inds,states_death_ERKhigh)
prob_initial[inds]=prob_states_erkon_tsmth[0,:,0]

In [ ]:
#for erki
prob_initial=np.zeros(n_states_death)
inds=np.intersect1d(states_death_SDlow,states_death_alive)
inds=np.intersect1d(inds,states_death_ERKlow)
prob_initial[inds]=prob_states_erkoff_tsmth[0,:,0]
#inds=np.intersect1d(states_death_SDlow,states_death_alive)
#inds=np.intersect1d(inds,states_death_ERKhigh)
#prob_initial[inds]=prob_states_erkon_tsmth[0,:,0]
prob_initial=prob_initial/np.sum(prob_initial)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_erk_switching(
    P_offs_SDlow_CR,   # shape (3,3): [mean, CI_low, CI_high] per state
    p_offs_SDhigh_CR,  # shape (3,3)
    p_ons_SDlow_CR,    # shape (3,3)
    p_ons_SDhigh_CR,   # shape (3,3)
    state_labels=("A", "B", "C"),
    rate_units="",
    figsize=(10, 4.5)
):
    """
    Plots ERK-switching kinetic ON/OFF rates with 95% CI error bars
    for SD-low and SD-high across three states.

    Each input array is (3,3): rows = states A,B,C; cols = [mean, lowCI, highCI].
    """
    def ci_to_yerr(arr):
        """Return means and asymmetric yerr=[[lower...],[upper...]] for plt.bar."""
        arr = np.asarray(arr, dtype=float)
        means = arr[:, 0]
        lower = means - arr[:, 1]
        upper = arr[:, 2] - means
        # Ensure non-negative error magnitudes
        lower = np.clip(lower, 0, None)
        upper = np.clip(upper, 0, None)
        yerr = np.vstack([lower, upper])
        return means, yerr

    # Prepare data
    on_low_mean,  on_low_yerr  = ci_to_yerr(p_ons_SDlow_CR)
    on_high_mean, on_high_yerr = ci_to_yerr(p_ons_SDhigh_CR)
    off_low_mean, off_low_yerr  = ci_to_yerr(P_offs_SDlow_CR)
    off_high_mean, off_high_yerr = ci_to_yerr(p_offs_SDhigh_CR)

    x = np.arange(len(state_labels))
    width = 0.38

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)

    # ON rates subplot
    ax = axes[0]
    ax.bar(x - width/2, on_low_mean,  width, yerr=on_low_yerr,  capsize=4, label='SD-low',color='dimgray')
    ax.bar(x + width/2, on_high_mean, width, yerr=on_high_yerr, capsize=4, label='SD-high',color='orange')
    ax.set_xticks(x, state_labels)
    ax.set_xlabel("state")
    ax.set_ylabel(f"transition rate ({rate_units})")
    ax.set_title("ERK-switching ON rates")
    #ax.grid(axis='y', linewidth=0.4, alpha=0.6)
    ax.legend(frameon=False)

    # OFF rates subplot
    ax = axes[1]
    ax.bar(x - width/2, off_low_mean,  width, yerr=off_low_yerr,  capsize=4, label='SD-low',color='dimgray')
    ax.bar(x + width/2, off_high_mean, width, yerr=off_high_yerr, capsize=4, label='SD-high',color='orange')
    ax.set_xticks(x, state_labels)
    ax.set_xlabel("state")
    ax.set_title("ERK-switching OFF rates")
    #ax.grid(axis='y', linewidth=0.4, alpha=0.6)
    ax.legend(frameon=False)

    fig.tight_layout()
    return fig, axes

# --- Example usage (remove once you plug in real arrays) ---
# p_ons_SDlow_CR   = np.array([[0.20, 0.15, 0.25],
#                              [0.35, 0.30, 0.42],
#                              [0.28, 0.22, 0.33]])
# p_ons_SDhigh_CR  = np.array([[0.18, 0.13, 0.24],
#                              [0.30, 0.24, 0.38],
#                              [0.25, 0.19, 0.32]])
# P_offs_SDlow_CR  = np.array([[0.12, 0.09, 0.16],
#                              [0.22, 0.18, 0.28],
#                              [0.17, 0.13, 0.22]])
# p_offs_SDhigh_CR = np.array([[0.15, 0.11, 0.20],
#                              [0.25, 0.20, 0.31],
#                              [0.19, 0.15, 0.24]])
plot_erk_switching(P_offs_SDlow_CR, P_offs_SDhigh_CR, P_ons_SDlow_CR, P_ons_SDhigh_CR)
plt.savefig(f'{date}/erk_switching_model.svg')
# plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_erk_deaths_state_avg(
    P_deaths_ERKhigh_CR,  # shape (3,3): rows=states A,B,C; cols=[mean, lowCI, highCI]
    P_deaths_ERKlow_CR,   # shape (3,3): rows=states A,B,C; cols=[mean, lowCI, highCI]
    ylabel="death rate (events/frame)",
    title="",
    figsize=(3, 4.5)
):
    """
    Make a single-panel bar plot of death probabilities for ERK-low vs ERK-high,
    where each is the average across states A/B/C. Error bars are asymmetric 95% CIs,
    formed by averaging the per-state CI bounds.

    Inputs:
      P_deaths_ERKhigh_CR: (3,3) array for ERK-high; cols = [mean, lowCI, highCI]
      P_deaths_ERKlow_CR : (3,3) array for ERK-low;  cols = [mean, lowCI, highCI]
    """

    def avg_across_states(arr):
        """Return aggregated [mean_of_means, mean_lowCI, mean_highCI] across states."""
        arr = np.asarray(arr, dtype=float)
        return np.nanmean(arr, axis=0)  # robust to NaNs if any

    def ci_to_yerr_from_triplet(triplet):
        """
        Convert [mean, lowCI, highCI] to asymmetric yerr for matplotlib.
        """
        mean, low, high = triplet
        lower = max(mean - low, 0.0)
        upper = max(high - mean, 0.0)
        return mean, np.array([[lower], [upper]])  # shape (2,1)

    # Aggregate across states
    high_triplet = avg_across_states(P_deaths_ERKhigh_CR)  # [mean, low, high]
    low_triplet  = avg_across_states(P_deaths_ERKlow_CR)

    # Convert to bar heights and error bars
    high_mean, high_yerr = ci_to_yerr_from_triplet(high_triplet)
    low_mean,  low_yerr  = ci_to_yerr_from_triplet(low_triplet)

    # Prepare plotting
    x = np.array([0, 1])
    heights = np.array([low_mean, high_mean])
    # Stack yerrs to match bars order: [ERK-low, ERK-high]
    yerr = np.hstack([low_yerr, high_yerr])  # shape (2,2)

    fig, ax = plt.subplots(figsize=figsize)
    ax.bar(x, heights, yerr=yerr, capsize=5,color='dimgray')
    ax.set_xticks(x, ["ERK-low", "ERK-high"])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    #ax.grid(axis='y', linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    return fig, ax

# --- Example usage (replace with your arrays) ---
# P_deaths_ERKlow_CR  = np.array([[0.12, 0.09, 0.16],
#                                 [0.18, 0.14, 0.24],
#                                 [0.15, 0.11, 0.20]])
# P_deaths_ERKhigh_CR = np.array([[0.22, 0.18, 0.27],
#                                 [0.25, 0.20, 0.31],
#                                 [0.24, 0.19, 0.30]])
plot_erk_deaths_state_avg(P_deaths_ERKhigh_CR, P_deaths_ERKlow_CR)
plt.savefig(f'{date}/death_rates_model.svg')
# plt.show()


### show some 3D segmentations

In [ ]:
vizpath=f'{date}/3Dsegmentations'
os.system(f'mkdir {vizpath}')

In [ ]:
def show_3dseg_proj(fig,im3d,labels,surfaces=None,surf_cmaps=None,labels2=None,fmsk=None,zscale=None,label_color='black',label2_color='red',img_cmap=plt.cm.gray_r):
    if zscale is not None:
        im3d=skimage.transform.rescale(im3d,(zscale,1,1),anti_aliasing=False)
        labels=skimage.transform.rescale(labels,(zscale,1,1),order=0)
        for isurf in range(len(surfaces)):
            surfaces[isurf]=skimage.transform.rescale(surfaces[isurf],(zscale,1,1),order=0)
        if labels2 is not None:
            labels2=skimage.transform.rescale(labels2,(zscale,1,1),order=0)
    if surf_cmaps is None and surfaces is not None:
        print(len(surfaces))
        surf_cmaps=[plt.cm.Blues,plt.cm.Greens,plt.cm.Reds,plt.cm.Purples,plt.cm.Oranges,plt.cm.Blues,plt.cm.Greens]
    if label_color is None:
        label_color='black'
    if label2_color is None:
        label2_color='red'
    if img_cmap is None:
        img_cmap=plt.cm.gray_r
    zproj=np.sum(im3d,axis=0)
    xproj=np.sum(im3d,axis=1)
    yproj=np.sum(im3d,axis=2)
    labels_zproj=np.max(labels,axis=0)
    labels_xproj=np.max(labels,axis=1)
    labels_yproj=np.max(labels,axis=2)
    plt.subplot(1,3,1)
    plt.imshow(zproj,cmap=img_cmap)
    plt.contour(labels_zproj,levels=np.arange(np.max(labels_zproj)),colors=label_color,linewidths=1)
    if surfaces is not None:
        for isurf in range(len(surfaces)):
            surf_viz=np.sum(surfaces[isurf],axis=0)
            plt.imshow(np.ma.masked_where(np.logical_not(surf_viz>0),surf_viz),alpha=.5,cmap=surf_cmaps[isurf])#,clim=(0,1))
    if labels2 is not None:
        labels2_zproj=np.max(labels2,axis=0)
        plt.contour(labels2_zproj,levels=np.arange(np.max(labels2_zproj)),colors=label2_color,linewidths=1)  
    plt.axis('equal');plt.axis('off')
    plt.subplot(1,3,2)
    plt.imshow(xproj,cmap=img_cmap)
    plt.contour(labels_xproj,levels=np.arange(np.max(labels_xproj)),colors=label_color,linewidths=1)
    if surfaces is not None:
        for isurf in range(len(surfaces)):
            surf_viz=np.sum(surfaces[isurf],axis=1)
            plt.imshow(np.ma.masked_where(np.logical_not(surf_viz>0),surf_viz),alpha=.5,cmap=surf_cmaps[isurf])#,clim=(0,1))
    if labels2 is not None:
        labels2_xproj=np.max(labels2,axis=1)
        plt.contour(labels2_xproj,levels=np.arange(np.max(labels2_xproj)),colors=label2_color,linewidths=1)
    plt.axis('equal');plt.axis('off')
    plt.subplot(1,3,3)
    plt.imshow(yproj,cmap=img_cmap)
    plt.contour(labels_yproj,levels=np.arange(np.max(labels_yproj)),colors=label_color,linewidths=1)
    if labels2 is not None:
        labels2_yproj=np.max(labels2,axis=2)
        plt.contour(labels2_yproj,levels=np.arange(np.max(labels2_yproj)),colors=label2_color,linewidths=1)
    if surfaces is not None:
        for isurf in range(len(surfaces)):
            surf_viz=np.sum(surfaces[isurf],axis=2)
            plt.imshow(np.ma.masked_where(np.logical_not(surf_viz>0),surf_viz),alpha=.5,cmap=surf_cmaps[isurf])#,clim=(0,1)) 
    plt.axis('equal');plt.axis('off')

In [ ]:
imodel=9
sctm=modelSet[imodel] #81]
#status=sctm.load_from_h5('/cell_data_m0/')
fmsks_lung=sctm.fmsk_lung.copy()
iS=2
cons_fmsk_lung=fmsks_lung #[...,iS] #np.median(fmsks_lung,axis=3)
cyto_msk_channel=np.where(sctm.mask_channels==np.array(['cyto']).astype('S32'))[0][0]
nuc_fmsk_channel=np.where(sctm.fmask_channels==np.array(['nuc']).astype('S32'))[0][0]
rep_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
bf_img_channel=np.where(sctm.channels==np.array(['BF']).astype('S32'))[0][0]
erk_img_channel=np.where(sctm.channels==np.array(['YFP']).astype('S32'))[0][0]
akt_img_channel=np.where(sctm.channels==np.array(['TRITC']).astype('S32'))[0][0]
nuc_img_channel=np.where(sctm.channels==np.array(['DAPI']).astype('S32'))[0][0]#Cy5']).astype('S32'))[0][0]
img=sctm.get_image_data(iS)
msk=sctm.get_mask_data(iS)
fmsks=sctm.get_fmask_data(iS,channel=5)
show_3dseg_proj(plt.figure(figsize=(12,5)),img[...,erk_img_channel],msk[...,0],surfaces=[cons_fmsk_lung,msk[...,0]>0],zscale=sctm.zscale,label_color='orange',surf_cmaps=[plt.cm.Greens,plt.cm.Oranges])
plt.title(f'{sctm.figid.decode()} frame {iS}')
plt.savefig(f'{vizpath}/ilastik_3dseg_{sctm.figid.decode()}_f{iS:04d}.svg')

In [ ]:
import cmapfile
import pathlib
im3d=np.zeros((2,1,img.shape[0],img.shape[1],img.shape[2]))
im3d[0,0,:,:,:]=img[:,:,:,erk_img_channel]
im3d[1,0,:,:,:]=img[:,:,:,nuc_img_channel]
cmapfile.array2cmap(im3d, 'CTZXY', pathlib.Path(f'{vizpath}/frame{iS}_{sctm.figid.decode()}_img3d'), subsample=1, step=[1,1,sctm.zscale])
label3d=np.zeros((2,1,msk.shape[0],msk.shape[1],msk.shape[2])).astype(int)
label3d[0,0,:,:,:]=msk[:,:,:,cyto_msk_channel].astype(int)
label3d[1,0,:,:,:]=cons_fmsk_lung.astype(int)
cmapfile.array2cmap(label3d.astype('int32'), 'CTZXY', pathlib.Path(f'{vizpath}/frame{iS}_{sctm.figid.decode()}_label3d'), subsample=1, step=[1,1,sctm.zscale])
print(f'wrote {vizpath}/frame{iS}_{sctm.figid.decode()}')

In [ ]:
import cmapfile
import pathlib
msk_T=np.zeros((msk.shape[0],msk.shape[2],msk.shape[1])).astype(int)
cons_fmsk_lung_T=np.zeros((msk.shape[0],msk.shape[2],msk.shape[1])).astype(int)
for iz in range(msk.shape[0]):
    cons_fmsk_lung_T[iz,...]=cons_fmsk_lung[iz,...].T
    msk_T[iz,...]=msk[iz,...,0].T
    
label3d=np.zeros((2,1,msk_T.shape[0],msk_T.shape[1],msk_T.shape[2])).astype(int)
label3d[0,0,:,:,:]=msk_T.astype(int)
label3d[1,0,:,:,:]=cons_fmsk_lung_T.astype(int)
cmapfile.array2cmap(label3d.astype('int32'), 'CTZXY', pathlib.Path(f'{vizpath}/frame{iS}_{sctm.figid.decode()}_label3dT'), subsample=1, step=[1,1,sctm.zscale])
print(f'wrote {vizpath}/frame{iS}_{sctm.figid.decode()}')

In [ ]:
imodel=3

In [ ]:
import cmapfile
import pathlib
msk_T=np.zeros((msk.shape[0],msk.shape[2],msk.shape[1])).astype(int)
cons_fmsk_lung_T=np.zeros((msk.shape[0],msk.shape[2],msk.shape[1])).astype(int)
for iz in range(msk.shape[0]):
    cons_fmsk_lung_T[iz,...]=cons_fmsk_lung[iz,...].T
    msk_T[iz,...]=msk[iz,...,0].T
    
label3d=np.zeros((n_states,1,msk_T.shape[0],msk_T.shape[1],msk_T.shape[2])).astype(int)
for istate in range(n_states):
    msk_state=msk_T.copy()
    indstate_traj=inds_states[istate]
    indmodel_traj=np.where(indtreatment_traj==imodel)[0]
    indstate_traj_model=np.intersect1d(indmodel_traj,indstate_traj)
    cellinds_model=cellinds0_traj[indstate_traj_model]
    cellinds_model_frame=cellinds_model[sctm.cells_frameSet[cellinds_model]==iS]
    labelids=sctm.cells_labelidSet[cellinds_model_frame]
    msk_state[np.logical_not(np.isin(msk_state,labelids))]=0
    label3d[istate,0,:,:,:]=msk_state.astype(int)
    plt.contour(np.isin(np.max(msk_T,axis=0),labelids),colors=[plt.cm.jet(istate/n_states)])

cmapfile.array2cmap(label3d.astype('int32'), 'CTZXY', pathlib.Path(f'{vizpath}/frame{iS}_{sctm.figid.decode()}_states_label3dT'), subsample=1, step=[1,1,sctm.zscale])#print(f'wrote {vizpath}/frame{iS}_states_{sctm.figid.decode()}')

In [ ]:
label3d=np.zeros((1,sctm.nt,msk.shape[0],msk.shape[1],msk.shape[2])).astype(int)
for iS in range(sctm.nt):
    msk=sctm.get_mask_data(iS)
    label3d[0,iS,:,:,:]=msk[:,:,:,cyto_msk_channel].astype(int)

cmapfile.array2cmap(label3d.astype('int32'), 'CTZXY', pathlib.Path(f'{vizpath}/allframes_{sctm.figid.decode()}_label3d'), subsample=1, step=[1,1,sctm.zscale])

In [ ]:
im3d_bf=img[...,bf_img_channel]
for islice in range(im3d_bf.shape[0]):
    plt.figure(figsize=(12,12))
    msk_cyto=msk[...,0]
    plt.imshow(im3d_bf[islice,...],cmap=plt.cm.binary)
    fmsk_cancer=msk_cyto[islice,...]>0
    #fmsk_cancer[cons_fmsk_lung[islice,...]==1]=False
    cons_fmsk_lung[islice,fmsk_cancer]=False
    plt.imshow(np.ma.masked_where(cons_fmsk_lung[islice,...]==0,np.ones_like(fmsks[islice,...]==0)),cmap=plt.cm.Greens,alpha=.2,clim=(0,1))
    #fmsk_cancer[fmasks[islice,...,0]==1]=False
    plt.imshow(np.ma.masked_where(fmsk_cancer==0,np.ones_like(fmsks[islice,...]==0)),cmap=plt.cm.Oranges,alpha=.2,clim=(0,1))
    plt.contour(cons_fmsk_lung[islice,...],levels=[0,1],colors='green',linewidths=1.5,alpha=.8)
    plt.contour(msk_cyto[islice,...],levels=np.unique(msk_cyto[islice,...]),colors='orange',linewidths=2,alpha=.8)
    plt.axis('off')
    plt.savefig(f'{vizpath}/segmentation2d_{sctm.figid.decode()}_z{islice}.svg')
    plt.show()